# NB_P2_12: Live-Compatible CRVSE Optuna Search

## Research question

This notebook tests whether live-compatible source-FPS tensors can produce a better CRVSE PhysFormer checkpoint when architecture and training hyperparameters are searched deliberately.

The notebook starts by reproducing the NB10/NB11 artifact context before training anything.

## Why this notebook exists

NB10 showed that shallow fine-tuning of the existing checkpoint produced only small gains and worsened ECG-Fitness out-of-domain behavior.

NB11 showed that stronger transfer learning improved live-compatible validation and held-out test performance, but still introduced ECG-Fitness / high-HR penalty. Scratch training was not competitive in that notebook.

NB12 therefore treats model improvement as a controlled search problem, not as another manual training run.

## ECG-Fitness interpretation rule

If ECG-Fitness is excluded from training, it is treated as out-of-domain exercise/high-HR evaluation.

If ECG-Fitness is included in training, it is no longer described as out-of-domain. It becomes held-out exercise/high-HR evaluation and must use subject-level separation.

## Imporst and Setup

In [1]:
import json, os, random, sys, time,  torch, math, re, copy, optuna
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler




SEED = 42
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
NB12_WORKING_DIR = KAGGLE_WORKING / "crvse_nb12_optuna_search"
NB12_WORKING_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [KAGGLE_INPUT, KAGGLE_WORKING]


def set_reproducibility_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible notebook checks."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def list_exact_file_candidates(filename: str, roots: list[Path]) -> list[Path]:
    """Return sorted paths matching one exact filename under existing roots.

    Parameters
    ----------
    filename:
        Exact file name to search for.
    roots:
        Directories that may contain Kaggle input or working artifacts.

    Returns
    -------
    list[Path]
        Sorted unique matching file paths.
    """
    candidates = []

    for root in roots:
        if not root.exists():
            continue

        candidates.extend(
            path
            for path in root.rglob(filename)
            if path.is_file()
        )

    unique_candidates = sorted({path.resolve() for path in candidates})
    return unique_candidates


def build_artifact_candidate_report(artifact_specs: list[dict], roots: list[Path]) -> pd.DataFrame:
    """Build an availability report for expected NB10/NB11/NB12 input artifacts.

    The function intentionally does not choose between duplicate candidates.
    Duplicate paths are reported so the notebook author can decide which artifact
    source should be trusted before any training begins.
    """
    rows = []

    for spec in artifact_specs:
        candidates = list_exact_file_candidates(
            filename=spec["filename"],
            roots=roots,
        )

        rows.append(
            {
                "artifact": spec["artifact"],
                "source": spec["source"],
                "filename": spec["filename"],
                "required_for_start": bool(spec["required_for_start"]),
                "candidate_count": len(candidates),
                "exists": len(candidates) > 0,
                "ambiguous": len(candidates) > 1,
                "candidate_paths": "\n".join(str(path) for path in candidates),
            }
        )

    return pd.DataFrame(rows)


set_reproducibility_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"NB12 working directory: {NB12_WORKING_DIR}")


ARTIFACT_SPECS = [
    {
        "artifact": "live_finetune_manifest",
        "source": "live-compatible preprocessing / NB10",
        "filename": "live_finetune_manifest.csv",
        "required_for_start": True,
    },
    {
        "artifact": "frozen_baseline_summary",
        "source": "NB10/NB11 baseline",
        "filename": "live_finetune_frozen_baseline_summary.csv",
        "required_for_start": True,
    },
    {
        "artifact": "frozen_baseline_predictions",
        "source": "NB10/NB11 baseline",
        "filename": "live_finetune_frozen_baseline_predictions.csv",
        "required_for_start": True,
    },
    {
        "artifact": "original_physformer_checkpoint",
        "source": "ensemble PhysFormer",
        "filename": "CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
        "required_for_start": True,
    },
    {
        "artifact": "train_tensors",
        "source": "NB10 materialized tensors",
        "filename": "finetune_train_sourcefps_tensors.npz",
        "required_for_start": True,
    },
    {
        "artifact": "train_metadata",
        "source": "NB10 materialized metadata",
        "filename": "finetune_train_sourcefps_metadata.csv",
        "required_for_start": True,
    },
    {
        "artifact": "val_tensors",
        "source": "NB10 materialized tensors",
        "filename": "finetune_val_sourcefps_tensors.npz",
        "required_for_start": True,
    },
    {
        "artifact": "val_metadata",
        "source": "NB10 materialized metadata",
        "filename": "finetune_val_sourcefps_metadata.csv",
        "required_for_start": True,
    },
    {
        "artifact": "test_tensors",
        "source": "NB10 materialized tensors",
        "filename": "finetune_test_sourcefps_tensors.npz",
        "required_for_start": True,
    },
    {
        "artifact": "test_metadata",
        "source": "NB10 materialized metadata",
        "filename": "finetune_test_sourcefps_metadata.csv",
        "required_for_start": True,
    },
    {
        "artifact": "ood_tensors",
        "source": "NB10 materialized tensors",
        "filename": "ood_eval_sourcefps_tensors.npz",
        "required_for_start": True,
    },
    {
        "artifact": "ood_metadata",
        "source": "NB10 materialized metadata",
        "filename": "ood_eval_sourcefps_metadata.csv",
        "required_for_start": True,
    },
    {
        "artifact": "nb10_conclusion",
        "source": "NB10 summary",
        "filename": "nb_p2_10_conclusion.json",
        "required_for_start": False,
    },
    {
        "artifact": "nb11_conclusion",
        "source": "NB11 summary",
        "filename": "nb11_final_conclusion.json",
        "required_for_start": False,
    },
    {
        "artifact": "nb11_scoreboard",
        "source": "NB11 summary",
        "filename": "nb11_final_scoreboard.csv",
        "required_for_start": False,
    },
]

artifact_report = build_artifact_candidate_report(artifact_specs=ARTIFACT_SPECS, roots=SEARCH_ROOTS,)
pd.set_option("display.max_colwidth", 240)
display(artifact_report)
missing_required = artifact_report[artifact_report["required_for_start"] & ~artifact_report["exists"]].copy()
ambiguous_required = artifact_report[artifact_report["required_for_start"] & artifact_report["ambiguous"]].copy()

startup_report = {
    "required_artifacts": int(artifact_report["required_for_start"].sum()),
    "missing_required": int(len(missing_required)),
    "ambiguous_required": int(len(ambiguous_required)),
    "ready_for_baseline_reproduction": bool(
        len(missing_required) == 0 and len(ambiguous_required) == 0
    ),
}

print("\nNB12 artifact startup report:")
print(json.dumps(startup_report, indent=2))

if len(missing_required) > 0:
    print("\nMissing required artifacts:")
    display(missing_required[["artifact", "filename", "source"]])

if len(ambiguous_required) > 0:
    print("\nAmbiguous required artifacts:")
    display(ambiguous_required[["artifact", "filename", "candidate_count", "candidate_paths"]])

Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.3.3
PyTorch: 2.10.0+cu128
Device: cuda
NB12 working directory: /kaggle/working/crvse_nb12_optuna_search


,artifact,source,filename,required_for_start,candidate_count,exists,ambiguous,candidate_paths
0,live_finetune_manifest,live-compatible preprocessing / NB10,live_finetune_manifest.csv,True,1,True,False,/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/live_finetune_manifest.csv
1,frozen_baseline_summary,NB10/NB11 baseline,live_finetune_frozen_baseline_summary.csv,True,1,True,False,/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/live_finetune_frozen_baseline_summary.csv
2,frozen_baseline_predictions,NB10/NB11 baseline,live_finetune_frozen_baseline_predictions.csv,True,1,True,False,/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/live_finetune_frozen_baseline_predictions.csv
3,original_physformer_checkpoint,ensemble PhysFormer,CRVSETransformer_Ensemble_physformer_multichannel_best.pt,True,1,True,False,/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/CRVSETransformer_Ensemble_physformer_multichannel_best.pt
4,train_tensors,NB10 materialized tensors,finetune_train_sourcefps_tensors.npz,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_train_sourcefps_tensors.npz
5,train_metadata,NB10 materialized metadata,finetune_train_sourcefps_metadata.csv,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_train_sourcefps_metadata.csv
6,val_tensors,NB10 materialized tensors,finetune_val_sourcefps_tensors.npz,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_val_sourcefps_tensors.npz
7,val_metadata,NB10 materialized metadata,finetune_val_sourcefps_metadata.csv,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_val_sourcefps_metadata.csv
8,test_tensors,NB10 materialized tensors,finetune_test_sourcefps_tensors.npz,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_test_sourcefps_tensors.npz
9,test_metadata,NB10 materialized metadata,finetune_test_sourcefps_metadata.csv,True,1,True,False,/kaggle/input/notebooks/cezarytubacki/nb-p2-10-live-compatible-crvse-finetune-physformer/crvse_live_sourcefps_materialized/finetune_test_sourcefps_metadata.csv



NB12 artifact startup report:
{
  "required_artifacts": 12,
  "missing_required": 0,
  "ambiguous_required": 0,
  "ready_for_baseline_reproduction": true
}


## Frozen Baseline And Tensor Cache Reproduction

This section verifies that NB12 can reproduce the frozen source-FPS baseline context before any search is started.

The section checks:

- manifest role counts,
- materialized tensor shapes,
- tensor and metadata alignment,
- baseline prediction modes,
- frozen source-FPS baseline MAE by role,
- NB10 and NB11 conclusion artifacts.

This is an audit step, not a training step. A search should only begin after this baseline context is reproduced.

In [2]:
EXPECTED_MANIFEST_ROLE_COUNTS = {
    "finetune_train": 12503,
    "finetune_val": 2839,
    "finetune_test": 2828,
    "ood_eval": 882,
    "excluded": 191,
}

EXPECTED_TENSOR_ROLE_ROWS = {
    "finetune_train": 12503,
    "finetune_val": 2839,
    "finetune_test": 2828,
    "ood_eval": 882,
}

EXPECTED_SOURCEFPS_BASELINE_MAE = {
    "finetune_train": 5.8236,
    "finetune_val": 6.0275,
    "finetune_test": 5.7696,
    "ood_eval": 14.0355,
}

EXPECTED_PREPROCESSING_MODES = {
    "stored_reference",
    "training_buffer_source_fps",
    "training_buffer_sim_15fps",
    "training_buffer_sim_10fps",
}

SOURCEFPS_MODE = "training_buffer_source_fps"
BASELINE_TOLERANCE_BPM = 0.02


def get_unique_artifact_path(artifact_name: str) -> Path:
    """Return the single resolved path for one artifact from the startup report."""
    if "artifact_report" not in globals():
        raise NameError("Run the NB12 artifact startup cell before this cell.")

    rows = artifact_report.loc[artifact_report["artifact"] == artifact_name]

    if len(rows) != 1:
        raise ValueError(f"Expected one report row for {artifact_name}, found {len(rows)}.")

    row = rows.iloc[0]

    if not bool(row["exists"]):
        raise FileNotFoundError(f"Artifact is missing: {artifact_name}")

    if bool(row["ambiguous"]):
        raise ValueError(f"Artifact is ambiguous: {artifact_name}")

    return Path(str(row["candidate_paths"]).splitlines()[0])


def load_optional_json(path: Path | None) -> dict | None:
    """Load a JSON file when a path is available."""
    if path is None or not path.exists():
        return None

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def parse_bool(value) -> bool:
    """Parse bool-like CSV values into Python booleans."""
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def summarize_manifest_roles(manifest_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize manifest rows by window role and compare them with NB10 counts."""
    if "window_role" not in manifest_df.columns:
        raise KeyError("Manifest is missing required column: window_role")

    role_counts = (
        manifest_df["window_role"]
        .value_counts()
        .rename_axis("window_role")
        .reset_index(name="actual_rows")
    )

    expected = pd.DataFrame(
        [
            {"window_role": role, "expected_rows": rows}
            for role, rows in EXPECTED_MANIFEST_ROLE_COUNTS.items()
        ]
    )

    summary = expected.merge(role_counts, on="window_role", how="left")
    summary["actual_rows"] = summary["actual_rows"].fillna(0).astype(int)
    summary["matches_expected"] = summary["actual_rows"] == summary["expected_rows"]

    return summary


def load_tensor_role(role: str, tensor_artifact: str, metadata_artifact: str) -> dict:
    """Load one materialized tensor role and its metadata."""
    tensor_path = get_unique_artifact_path(tensor_artifact)
    metadata_path = get_unique_artifact_path(metadata_artifact)

    loaded = np.load(tensor_path)

    missing_keys = {"x", "y"} - set(loaded.files)
    if missing_keys:
        raise KeyError(f"{tensor_path} is missing required arrays: {sorted(missing_keys)}")

    x = loaded["x"].astype(np.float32, copy=False)
    y = loaded["y"].astype(np.float32, copy=False)
    metadata = pd.read_csv(metadata_path)

    return {
        "role": role,
        "tensor_path": tensor_path,
        "metadata_path": metadata_path,
        "x": x,
        "y": y,
        "metadata": metadata,
    }


def summarize_tensor_role(cache_entry: dict) -> dict:
    """Summarize shape, finite-value status, and metadata alignment for one tensor role."""
    role = cache_entry["role"]
    x = cache_entry["x"]
    y = cache_entry["y"]
    metadata = cache_entry["metadata"]

    metadata_roles = (
        sorted(metadata["window_role"].dropna().unique().tolist())
        if "window_role" in metadata.columns
        else []
    )

    datasets = (
        metadata["dataset"].value_counts().to_dict()
        if "dataset" in metadata.columns
        else {}
    )

    expected_rows = EXPECTED_TENSOR_ROLE_ROWS[role]

    return {
        "role": role,
        "expected_rows": expected_rows,
        "x_shape": tuple(x.shape),
        "y_shape": tuple(y.shape),
        "metadata_rows": int(len(metadata)),
        "x_dtype": str(x.dtype),
        "y_dtype": str(y.dtype),
        "x_finite": bool(np.isfinite(x).all()),
        "y_finite": bool(np.isfinite(y).all()),
        "row_count_ok": bool(x.shape[0] == y.shape[0] == len(metadata) == expected_rows),
        "shape_ok": bool(x.ndim == 3 and x.shape[1] == 3 and x.shape[2] == 240),
        "metadata_role_values": metadata_roles,
        "metadata_role_ok": metadata_roles == [role],
        "target_hr_min": float(np.min(y)),
        "target_hr_max": float(np.max(y)),
        "dataset_counts": datasets,
    }


def find_first_existing_column(
    df: pd.DataFrame,
    candidates: list[str],
    label: str,
) -> str:
    """Return the first available column from a list of candidate names."""
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    raise KeyError(
        f"Could not find {label} column. Tried {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def summarize_prediction_errors(
    predictions: pd.DataFrame,
    group_cols: list[str],
    target_col: str,
    pred_col: str,
) -> pd.DataFrame:
    """Summarize absolute and signed HR error for grouped prediction rows."""
    table = predictions.copy()
    table["target_hr"] = table[target_col].astype(float)
    table["pred_hr"] = table[pred_col].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    summary = (
        table
        .groupby(group_cols, dropna=False)
        .agg(
            rows=("abs_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda values: values.quantile(0.90)),
            mae_p95=("abs_error", lambda values: values.quantile(0.95)),
            mae_max=("abs_error", "max"),
            signed_error_mean=("signed_error", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


artifact_paths = {
    spec["artifact"]: get_unique_artifact_path(spec["artifact"])
    for spec in ARTIFACT_SPECS
    if bool(artifact_report.loc[artifact_report["artifact"] == spec["artifact"], "exists"].iloc[0])
}

manifest = pd.read_csv(artifact_paths["live_finetune_manifest"])
baseline_summary = pd.read_csv(artifact_paths["frozen_baseline_summary"])
baseline_predictions = pd.read_csv(
    artifact_paths["frozen_baseline_predictions"],
    low_memory=False,
)

nb10_conclusion = load_optional_json(artifact_paths.get("nb10_conclusion"))
nb11_conclusion = load_optional_json(artifact_paths.get("nb11_conclusion"))

if "include_in_finetune" in manifest.columns and "include_in_finetune_bool" not in manifest.columns:
    manifest["include_in_finetune_bool"] = manifest["include_in_finetune"].apply(parse_bool)

manifest_role_summary = summarize_manifest_roles(manifest)

tensor_artifacts = {
    "finetune_train": ("train_tensors", "train_metadata"),
    "finetune_val": ("val_tensors", "val_metadata"),
    "finetune_test": ("test_tensors", "test_metadata"),
    "ood_eval": ("ood_tensors", "ood_metadata"),
}

TENSOR_CACHE = {
    role: load_tensor_role(role, tensor_artifact, metadata_artifact)
    for role, (tensor_artifact, metadata_artifact) in tensor_artifacts.items()
}

tensor_report = pd.DataFrame(
    [summarize_tensor_role(cache_entry) for cache_entry in TENSOR_CACHE.values()]
)

mode_col = find_first_existing_column(
    baseline_predictions,
    ["preprocessing_mode", "mode"],
    "preprocessing mode",
)

role_col = find_first_existing_column(
    baseline_predictions,
    ["window_role", "role"],
    "role",
)

dataset_col = find_first_existing_column(
    baseline_predictions,
    ["dataset", "dataset_name"],
    "dataset",
)

target_col = find_first_existing_column(
    baseline_predictions,
    ["target_hr_from_tensor", "target_hr", "target_hr_mean"],
    "target HR",
)

pred_col = find_first_existing_column(
    baseline_predictions,
    ["notebook_model_hr", "pred_hr", "predicted_hr", "model_hr"],
    "predicted HR",
)

mode_counts = (
    baseline_predictions[mode_col]
    .value_counts()
    .rename_axis("preprocessing_mode")
    .reset_index(name="rows")
)

sourcefps_predictions = baseline_predictions.loc[
    baseline_predictions[mode_col] == SOURCEFPS_MODE
].copy()

sourcefps_role_summary = summarize_prediction_errors(
    predictions=sourcefps_predictions,
    group_cols=[role_col],
    target_col=target_col,
    pred_col=pred_col,
).rename(columns={role_col: "role"})

sourcefps_dataset_role_summary = summarize_prediction_errors(
    predictions=sourcefps_predictions,
    group_cols=[dataset_col, role_col],
    target_col=target_col,
    pred_col=pred_col,
).rename(columns={dataset_col: "dataset", role_col: "role"})

expected_baseline = pd.DataFrame(
    [
        {"role": role, "expected_mae_mean": mae}
        for role, mae in EXPECTED_SOURCEFPS_BASELINE_MAE.items()
    ]
)

baseline_comparison = sourcefps_role_summary.merge(
    expected_baseline,
    on="role",
    how="left",
)

baseline_comparison["delta_vs_expected"] = (
    baseline_comparison["mae_mean"] - baseline_comparison["expected_mae_mean"]
).round(4)

baseline_comparison["matches_expected"] = (
    baseline_comparison["delta_vs_expected"].abs() <= BASELINE_TOLERANCE_BPM
)

manifest_roles_ok = bool(manifest_role_summary["matches_expected"].all())
tensor_cache_ok = bool(
    tensor_report["row_count_ok"].all()
    and tensor_report["shape_ok"].all()
    and tensor_report["x_finite"].all()
    and tensor_report["y_finite"].all()
    and tensor_report["metadata_role_ok"].all()
)

mode_set = set(mode_counts["preprocessing_mode"].astype(str))
baseline_modes_ok = mode_set == EXPECTED_PREPROCESSING_MODES
sourcefps_rows_ok = len(sourcefps_predictions) == sum(EXPECTED_TENSOR_ROLE_ROWS.values())
baseline_metrics_ok = bool(baseline_comparison["matches_expected"].all())

NB12_BASELINE_REPRODUCTION = {
    "manifest_roles_ok": manifest_roles_ok,
    "tensor_cache_ok": tensor_cache_ok,
    "baseline_modes_ok": baseline_modes_ok,
    "sourcefps_rows": int(len(sourcefps_predictions)),
    "sourcefps_rows_ok": sourcefps_rows_ok,
    "baseline_metrics_ok": baseline_metrics_ok,
    "ready_for_model_definition_check": bool(
        manifest_roles_ok
        and tensor_cache_ok
        and baseline_modes_ok
        and sourcefps_rows_ok
        and baseline_metrics_ok
    ),
    "nb10_adopt_checkpoint": None if nb10_conclusion is None else nb10_conclusion.get("adopt_checkpoint"),
    "nb11_adopt_checkpoint_now": None if nb11_conclusion is None else nb11_conclusion.get("adopt_checkpoint_now"),
    "nb11_current_app_checkpoint": None if nb11_conclusion is None else nb11_conclusion.get("current_app_checkpoint_remains"),
}

print("Manifest role summary:")
display(manifest_role_summary)

print("\nTensor cache report:")
display(tensor_report)

print("\nBaseline prediction mode counts:")
display(mode_counts)

print("\nFrozen source-FPS baseline role comparison:")
display(baseline_comparison)

print("\nFrozen source-FPS baseline by dataset and role:")
display(sourcefps_dataset_role_summary.sort_values(["role", "dataset"]))

print("\nNB12 baseline reproduction report:")
print(json.dumps(NB12_BASELINE_REPRODUCTION, indent=2))

Manifest role summary:


,window_role,expected_rows,actual_rows,matches_expected
0,finetune_train,12503,12503,True
1,finetune_val,2839,2839,True
2,finetune_test,2828,2828,True
3,ood_eval,882,882,True
4,excluded,191,191,True



Tensor cache report:


,role,expected_rows,x_shape,y_shape,metadata_rows,x_dtype,y_dtype,x_finite,y_finite,row_count_ok,shape_ok,metadata_role_values,metadata_role_ok,target_hr_min,target_hr_max,dataset_counts
0,finetune_train,12503,"(12503, 3, 240)","(12503,)",12503,float32,float32,True,True,True,True,[finetune_train],True,40.056313,127.000000,"{'mcd_rppg': 7446, 'ubfc_phys': 4595, 'ubfc_rppg': 462}"
1,finetune_val,2839,"(2839, 3, 240)","(2839,)",2839,float32,float32,True,True,True,True,[finetune_val],True,40.296436,127.000000,"{'mcd_rppg': 1743, 'ubfc_phys': 986, 'ubfc_rppg': 110}"
2,finetune_test,2828,"(2828, 3, 240)","(2828,)",2828,float32,float32,True,True,True,True,[finetune_test],True,47.631721,129.071884,"{'mcd_rppg': 2094, 'ubfc_phys': 658, 'ubfc_rppg': 76}"
3,ood_eval,882,"(882, 3, 240)","(882,)",882,float32,float32,True,True,True,True,[ood_eval],True,53.662189,167.968079,{'ecg_fitness': 882}



Baseline prediction mode counts:


...


Frozen source-FPS baseline role comparison:


,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,expected_mae_mean,delta_vs_expected,matches_expected
0,finetune_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,72.2401,1.4174,5.7696,0.0,True
1,finetune_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,54.9838,1.5658,5.8236,0.0,True
2,finetune_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,50.2965,1.4724,6.0275,0.0,True
3,ood_eval,882,109.2048,101.2561,14.0355,7.6839,38.1499,49.9899,89.9549,-7.9487,14.0355,0.0,True



Frozen source-FPS baseline by dataset and role:


,dataset,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean
1,mcd_rppg,finetune_test,2094,79.8533,80.7775,4.8847,3.1502,10.9700,15.5490,49.4722,0.9242
4,ubfc_phys,finetune_test,658,77.4205,80.6185,8.8584,5.6735,20.2862,25.6733,72.2401,3.1980
7,ubfc_rppg,finetune_test,76,98.2693,97.8591,3.4085,3.1453,6.0835,6.8394,13.5339,-0.4102
2,mcd_rppg,finetune_train,7446,79.0648,80.3294,4.6876,3.1639,10.3722,13.9969,54.9838,1.2645
5,ubfc_phys,finetune_train,4595,80.3055,82.6246,7.7819,5.4248,18.5928,23.4842,50.9827,2.3192
8,ubfc_rppg,finetune_train,462,98.4492,97.3769,4.6548,3.1447,9.9189,13.1687,30.5685,-1.0723
3,mcd_rppg,finetune_val,1743,79.1086,80.3743,5.1325,3.3746,11.5368,15.4784,46.6087,1.2656
6,ubfc_phys,finetune_val,986,80.0024,82.2850,7.7629,4.8461,18.2725,24.2656,50.2965,2.2826
9,ubfc_rppg,finetune_val,110,98.2772,95.7647,4.6533,2.4438,12.3774,15.4017,23.5147,-2.5126
0,ecg_fitness,ood_eval,882,109.2048,101.2561,14.0355,7.6839,38.1499,49.9899,89.9549,-7.9487



NB12 baseline reproduction report:
{
  "manifest_roles_ok": true,
  "tensor_cache_ok": true,
  "baseline_modes_ok": true,
  "sourcefps_rows": 19052,
  "sourcefps_rows_ok": true,
  "baseline_metrics_ok": true,
  "ready_for_model_definition_check": true,
  "nb10_adopt_checkpoint": false,
  "nb11_adopt_checkpoint_now": false,
  "nb11_current_app_checkpoint": "frozen_sourcefps_baseline"
}


## Frozen Checkpoint Model Reproduction

This section rebuilds the CRVSE PhysFormer architecture used by the frozen checkpoint and evaluates it directly on the materialized source-FPS tensors.

This is still not training.

The purpose is to verify that NB12 can:

- reconstruct the checkpoint-compatible model,
- load the frozen weights,
- run inference on the cached tensors,
- reproduce the saved frozen baseline metrics,
- and establish a trusted evaluation function for later Optuna trials.

If this step does not match the saved baseline, the search should not start.

In [3]:
EVAL_BATCH_SIZE = 256
MODEL_REPRODUCTION_TOLERANCE_BPM = 0.03
ROLE_ORDER = ["finetune_train", "finetune_val", "finetune_test", "ood_eval"]


class MaterializedTensorDataset(Dataset):
    """Dataset backed by already-loaded source-FPS tensors and metadata."""

    def __init__(self, role: str, cache_entry: dict) -> None:
        self.role = role
        self.x = cache_entry["x"]
        self.y = cache_entry["y"]
        self.metadata = cache_entry["metadata"].copy()

        if len(self.x) != len(self.y):
            raise ValueError(f"{role}: x/y row mismatch: {len(self.x)} vs {len(self.y)}.")

        if len(self.x) != len(self.metadata):
            raise ValueError(
                f"{role}: tensor/metadata row mismatch: {len(self.x)} vs {len(self.metadata)}."
            )

    def __len__(self) -> int:
        return int(len(self.x))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.x[index]).float()
        y = torch.tensor(self.y[index], dtype=torch.float32)
        return x, y


def make_eval_loader(dataset: Dataset, batch_size: int = EVAL_BATCH_SIZE) -> DataLoader:
    """Create a deterministic evaluation DataLoader."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer time tokens."""

    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Input sequence length {x.size(1)} exceeds positional encoding max length "
                f"{self.pe.size(1)}."
            )

        return self.dropout(x + self.pe[:, : x.size(1), :])


class TransformerEncoderLayerCustom(nn.Module):
    """Custom pre-norm transformer encoder layer used by the CRVSE checkpoint."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, time_steps, -1)

        return self.out_proj(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class CRVSEPhysFormer(nn.Module):
    """CNN, FFT, and Transformer model for source-FPS rPPG HR estimation."""

    def __init__(
        self,
        in_channels: int = 3,
        cnn_channels: int = 16,
        freq_channels: int = 64,
        n_heads: int = 4,
        n_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.11331939348791525,
        hr_min: float = 40.0,
        hr_max: float = 180.0,
        target_frames: int = 240,
        max_positional_length: int = 300,
    ) -> None:
        super().__init__()

        self.in_channels = in_channels
        self.target_frames = target_frames
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.d_model = cnn_channels + freq_channels

        if self.d_model % n_heads != 0:
            raise ValueError(f"d_model={self.d_model} must be divisible by n_heads={n_heads}.")

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        n_fft_bins = target_frames // 2 + 1

        self.freq_proj = nn.Sequential(
            nn.Linear(n_fft_bins, freq_channels * 4),
            nn.ReLU(),
            nn.Linear(freq_channels * 4, freq_channels),
        )

        self.pos_enc = PositionalEncoding(
            d_model=self.d_model,
            max_len=max_positional_length,
            dropout=dropout,
        )

        self.transformer_layers = nn.ModuleList(
            [
                TransformerEncoderLayerCustom(
                    d_model=self.d_model,
                    n_heads=n_heads,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(self.d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected x shape (batch, channels, time), got {tuple(x.shape)}.")

        _, channels, time_steps = x.shape

        if channels != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} channels, got {channels}.")

        if time_steps != self.target_frames:
            raise ValueError(f"Expected {self.target_frames} frames, got {time_steps}.")

        time_feat = self.encoder(x).permute(0, 2, 1)

        freq = torch.fft.rfft(x, norm="ortho")
        freq_mag = freq.abs().mean(dim=1)
        freq_feat = self.freq_proj(freq_mag)
        freq_feat = freq_feat.unsqueeze(1).expand(-1, time_steps, -1)

        combined = torch.cat([time_feat, freq_feat], dim=-1)
        combined = self.pos_enc(combined)

        for layer in self.transformer_layers:
            combined = layer(combined)

        out = self.head(combined.mean(dim=1)).squeeze(-1)

        if not self.training:
            out = out.clamp(self.hr_min, self.hr_max)

        return out


def load_checkpoint_file(checkpoint_path: Path) -> dict:
    """Load a trusted local Kaggle checkpoint file."""
    try:
        return torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(checkpoint_path, map_location="cpu")


def build_checkpoint_model_config(state_dict: dict, best_params: dict) -> dict:
    """Infer the CRVSE PhysFormer config from checkpoint tensors."""
    layer_indices = [
        int(match.group(1))
        for key in state_dict
        for match in [re.match(r"transformer_layers\.(\d+)\.", key)]
        if match is not None
    ]

    if not layer_indices:
        raise ValueError("Could not infer transformer layer count from checkpoint.")

    return {
        "in_channels": int(state_dict["encoder.0.weight"].shape[1]),
        "cnn_channels": int(state_dict["encoder.8.weight"].shape[0]),
        "freq_channels": int(state_dict["freq_proj.2.weight"].shape[0]),
        "n_heads": int(best_params.get("n_heads", 4)),
        "n_layers": max(layer_indices) + 1,
        "dim_feedforward": int(state_dict["transformer_layers.0.ff.0.weight"].shape[0]),
        "dropout": float(best_params.get("dropout", 0.11331939348791525)),
        "hr_min": 40.0,
        "hr_max": 180.0,
        "target_frames": int((state_dict["freq_proj.0.weight"].shape[1] - 1) * 2),
        "max_positional_length": 300,
    }


def build_frozen_checkpoint_model(checkpoint_path: Path) -> tuple[CRVSEPhysFormer, dict, dict]:
    """Build the frozen checkpoint model and load its weights."""
    checkpoint = load_checkpoint_file(checkpoint_path)

    if "model_state" not in checkpoint:
        raise KeyError("Checkpoint is missing required key: model_state.")

    state_dict = checkpoint["model_state"]
    best_params = checkpoint.get("best_params", {})
    model_config = build_checkpoint_model_config(state_dict, best_params)

    model = CRVSEPhysFormer(**model_config)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    return model, checkpoint, model_config


def count_model_parameters(model: nn.Module) -> dict:
    """Count total and trainable model parameters."""
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

    return {
        "total_parameters": int(total),
        "trainable_parameters": int(trainable),
    }


def evaluate_hr_model(
    model: nn.Module,
    dataset: MaterializedTensorDataset,
    loader: DataLoader,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate one HR model on one materialized tensor dataset."""
    prediction_batches = []
    target_batches = []

    model.eval()

    with torch.inference_mode():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_pred = model(batch_x).detach().cpu().numpy().astype(np.float32)

            prediction_batches.append(batch_pred)
            target_batches.append(batch_y.numpy().astype(np.float32))

    pred_hr = np.concatenate(prediction_batches)
    target_hr = np.concatenate(target_batches)
    abs_error = np.abs(pred_hr - target_hr)
    signed_error = pred_hr - target_hr

    rows = dataset.metadata.copy()
    rows["notebook_model_hr"] = pred_hr
    rows["target_hr_from_tensor"] = target_hr
    rows["notebook_abs_error"] = abs_error
    rows["notebook_signed_error"] = signed_error
    rows["role"] = dataset.role

    metrics = {
        "role": dataset.role,
        "rows": int(len(rows)),
        "target_hr_mean": float(target_hr.mean()),
        "pred_hr_mean": float(pred_hr.mean()),
        "mae_mean": float(abs_error.mean()),
        "mae_median": float(np.median(abs_error)),
        "mae_p90": float(np.quantile(abs_error, 0.90)),
        "mae_p95": float(np.quantile(abs_error, 0.95)),
        "mae_max": float(abs_error.max()),
        "signed_error_mean": float(signed_error.mean()),
    }

    return rows, metrics


def summarize_predictions_by_dataset_role(predictions: pd.DataFrame) -> pd.DataFrame:
    """Summarize direct model predictions by dataset and role."""
    summary = (
        predictions
        .groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("notebook_abs_error", "size"),
            target_hr_mean=("target_hr_from_tensor", "mean"),
            pred_hr_mean=("notebook_model_hr", "mean"),
            mae_mean=("notebook_abs_error", "mean"),
            mae_median=("notebook_abs_error", "median"),
            mae_p90=("notebook_abs_error", lambda values: values.quantile(0.90)),
            mae_p95=("notebook_abs_error", lambda values: values.quantile(0.95)),
            mae_max=("notebook_abs_error", "max"),
            signed_error_mean=("notebook_signed_error", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary.sort_values(["window_role", "dataset"]).reset_index(drop=True)


DATASETS = {
    role: MaterializedTensorDataset(role, TENSOR_CACHE[role])
    for role in ROLE_ORDER
}

EVAL_LOADERS = {
    role: make_eval_loader(DATASETS[role])
    for role in ROLE_ORDER
}

FROZEN_MODEL, FROZEN_CHECKPOINT, FROZEN_MODEL_CONFIG = build_frozen_checkpoint_model(
    artifact_paths["original_physformer_checkpoint"]
)

parameter_report = count_model_parameters(FROZEN_MODEL)

checkpoint_report = {
    "checkpoint_input_mode": FROZEN_CHECKPOINT.get("input_mode"),
    "checkpoint_best_val_mae": FROZEN_CHECKPOINT.get("best_val_mae"),
    "checkpoint_best_n_epochs": FROZEN_CHECKPOINT.get("best_n_epochs"),
    "model_config": FROZEN_MODEL_CONFIG,
    **parameter_report,
    "device": str(DEVICE),
}

frozen_prediction_frames = []
frozen_metric_rows = []

for role in ROLE_ORDER:
    role_predictions, role_metrics = evaluate_hr_model(
        model=FROZEN_MODEL,
        dataset=DATASETS[role],
        loader=EVAL_LOADERS[role],
    )
    frozen_prediction_frames.append(role_predictions)
    frozen_metric_rows.append(role_metrics)

FROZEN_DIRECT_PREDICTIONS = pd.concat(frozen_prediction_frames, axis=0).reset_index(drop=True)

frozen_direct_role_summary = pd.DataFrame(frozen_metric_rows)
numeric_cols = frozen_direct_role_summary.select_dtypes(include=[np.number]).columns
frozen_direct_role_summary[numeric_cols] = frozen_direct_role_summary[numeric_cols].round(4)

frozen_direct_dataset_role_summary = summarize_predictions_by_dataset_role(
    FROZEN_DIRECT_PREDICTIONS
)

frozen_direct_comparison = frozen_direct_role_summary.merge(
    baseline_comparison[
        [
            "role",
            "rows",
            "mae_mean",
            "mae_median",
            "mae_p90",
            "mae_p95",
            "mae_max",
            "signed_error_mean",
        ]
    ].rename(
        columns={
            "mae_mean": "saved_mae_mean",
            "mae_median": "saved_mae_median",
            "mae_p90": "saved_mae_p90",
            "mae_p95": "saved_mae_p95",
            "mae_max": "saved_mae_max",
            "signed_error_mean": "saved_signed_error_mean",
        }
    ),
    on=["role", "rows"],
    how="outer",
    indicator=True,
)

if not frozen_direct_comparison["_merge"].eq("both").all():
    display(frozen_direct_comparison)
    raise AssertionError("Direct frozen predictions and saved baseline role summaries do not align.")

for metric in ["mae_mean", "mae_median", "mae_p90", "mae_p95", "mae_max", "signed_error_mean"]:
    frozen_direct_comparison[f"delta_{metric}"] = (
        frozen_direct_comparison[metric] - frozen_direct_comparison[f"saved_{metric}"]
    ).round(4)

delta_cols = [col for col in frozen_direct_comparison.columns if col.startswith("delta_")]

frozen_direct_comparison["matches_saved_baseline"] = (
    frozen_direct_comparison[delta_cols].abs().max(axis=1)
    <= MODEL_REPRODUCTION_TOLERANCE_BPM
)

frozen_direct_comparison = frozen_direct_comparison.drop(columns=["_merge"])

NB12_MODEL_REPRODUCTION = {
    "model_loaded": True,
    "model_config": FROZEN_MODEL_CONFIG,
    "total_parameters": parameter_report["total_parameters"],
    "trainable_parameters": parameter_report["trainable_parameters"],
    "direct_inference_rows": int(len(FROZEN_DIRECT_PREDICTIONS)),
    "role_metrics_match_saved_baseline": bool(
        frozen_direct_comparison["matches_saved_baseline"].all()
    ),
    "ready_for_training_helpers": bool(
        frozen_direct_comparison["matches_saved_baseline"].all()
        and parameter_report["trainable_parameters"] == 0
    ),
}

print("Frozen checkpoint report:")
print(json.dumps(checkpoint_report, indent=2))

print("\nDirect frozen checkpoint role summary:")
display(frozen_direct_role_summary)

print("\nDirect frozen checkpoint versus saved source-FPS baseline:")
display(frozen_direct_comparison)

print("\nDirect frozen checkpoint by dataset and role:")
display(frozen_direct_dataset_role_summary)

print("\nNB12 model reproduction report:")
print(json.dumps(NB12_MODEL_REPRODUCTION, indent=2))

Frozen checkpoint report:
{
  "checkpoint_input_mode": "multichannel",
  "checkpoint_best_val_mae": 6.900240182876587,
  "checkpoint_best_n_epochs": 50,
  "model_config": {
    "in_channels": 3,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "dim_feedforward": 256,
    "dropout": 0.11331939348791525,
    "hr_min": 40.0,
    "hr_max": 180.0,
    "target_frames": 240,
    "max_positional_length": 300
  },
  "total_parameters": 322145,
  "trainable_parameters": 0,
  "device": "cuda"
}

Direct frozen checkpoint role summary:


,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean
0,finetune_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,54.9838,1.5658
1,finetune_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,50.2965,1.4724
2,finetune_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,72.2401,1.4174
3,ood_eval,882,109.2048,101.2561,14.0355,7.6838,38.1498,49.9899,89.9549,-7.9487



Direct frozen checkpoint versus saved source-FPS baseline:


,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,...,saved_mae_p95,saved_mae_max,saved_signed_error_mean,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_mae_max,delta_signed_error_mean,matches_saved_baseline
0,finetune_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,72.2401,1.4174,...,19.0461,72.2401,1.4174,0.0,0.0000,0.0000,0.0,0.0,0.0,True
1,finetune_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,54.9838,1.5658,...,18.9426,54.9838,1.5658,0.0,0.0000,0.0000,0.0,0.0,0.0,True
2,finetune_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,50.2965,1.4724,...,19.2319,50.2965,1.4724,0.0,0.0000,0.0000,0.0,0.0,0.0,True
3,ood_eval,882,109.2048,101.2561,14.0355,7.6838,38.1498,49.9899,89.9549,-7.9487,...,49.9899,89.9549,-7.9487,0.0,-0.0001,-0.0001,0.0,0.0,0.0,True



Direct frozen checkpoint by dataset and role:


,dataset,window_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean
0,mcd_rppg,finetune_test,2094,79.853302,80.777496,4.8847,3.1502,10.9700,15.5490,49.472198,0.9242
1,ubfc_phys,finetune_test,658,77.420601,80.618500,8.8584,5.6735,20.2862,25.6733,72.240097,3.1980
2,ubfc_rppg,finetune_test,76,98.269302,97.859100,3.4085,3.1453,6.0835,6.8394,13.533900,-0.4102
3,mcd_rppg,finetune_train,7446,79.064796,80.329399,4.6876,3.1639,10.3722,13.9969,54.983799,1.2645
4,ubfc_phys,finetune_train,4595,80.305496,82.624603,7.7819,5.4248,18.5928,23.4842,50.982700,2.3192
5,ubfc_rppg,finetune_train,462,98.449203,97.376900,4.6548,3.1447,9.9189,13.1687,30.568399,-1.0723
6,mcd_rppg,finetune_val,1743,79.108597,80.374298,5.1325,3.3746,11.5368,15.4784,46.608799,1.2656
7,ubfc_phys,finetune_val,986,80.002403,82.285004,7.7629,4.8461,18.2725,24.2656,50.296501,2.2826
8,ubfc_rppg,finetune_val,110,98.277199,95.764702,4.6533,2.4438,12.3774,15.4017,23.514700,-2.5125
9,ecg_fitness,ood_eval,882,109.204803,101.256104,14.0355,7.6838,38.1498,49.9899,89.954903,-7.9487



NB12 model reproduction report:
{
  "model_loaded": true,
  "model_config": {
    "in_channels": 3,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "dim_feedforward": 256,
    "dropout": 0.11331939348791525,
    "hr_min": 40.0,
    "hr_max": 180.0,
    "target_frames": 240,
    "max_positional_length": 300
  },
  "total_parameters": 322145,
  "trainable_parameters": 0,
  "direct_inference_rows": 19052,
  "role_metrics_match_saved_baseline": true,
  "ready_for_training_helpers": true
}


## NB12 Candidate Decision Policy

This section defines how NB12 candidate models will be judged before any Optuna search is started.

The policy separates three questions:

1. Does the candidate improve live-compatible in-distribution validation and test performance?
2. Does the candidate preserve ECG-Fitness behavior when ECG-Fitness is excluded from training?
3. Does the candidate worsen high-HR or exercise-like underprediction?

ECG-Fitness remains an out-of-domain exercise/high-HR stress test in the first NB12 branch because it is not included in training.

A candidate should not be considered useful only because validation MAE improves. It should also remain stable on held-out test windows and should not hide or worsen high-HR underprediction.

In [4]:
NB12_DECISION_THRESHOLDS = {
    "minimum_val_gain_bpm": 0.20,
    "minimum_test_gain_bpm": 0.10,
    "maximum_ood_penalty_bpm": 0.50,
    "maximum_high_hr_mae_penalty_bpm": 0.25,
    "maximum_severe_underprediction_rate_increase": 0.02,
    "high_hr_threshold_bpm": 100.0,
    "exercise_tachy_threshold_bpm": 120.0,
    "severe_underprediction_bpm": -10.0,
}


def assert_decision_policy_prerequisites() -> None:
    """Check that frozen model reproduction has been completed."""
    required_names = [
        "FROZEN_DIRECT_PREDICTIONS",
        "NB12_MODEL_REPRODUCTION",
        "ROLE_ORDER",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen checkpoint reproduction cell before this policy cell. "
            f"Missing names: {missing}"
        )


def prepare_hr_error_table(
    predictions: pd.DataFrame,
    experiment_name: str,
    target_col: str = "target_hr_from_tensor",
    pred_col: str = "notebook_model_hr",
) -> pd.DataFrame:
    """Add standard HR error columns used by NB12 decision metrics."""
    required_cols = [target_col, pred_col, "role", "dataset"]

    missing_cols = [col for col in required_cols if col not in predictions.columns]

    if missing_cols:
        raise KeyError(
            f"Prediction table is missing required columns: {missing_cols}. "
            f"Available columns: {list(predictions.columns)}"
        )

    table = predictions.copy()
    table["experiment"] = experiment_name
    table["target_hr"] = table[target_col].astype(float)
    table["pred_hr"] = table[pred_col].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    table["is_high_hr_ge_100"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    )
    table["is_exercise_tachy_ge_120"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]
    )
    table["is_severe_underprediction"] = (
        table["signed_error"] <= NB12_DECISION_THRESHOLDS["severe_underprediction_bpm"]
    )

    return table


def summarize_group_errors(error_table: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize absolute error, signed error, and severe underprediction by group."""
    summary = (
        error_table
        .groupby(group_cols, dropna=False)
        .agg(
            rows=("abs_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda values: values.quantile(0.90)),
            mae_p95=("abs_error", lambda values: values.quantile(0.95)),
            mae_max=("abs_error", "max"),
            signed_error_mean=("signed_error", "mean"),
            severe_underprediction_rate=("is_severe_underprediction", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def summarize_error_slice(
    error_table: pd.DataFrame,
    slice_name: str,
    mask: pd.Series,
) -> dict:
    """Summarize one named analysis slice."""
    part = error_table.loc[mask].copy()

    row = {
        "slice": slice_name,
        "rows": int(len(part)),
    }

    if len(part) == 0:
        row.update(
            {
                "target_hr_mean": np.nan,
                "pred_hr_mean": np.nan,
                "mae_mean": np.nan,
                "mae_p90": np.nan,
                "signed_error_mean": np.nan,
                "severe_underprediction_rate": np.nan,
            }
        )
        return row

    row.update(
        {
            "target_hr_mean": float(part["target_hr"].mean()),
            "pred_hr_mean": float(part["pred_hr"].mean()),
            "mae_mean": float(part["abs_error"].mean()),
            "mae_p90": float(part["abs_error"].quantile(0.90)),
            "signed_error_mean": float(part["signed_error"].mean()),
            "severe_underprediction_rate": float(part["is_severe_underprediction"].mean()),
        }
    )

    return row


def build_slice_summary(error_table: pd.DataFrame) -> pd.DataFrame:
    """Build NB12 high-HR and ECG-Fitness slice summaries."""
    all_rows = pd.Series(True, index=error_table.index)
    high_hr = error_table["is_high_hr_ge_100"]
    exercise_tachy = error_table["is_exercise_tachy_ge_120"]
    ecg_fitness = error_table["role"].eq("ood_eval")

    rows = [
        summarize_error_slice(error_table, "all_windows", all_rows),
        summarize_error_slice(error_table, "high_hr_ge_100", high_hr),
        summarize_error_slice(error_table, "exercise_tachy_ge_120", exercise_tachy),
        summarize_error_slice(error_table, "ecg_fitness_ood", ecg_fitness),
        summarize_error_slice(error_table, "ecg_fitness_high_hr_ge_100", ecg_fitness & high_hr),
        summarize_error_slice(error_table, "ecg_fitness_exercise_tachy_ge_120", ecg_fitness & exercise_tachy),
    ]

    summary = pd.DataFrame(rows)
    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def sort_role_summary(summary: pd.DataFrame) -> pd.DataFrame:
    """Sort role summaries in the standard NB12 role order."""
    role_order = {role: index for index, role in enumerate(ROLE_ORDER)}
    table = summary.copy()
    table["role_order"] = table["role"].map(role_order).fillna(999).astype(int)
    return table.sort_values(["role_order", "role"]).drop(columns=["role_order"]).reset_index(drop=True)


def extract_summary_metric(
    summary: pd.DataFrame,
    key_col: str,
    key_value: str,
    metric_col: str,
) -> float:
    """Extract one metric from a summary table."""
    rows = summary.loc[summary[key_col] == key_value]

    if len(rows) != 1:
        raise ValueError(
            f"Expected one row for {key_col}={key_value}, found {len(rows)}."
        )

    return float(rows.iloc[0][metric_col])


def build_decision_metric_row(
    role_summary: pd.DataFrame,
    slice_summary: pd.DataFrame,
    experiment_name: str,
) -> dict:
    """Flatten role and slice summaries into one decision row."""
    return {
        "experiment": experiment_name,
        "val_mae_mean": extract_summary_metric(role_summary, "role", "finetune_val", "mae_mean"),
        "test_mae_mean": extract_summary_metric(role_summary, "role", "finetune_test", "mae_mean"),
        "ood_mae_mean": extract_summary_metric(role_summary, "role", "ood_eval", "mae_mean"),
        "val_mae_p90": extract_summary_metric(role_summary, "role", "finetune_val", "mae_p90"),
        "test_mae_p90": extract_summary_metric(role_summary, "role", "finetune_test", "mae_p90"),
        "ood_mae_p90": extract_summary_metric(role_summary, "role", "ood_eval", "mae_p90"),
        "high_hr_mae_mean": extract_summary_metric(slice_summary, "slice", "high_hr_ge_100", "mae_mean"),
        "high_hr_signed_error_mean": extract_summary_metric(slice_summary, "slice", "high_hr_ge_100", "signed_error_mean"),
        "high_hr_severe_underprediction_rate": extract_summary_metric(
            slice_summary,
            "slice",
            "high_hr_ge_100",
            "severe_underprediction_rate",
        ),
        "ecg_fitness_mae_mean": extract_summary_metric(slice_summary, "slice", "ecg_fitness_ood", "mae_mean"),
        "ecg_fitness_signed_error_mean": extract_summary_metric(
            slice_summary,
            "slice",
            "ecg_fitness_ood",
            "signed_error_mean",
        ),
        "ecg_fitness_severe_underprediction_rate": extract_summary_metric(
            slice_summary,
            "slice",
            "ecg_fitness_ood",
            "severe_underprediction_rate",
        ),
    }


def build_experiment_metric_bundle(
    predictions: pd.DataFrame,
    experiment_name: str,
) -> dict:
    """Build all NB12 decision summaries for one experiment."""
    error_table = prepare_hr_error_table(
        predictions=predictions,
        experiment_name=experiment_name,
    )

    role_summary = sort_role_summary(
        summarize_group_errors(error_table, ["role"])
    )

    dataset_role_summary = summarize_group_errors(
        error_table,
        ["dataset", "role"],
    ).sort_values(["role", "dataset"]).reset_index(drop=True)

    slice_summary = build_slice_summary(error_table)

    decision_metrics = build_decision_metric_row(
        role_summary=role_summary,
        slice_summary=slice_summary,
        experiment_name=experiment_name,
    )

    return {
        "error_table": error_table,
        "role_summary": role_summary,
        "dataset_role_summary": dataset_role_summary,
        "slice_summary": slice_summary,
        "decision_metrics": decision_metrics,
    }


def assign_nb12_candidate_status(decision_row: dict) -> str:
    """Assign a cautious NB12 candidate status from baseline-relative deltas."""
    thresholds = NB12_DECISION_THRESHOLDS

    if (
        decision_row["val_delta_mean"] <= -thresholds["minimum_val_gain_bpm"]
        and decision_row["test_delta_mean"] <= -thresholds["minimum_test_gain_bpm"]
        and decision_row["ood_delta_mean"] <= thresholds["maximum_ood_penalty_bpm"]
        and decision_row["high_hr_delta_mae"] <= thresholds["maximum_high_hr_mae_penalty_bpm"]
        and decision_row["high_hr_severe_underprediction_rate_delta"]
        <= thresholds["maximum_severe_underprediction_rate_increase"]
    ):
        return "research_candidate_for_review"

    if decision_row["val_delta_mean"] < 0 and decision_row["test_delta_mean"] < 0:
        return "analysis_only_tradeoff"

    return "reject_or_debug"


def compare_candidate_to_frozen_metrics(
    candidate_metrics: dict,
    frozen_metrics: dict,
) -> dict:
    """Compare one future candidate against the frozen baseline decision metrics."""
    row = dict(candidate_metrics)

    row["val_delta_mean"] = candidate_metrics["val_mae_mean"] - frozen_metrics["val_mae_mean"]
    row["test_delta_mean"] = candidate_metrics["test_mae_mean"] - frozen_metrics["test_mae_mean"]
    row["ood_delta_mean"] = candidate_metrics["ood_mae_mean"] - frozen_metrics["ood_mae_mean"]

    row["high_hr_delta_mae"] = (
        candidate_metrics["high_hr_mae_mean"] - frozen_metrics["high_hr_mae_mean"]
    )
    row["high_hr_signed_error_delta"] = (
        candidate_metrics["high_hr_signed_error_mean"]
        - frozen_metrics["high_hr_signed_error_mean"]
    )
    row["high_hr_severe_underprediction_rate_delta"] = (
        candidate_metrics["high_hr_severe_underprediction_rate"]
        - frozen_metrics["high_hr_severe_underprediction_rate"]
    )

    row["in_distribution_mean_gain"] = -0.5 * (
        row["val_delta_mean"] + row["test_delta_mean"]
    )

    row["decision_status"] = assign_nb12_candidate_status(row)

    return row


assert_decision_policy_prerequisites()

FROZEN_DECISION_BUNDLE = build_experiment_metric_bundle(
    predictions=FROZEN_DIRECT_PREDICTIONS,
    experiment_name="frozen_sourcefps_baseline",
)

FROZEN_DECISION_METRICS = FROZEN_DECISION_BUNDLE["decision_metrics"]

threshold_table = pd.DataFrame(
    [
        {"threshold": key, "value": value}
        for key, value in NB12_DECISION_THRESHOLDS.items()
    ]
)

frozen_reference_row = dict(FROZEN_DECISION_METRICS)
frozen_reference_row.update(
    {
        "val_delta_mean": 0.0,
        "test_delta_mean": 0.0,
        "ood_delta_mean": 0.0,
        "high_hr_delta_mae": 0.0,
        "high_hr_signed_error_delta": 0.0,
        "high_hr_severe_underprediction_rate_delta": 0.0,
        "in_distribution_mean_gain": 0.0,
        "decision_status": "frozen_reference",
    }
)

frozen_decision_reference = pd.DataFrame([frozen_reference_row])
numeric_cols = frozen_decision_reference.select_dtypes(include=[np.number]).columns
frozen_decision_reference[numeric_cols] = frozen_decision_reference[numeric_cols].round(4)

NB12_DECISION_POLICY_REPORT = {
    "decision_policy_defined": True,
    "frozen_reference_experiment": "frozen_sourcefps_baseline",
    "ecg_fitness_policy": "excluded_from_first_search_branch_and_treated_as_ood_stress_test",
    "ready_for_optuna_setup": bool(NB12_MODEL_REPRODUCTION["ready_for_training_helpers"]),
}

print("NB12 decision thresholds:")
display(threshold_table)

print("\nFrozen baseline role decision summary:")
display(FROZEN_DECISION_BUNDLE["role_summary"])

print("\nFrozen baseline high-HR and ECG-Fitness slices:")
display(FROZEN_DECISION_BUNDLE["slice_summary"])

print("\nFrozen baseline decision reference row:")
display(frozen_decision_reference)

print("\nNB12 decision policy report:")
print(json.dumps(NB12_DECISION_POLICY_REPORT, indent=2))

NB12 decision thresholds:


,threshold,value
0,minimum_val_gain_bpm,0.20
1,minimum_test_gain_bpm,0.10
2,maximum_ood_penalty_bpm,0.50
3,maximum_high_hr_mae_penalty_bpm,0.25
4,maximum_severe_underprediction_rate_increase,0.02
5,high_hr_threshold_bpm,100.00
6,exercise_tachy_threshold_bpm,120.00
7,severe_underprediction_bpm,-10.00



Frozen baseline role decision summary:


,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,finetune_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,54.9838,1.5658,0.0571
1,finetune_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,50.2965,1.4724,0.0683
2,finetune_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,72.2401,1.4174,0.0622
3,ood_eval,882,109.2048,101.2561,14.0355,7.6838,38.1498,49.9899,89.9549,-7.9487,0.3061



Frozen baseline high-HR and ECG-Fitness slices:


,slice,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_p90,signed_error_mean,severe_underprediction_rate
0,all_windows,19052,81.4994,82.5887,6.2261,14.5286,1.0894,0.0711
1,high_hr_ge_100,2125,111.5762,104.4080,10.6511,25.1903,-7.1682,0.3026
2,exercise_tachy_ge_120,350,133.0930,118.3619,17.8299,54.2710,-14.7311,0.3829
3,ecg_fitness_ood,882,109.2048,101.2561,14.0355,38.1498,-7.9487,0.3061
4,ecg_fitness_high_hr_ge_100,584,122.0550,110.8209,17.3247,44.4023,-11.2341,0.3904
5,ecg_fitness_exercise_tachy_ge_120,275,135.6331,118.2178,20.4630,54.8206,-17.4154,0.4327



Frozen baseline decision reference row:


,experiment,val_mae_mean,test_mae_mean,ood_mae_mean,val_mae_p90,test_mae_p90,ood_mae_p90,high_hr_mae_mean,high_hr_signed_error_mean,high_hr_severe_underprediction_rate,...,ecg_fitness_signed_error_mean,ecg_fitness_severe_underprediction_rate,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,decision_status
0,frozen_sourcefps_baseline,6.0275,5.7696,14.0355,14.0701,13.6984,38.1498,10.6511,-7.1682,0.3026,...,-7.9487,0.3061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,frozen_reference



NB12 decision policy report:
{
  "decision_policy_defined": true,
  "frozen_reference_experiment": "frozen_sourcefps_baseline",
  "ecg_fitness_policy": "excluded_from_first_search_branch_and_treated_as_ood_stress_test",
  "ready_for_optuna_setup": true
}


## Optuna Setup And First Search Branch

This section checks whether Optuna is available in the Kaggle runtime and defines the first NB12 search branch.

The first branch keeps ECG-Fitness excluded from training. ECG-Fitness remains an out-of-domain exercise/high-HR stress test.

The first practical search target is transfer learning from the frozen checkpoint, not scratch training. NB11 showed that scratch training was not competitive, while transfer learning produced small but real in-distribution gains.

This section only defines the search configuration and hyperparameter space. It does not start the search yet.

In [5]:
def check_optuna_availability() -> dict:
    """Check whether Optuna is available in the current Kaggle runtime."""
    try:
        import optuna

        return {
            "available": True,
            "version": optuna.__version__,
            "error": None,
        }
    except Exception as exc:
        return {
            "available": False,
            "version": None,
            "error": repr(exc),
        }


OPTUNA_STATUS = check_optuna_availability()

NB12_FIRST_BRANCH_CONFIG = {
    "branch_name": "transfer_exclude_ecg_fitness",
    "training_policy": "transfer_learning_from_frozen_checkpoint",
    "ecg_fitness_policy": "excluded_from_training_and_used_as_ood_stress_test",
    "train_role": "finetune_train",
    "validation_role": "finetune_val",
    "test_role": "finetune_test",
    "ood_role": "ood_eval",
    "primary_optimization_metric": "risk_adjusted_validation_score",
    "baseline_reference": "frozen_sourcefps_baseline",
    "why_transfer_first": (
        "NB11 showed that scratch PhysFormer training was not competitive, while "
        "full-model transfer produced small in-distribution gains. NB12 therefore "
        "starts with transfer search before spending compute on scratch search."
    ),
}


NB12_TRANSFER_SEARCH_SPACE = {
    "lr": {
        "type": "float_log",
        "low": 5e-6,
        "high": 2e-4,
        "reason": "Transfer learning should use smaller learning rates than scratch training.",
    },
    "weight_decay": {
        "type": "float_log",
        "low": 1e-6,
        "high": 5e-3,
        "reason": "Regularization may reduce over-adaptation to the dominant training datasets.",
    },
    "dropout": {
        "type": "float",
        "low": 0.05,
        "high": 0.30,
        "reason": "Dropout controls model adaptation strength and overfitting risk.",
    },
    "huber_delta": {
        "type": "float",
        "low": 3.0,
        "high": 12.0,
        "reason": "Huber loss controls sensitivity to large HR errors and unstable labels.",
    },
    "batch_size": {
        "type": "categorical",
        "choices": [128, 256],
        "reason": "Batch size affects optimization stability and GPU memory use.",
    },
    "grad_clip": {
        "type": "float",
        "low": 1.0,
        "high": 8.0,
        "reason": "Gradient clipping protects transfer runs from unstable updates.",
    },
    "freeze_encoder": {
        "type": "categorical",
        "choices": [False, True],
        "reason": "Freezing the CNN encoder tests whether lower-level temporal filters should remain stable.",
    },
    "freeze_batchnorm": {
        "type": "categorical",
        "choices": [False, True],
        "reason": "NB11 showed BatchNorm freezing alone did not solve OOD shift, but it remains worth searching jointly.",
    },
    "use_balanced_sampler": {
        "type": "categorical",
        "choices": [False, True],
        "reason": "Balanced sampling tests whether dominant datasets are steering adaptation too strongly.",
    },
}


def build_search_space_table(search_space: dict) -> pd.DataFrame:
    """Convert a search-space dictionary into a readable table."""
    rows = []

    for name, spec in search_space.items():
        row = {
            "parameter": name,
            "type": spec["type"],
            "low": spec.get("low"),
            "high": spec.get("high"),
            "choices": spec.get("choices"),
            "reason": spec["reason"],
        }
        rows.append(row)

    return pd.DataFrame(rows)


def validate_first_branch_config() -> dict:
    """Validate that the first NB12 search branch is consistent with prior audit cells."""
    required_ready_flags = {
        "artifact_startup_ready": startup_report["ready_for_baseline_reproduction"],
        "baseline_reproduction_ready": NB12_BASELINE_REPRODUCTION["ready_for_model_definition_check"],
        "model_reproduction_ready": NB12_MODEL_REPRODUCTION["ready_for_training_helpers"],
        "decision_policy_ready": NB12_DECISION_POLICY_REPORT["ready_for_optuna_setup"],
    }

    return {
        **required_ready_flags,
        "optuna_available": bool(OPTUNA_STATUS["available"]),
        "ready_to_define_objective": bool(
            all(required_ready_flags.values())
            and OPTUNA_STATUS["available"]
        ),
    }


search_space_table = build_search_space_table(NB12_TRANSFER_SEARCH_SPACE)
NB12_FIRST_BRANCH_READINESS = validate_first_branch_config()

print("Optuna status:")
print(json.dumps(OPTUNA_STATUS, indent=2))

print("\nNB12 first branch config:")
print(json.dumps(NB12_FIRST_BRANCH_CONFIG, indent=2))

print("\nNB12 transfer search space:")
display(search_space_table)

print("\nNB12 first branch readiness:")
print(json.dumps(NB12_FIRST_BRANCH_READINESS, indent=2))

Optuna status:
{
  "available": true,
  "version": "4.9.0",
  "error": null
}

NB12 first branch config:
{
  "branch_name": "transfer_exclude_ecg_fitness",
  "training_policy": "transfer_learning_from_frozen_checkpoint",
  "ecg_fitness_policy": "excluded_from_training_and_used_as_ood_stress_test",
  "train_role": "finetune_train",
  "validation_role": "finetune_val",
  "test_role": "finetune_test",
  "ood_role": "ood_eval",
  "primary_optimization_metric": "risk_adjusted_validation_score",
  "baseline_reference": "frozen_sourcefps_baseline",
  "why_transfer_first": "NB11 showed that scratch PhysFormer training was not competitive, while full-model transfer produced small in-distribution gains. NB12 therefore starts with transfer search before spending compute on scratch search."
}

NB12 transfer search space:


,parameter,type,low,high,choices,reason
0,lr,float_log,0.000005,0.0002,None,Transfer learning should use smaller learning rates than scratch training.
1,weight_decay,float_log,0.000001,0.0050,None,Regularization may reduce over-adaptation to the dominant training datasets.
2,dropout,float,0.050000,0.3000,None,Dropout controls model adaptation strength and overfitting risk.
3,huber_delta,float,3.000000,12.0000,None,Huber loss controls sensitivity to large HR errors and unstable labels.
4,batch_size,categorical,NaN,NaN,"[128, 256]",Batch size affects optimization stability and GPU memory use.
5,grad_clip,float,1.000000,8.0000,None,Gradient clipping protects transfer runs from unstable updates.
6,freeze_encoder,categorical,NaN,NaN,"[False, True]",Freezing the CNN encoder tests whether lower-level temporal filters should remain stable.
7,freeze_batchnorm,categorical,NaN,NaN,"[False, True]","NB11 showed BatchNorm freezing alone did not solve OOD shift, but it remains worth searching jointly."
8,use_balanced_sampler,categorical,NaN,NaN,"[False, True]",Balanced sampling tests whether dominant datasets are steering adaptation too strongly.



NB12 first branch readiness:
{
  "artifact_startup_ready": true,
  "baseline_reproduction_ready": true,
  "model_reproduction_ready": true,
  "decision_policy_ready": true,
  "optuna_available": true,
  "ready_to_define_objective": true
}


## One-Trial Optuna Smoke Test

This section defines the transfer-learning objective and runs one small Optuna smoke trial.

The smoke trial is not intended to produce a useful checkpoint. It uses a small training subset and only a few epochs.

The purpose is to verify that the training objective works end-to-end before running a real search.

The Optuna objective uses validation behavior for optimization. Test and ECG-Fitness results are reported for audit, but they should not be treated as tuned selection metrics in this first smoke test.

In [6]:
SMOKE_ROWS_PER_DATASET = 256
SMOKE_MAX_EPOCHS = 2
FULL_SEARCH_MAX_EPOCHS = 12
FULL_SEARCH_PATIENCE = 4
TRIAL_MIN_DELTA = 1e-3

NB12_VALIDATION_RISK_WEIGHTS = {
    "p90_worsening": 0.10,
    "severe_underprediction_rate_worsening": 5.0,
}


class IndexedTensorDataset(Dataset):
    """Dataset view over selected row indices from a materialized tensor dataset."""

    def __init__(
        self,
        base_dataset: MaterializedTensorDataset,
        indices: np.ndarray,
        role: str | None = None,
    ) -> None:
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.role = role or base_dataset.role
        self.metadata = base_dataset.metadata.iloc[self.indices].reset_index(drop=True)

        if len(self.indices) == 0:
            raise ValueError("IndexedTensorDataset received no indices.")

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.base_dataset[int(self.indices[index])]


def select_smoke_indices_by_dataset(
    dataset: MaterializedTensorDataset,
    rows_per_dataset: int,
    seed: int,
) -> np.ndarray:
    """Select a small dataset-balanced subset for objective smoke testing."""
    if "dataset" not in dataset.metadata.columns:
        raise KeyError("Training metadata is missing required column: dataset")

    rng = np.random.default_rng(seed)
    selected_indices = []

    metadata = dataset.metadata.copy()
    metadata["_row_position"] = np.arange(len(metadata))

    for _, group in metadata.groupby("dataset", sort=True):
        positions = group["_row_position"].to_numpy()
        take = min(rows_per_dataset, len(positions))
        chosen = rng.choice(positions, size=take, replace=False)
        selected_indices.extend(chosen.tolist())

    return np.asarray(sorted(selected_indices), dtype=np.int64)


def make_dataset_balanced_sampler(
    dataset: Dataset,
    seed: int,
) -> WeightedRandomSampler:
    """Create an inverse-frequency dataset sampler using metadata dataset labels."""
    if not hasattr(dataset, "metadata"):
        raise AttributeError("Balanced sampler requires dataset.metadata.")

    metadata = dataset.metadata

    if "dataset" not in metadata.columns:
        raise KeyError("Dataset metadata is missing required column: dataset")

    counts = metadata["dataset"].value_counts()
    weights = metadata["dataset"].map(lambda name: 1.0 / float(counts[name])).to_numpy()
    weights = torch.as_tensor(weights, dtype=torch.double)

    generator = torch.Generator()
    generator.manual_seed(seed)

    return WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )


def make_transfer_training_dataset(smoke_mode: bool) -> Dataset:
    """Return the training dataset for smoke or full transfer search."""
    train_dataset = DATASETS["finetune_train"]

    if not smoke_mode:
        return train_dataset

    smoke_indices = select_smoke_indices_by_dataset(
        dataset=train_dataset,
        rows_per_dataset=SMOKE_ROWS_PER_DATASET,
        seed=SEED,
    )

    return IndexedTensorDataset(
        base_dataset=train_dataset,
        indices=smoke_indices,
        role="finetune_train_smoke",
    )


def make_transfer_training_loader(
    dataset: Dataset,
    batch_size: int,
    use_balanced_sampler: bool,
    seed: int,
) -> DataLoader:
    """Create a training DataLoader with optional dataset-balanced sampling."""
    generator = torch.Generator()
    generator.manual_seed(seed)

    if use_balanced_sampler:
        sampler = make_dataset_balanced_sampler(dataset, seed=seed)

        return DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=sampler,
            shuffle=False,
            num_workers=0,
            pin_memory=(DEVICE.type == "cuda"),
        )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


def freeze_batchnorm_layers(model: nn.Module) -> list[str]:
    """Freeze BatchNorm1d parameters and keep their running statistics fixed."""
    frozen_names = []

    for module_name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()
            frozen_names.append(module_name)

            for parameter in module.parameters():
                parameter.requires_grad = False

    return frozen_names


def apply_transfer_freezing(
    model: nn.Module,
    freeze_encoder: bool,
    freeze_batchnorm: bool,
) -> dict:
    """Apply trial-specific freezing choices to a transfer model."""
    frozen_encoder = False
    frozen_batchnorm_names = []

    if freeze_encoder:
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False
        frozen_encoder = True

    if freeze_batchnorm:
        frozen_batchnorm_names = freeze_batchnorm_layers(model)

    return {
        "freeze_encoder": bool(frozen_encoder),
        "freeze_batchnorm": bool(freeze_batchnorm),
        "frozen_batchnorm_names": frozen_batchnorm_names,
    }


def enforce_transfer_training_modes(
    model: nn.Module,
    freeze_encoder: bool,
    freeze_batchnorm: bool,
) -> None:
    """Restore eval mode for frozen modules after model.train() is called."""
    if freeze_encoder:
        model.encoder.eval()

    if freeze_batchnorm:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm1d):
                module.eval()


def build_transfer_trial_model(params: dict) -> tuple[CRVSEPhysFormer, dict]:
    """Build a checkpoint-initialized model for one transfer-learning trial."""
    model_config = dict(FROZEN_MODEL_CONFIG)
    model_config["dropout"] = float(params["dropout"])

    model = CRVSEPhysFormer(**model_config)
    model.load_state_dict(FROZEN_CHECKPOINT["model_state"], strict=True)
    model.to(DEVICE)

    freezing_report = apply_transfer_freezing(
        model=model,
        freeze_encoder=bool(params["freeze_encoder"]),
        freeze_batchnorm=bool(params["freeze_batchnorm"]),
    )

    trainable_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if len(trainable_parameters) == 0:
        raise ValueError("Trial produced a model with no trainable parameters.")

    freezing_report.update(count_model_parameters(model))

    return model, freezing_report


def suggest_transfer_trial_params(trial: optuna.Trial) -> dict:
    """Suggest one transfer-learning hyperparameter set."""
    return {
        "lr": trial.suggest_float("lr", 5e-6, 2e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.05, 0.30),
        "huber_delta": trial.suggest_float("huber_delta", 3.0, 12.0),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
        "grad_clip": trial.suggest_float("grad_clip", 1.0, 8.0),
        "freeze_encoder": trial.suggest_categorical("freeze_encoder", [False, True]),
        "freeze_batchnorm": trial.suggest_categorical("freeze_batchnorm", [False, True]),
        "use_balanced_sampler": trial.suggest_categorical("use_balanced_sampler", [False, True]),
    }


def train_one_transfer_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    grad_clip: float,
    freeze_encoder: bool,
    freeze_batchnorm: bool,
) -> dict:
    """Train one transfer-learning epoch and return aggregate training metrics."""
    model.train()
    enforce_transfer_training_modes(
        model=model,
        freeze_encoder=freeze_encoder,
        freeze_batchnorm=freeze_batchnorm,
    )

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        trainable_parameters = [
            parameter for parameter in model.parameters()
            if parameter.requires_grad
        ]

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=float(grad_clip),
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def evaluate_model_on_all_roles(
    model: nn.Module,
    experiment_name: str,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate a model on all NB12 roles and build decision summaries."""
    prediction_frames = []

    for role in ROLE_ORDER:
        role_predictions, _ = evaluate_hr_model(
            model=model,
            dataset=DATASETS[role],
            loader=EVAL_LOADERS[role],
        )
        prediction_frames.append(role_predictions)

    predictions = pd.concat(prediction_frames, axis=0).reset_index(drop=True)

    bundle = build_experiment_metric_bundle(
        predictions=predictions,
        experiment_name=experiment_name,
    )

    return predictions, bundle


def extract_validation_reference() -> dict:
    """Extract frozen validation metrics used by the validation risk score."""
    role_summary = FROZEN_DECISION_BUNDLE["role_summary"]

    return {
        "val_mae_mean": extract_summary_metric(role_summary, "role", "finetune_val", "mae_mean"),
        "val_mae_p90": extract_summary_metric(role_summary, "role", "finetune_val", "mae_p90"),
        "val_severe_underprediction_rate": extract_summary_metric(
            role_summary,
            "role",
            "finetune_val",
            "severe_underprediction_rate",
        ),
    }


FROZEN_VALIDATION_REFERENCE = extract_validation_reference()


def compute_validation_risk_score(role_summary: pd.DataFrame) -> dict:
    """Compute the validation-only score minimized by Optuna."""
    val_mae = extract_summary_metric(role_summary, "role", "finetune_val", "mae_mean")
    val_p90 = extract_summary_metric(role_summary, "role", "finetune_val", "mae_p90")
    val_under_rate = extract_summary_metric(
        role_summary,
        "role",
        "finetune_val",
        "severe_underprediction_rate",
    )

    p90_worsening = max(0.0, val_p90 - FROZEN_VALIDATION_REFERENCE["val_mae_p90"])
    under_rate_worsening = max(
        0.0,
        val_under_rate - FROZEN_VALIDATION_REFERENCE["val_severe_underprediction_rate"],
    )

    risk_score = (
        val_mae
        + NB12_VALIDATION_RISK_WEIGHTS["p90_worsening"] * p90_worsening
        + NB12_VALIDATION_RISK_WEIGHTS["severe_underprediction_rate_worsening"]
        * under_rate_worsening
    )

    return {
        "validation_risk_score": float(risk_score),
        "val_mae_mean": float(val_mae),
        "val_mae_p90": float(val_p90),
        "val_severe_underprediction_rate": float(val_under_rate),
        "val_p90_worsening": float(p90_worsening),
        "val_severe_underprediction_rate_worsening": float(under_rate_worsening),
    }


def build_trial_summary_row(
    trial_number: int,
    params: dict,
    freezing_report: dict,
    history: list[dict],
    bundle: dict,
    validation_risk: dict,
    smoke_mode: bool,
) -> dict:
    """Build one compact trial summary row for review."""
    candidate_metrics = bundle["decision_metrics"]
    comparison_row = compare_candidate_to_frozen_metrics(
        candidate_metrics=candidate_metrics,
        frozen_metrics=FROZEN_DECISION_METRICS,
    )

    summary = {
        "trial_number": int(trial_number),
        "smoke_mode": bool(smoke_mode),
        "epochs_run": int(len(history)),
        **params,
        **freezing_report,
        **validation_risk,
        **comparison_row,
    }

    return summary


def run_transfer_trial(
    trial: optuna.Trial,
    smoke_mode: bool,
) -> dict:
    """Run one transfer-learning trial and return its full NB12 summary."""
    params = suggest_transfer_trial_params(trial)

    max_epochs = SMOKE_MAX_EPOCHS if smoke_mode else FULL_SEARCH_MAX_EPOCHS
    patience = max_epochs if smoke_mode else FULL_SEARCH_PATIENCE

    training_dataset = make_transfer_training_dataset(smoke_mode=smoke_mode)
    training_loader = make_transfer_training_loader(
        dataset=training_dataset,
        batch_size=int(params["batch_size"]),
        use_balanced_sampler=bool(params["use_balanced_sampler"]),
        seed=SEED + trial.number,
    )

    model, freezing_report = build_transfer_trial_model(params)

    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"]),
    )

    loss_fn = nn.HuberLoss(delta=float(params["huber_delta"]))

    best_val_mae = math.inf
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_metrics = train_one_transfer_epoch(
            model=model,
            loader=training_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            grad_clip=float(params["grad_clip"]),
            freeze_encoder=bool(params["freeze_encoder"]),
            freeze_batchnorm=bool(params["freeze_batchnorm"]),
        )

        val_predictions, val_metrics = evaluate_hr_model(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=EVAL_LOADERS["finetune_val"],
        )

        improved = val_metrics["mae_mean"] < (best_val_mae - TRIAL_MIN_DELTA)

        if improved:
            best_val_mae = float(val_metrics["mae_mean"])
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_row = {
            "epoch": epoch,
            **train_metrics,
            "val_mae": float(val_metrics["mae_mean"]),
            "val_p90": float(val_metrics["mae_p90"]),
            "best_epoch_so_far": int(best_epoch),
            "best_val_mae_so_far": float(best_val_mae),
            "improved": bool(improved),
        }
        history.append(history_row)

        trial.report(float(val_metrics["mae_mean"]), step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    predictions, bundle = evaluate_model_on_all_roles(
        model=model,
        experiment_name=f"trial_{trial.number:03d}",
    )

    validation_risk = compute_validation_risk_score(bundle["role_summary"])

    summary_row = build_trial_summary_row(
        trial_number=trial.number,
        params=params,
        freezing_report=freezing_report,
        history=history,
        bundle=bundle,
        validation_risk=validation_risk,
        smoke_mode=smoke_mode,
    )

    trial.set_user_attr("summary", summary_row)
    trial.set_user_attr("history", history)

    return {
        "summary": summary_row,
        "history": pd.DataFrame(history),
        "predictions": predictions,
        "bundle": bundle,
        "objective_value": validation_risk["validation_risk_score"],
    }


NB12_SMOKE_TRIAL_RESULTS = []


def nb12_transfer_smoke_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the one-trial NB12 smoke test."""
    result = run_transfer_trial(
        trial=trial,
        smoke_mode=True,
    )

    NB12_SMOKE_TRIAL_RESULTS.append(result)

    return float(result["objective_value"])


smoke_sampler = optuna.samplers.TPESampler(seed=SEED)
smoke_study = optuna.create_study(
    direction="minimize",
    sampler=smoke_sampler,
    study_name="nb12_transfer_exclude_ecg_fitness_smoke",
)

smoke_study.enqueue_trial(
    {
        "lr": 3e-5,
        "weight_decay": 1e-4,
        "dropout": FROZEN_MODEL_CONFIG["dropout"],
        "huber_delta": 6.5,
        "batch_size": 256,
        "grad_clip": 5.0,
        "freeze_encoder": False,
        "freeze_batchnorm": False,
        "use_balanced_sampler": False,
    }
)

smoke_study.optimize(
    nb12_transfer_smoke_objective,
    n_trials=1,
    show_progress_bar=False,
)

NB12_SMOKE_RESULT = NB12_SMOKE_TRIAL_RESULTS[0]
NB12_SMOKE_HISTORY = NB12_SMOKE_RESULT["history"]
NB12_SMOKE_BUNDLE = NB12_SMOKE_RESULT["bundle"]
NB12_SMOKE_SUMMARY = pd.DataFrame([NB12_SMOKE_RESULT["summary"]])

numeric_cols = NB12_SMOKE_SUMMARY.select_dtypes(include=[np.number]).columns
NB12_SMOKE_SUMMARY[numeric_cols] = NB12_SMOKE_SUMMARY[numeric_cols].round(4)

print("NB12 smoke trial history:")
display(NB12_SMOKE_HISTORY.round(4))

print("\nNB12 smoke trial role summary:")
display(NB12_SMOKE_BUNDLE["role_summary"])

print("\nNB12 smoke trial slice summary:")
display(NB12_SMOKE_BUNDLE["slice_summary"])

print("\nNB12 smoke trial summary:")
display(NB12_SMOKE_SUMMARY)

print("\nBest smoke study value:")
print(smoke_study.best_value)

[I 2026-07-16 16:35:58,278] A new study created in memory with name: nb12_transfer_exclude_ecg_fitness_smoke
[I 2026-07-16 16:36:11,559] Trial 0 finished with value: 6.0821 and parameters: {'lr': 3e-05, 'weight_decay': 0.0001, 'dropout': 0.11331939348791525, 'huber_delta': 6.5, 'batch_size': 256, 'grad_clip': 5.0, 'freeze_encoder': False, 'freeze_batchnorm': False, 'use_balanced_sampler': False}. Best is trial 0 with value: 6.0821.


NB12 smoke trial history:


,epoch,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_p90,best_epoch_so_far,best_val_mae_so_far,improved
0,1,40.9302,9.0078,107.5997,124.2253,6.0785,14.1061,1,6.0785,True
1,2,41.9334,9.1508,111.8911,160.8631,6.1600,14.0745,1,6.0785,False



NB12 smoke trial role summary:


,role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,finetune_train,12503,80.2370,82.1031,5.8650,3.7718,13.7772,19.0284,55.3639,1.8660,0.0540
1,finetune_val,2839,80.1617,81.9351,6.0785,3.8550,14.1061,19.4000,50.7650,1.7734,0.0659
2,finetune_test,2828,79.7822,81.4939,5.8241,3.7133,13.8359,19.0735,73.0231,1.7117,0.0580
3,ood_eval,882,109.2048,101.6886,14.0522,7.8395,37.7485,49.5336,89.5761,-7.5162,0.3027



NB12 smoke trial slice summary:


,slice,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_p90,signed_error_mean,severe_underprediction_rate
0,all_windows,19052,81.4994,82.8943,6.2697,14.5899,1.3950,0.0679
1,high_hr_ge_100,2125,111.5762,104.9143,10.5986,24.8629,-6.6619,0.2960
2,exercise_tachy_ge_120,350,133.0930,118.9673,17.7912,53.8631,-14.1256,0.3800
3,ecg_fitness_ood,882,109.2048,101.6886,14.0522,37.7485,-7.5162,0.3027
4,ecg_fitness_high_hr_ge_100,584,122.0550,111.3458,17.3720,44.1649,-10.7093,0.3887
5,ecg_fitness_exercise_tachy_ge_120,275,135.6331,118.7850,20.3906,54.6615,-16.8482,0.4291



NB12 smoke trial summary:


,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,ecg_fitness_signed_error_mean,ecg_fitness_severe_underprediction_rate,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,decision_status
0,0,True,2,0.0,0.0001,0.1133,6.5,256,5.0,False,...,-7.5162,0.3027,0.051,0.0545,0.0167,-0.0525,0.5063,-0.0066,-0.0528,reject_or_debug



Best smoke study value:
6.0821


## Smoke Trial Interpretation

The one-trial Optuna smoke test verified that the transfer-learning objective runs end-to-end.

The smoke trial is not evidence that the model improved or worsened in a generalizable way because it used a small subset of the training data and only two epochs.

The useful conclusion is limited to implementation readiness:

- the checkpoint-compatible model can be rebuilt,
- trial parameters can be sampled,
- transfer-learning updates can be applied,
- validation metrics can be computed during training,
- the best trial epoch can be restored,
- and final validation, test, high-HR, and ECG-Fitness summaries can be generated.

The smoke result should not be used as a candidate checkpoint.

The next section may run a real Optuna search with conservative compute limits.

In [7]:
NB12_TRANSFER_SEARCH_DIR = NB12_WORKING_DIR / "transfer_exclude_ecg_fitness_search"
NB12_TRANSFER_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

NB12_SEARCH_N_TRIALS = 12

NB12_FULL_TRIAL_RESULTS = []


def nb12_transfer_full_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the NB12 transfer search."""
    result = run_transfer_trial(
        trial=trial,
        smoke_mode=False,
    )

    NB12_FULL_TRIAL_RESULTS.append(result)

    return float(result["objective_value"])


def build_completed_trial_summary(study: optuna.Study) -> pd.DataFrame:
    """Build a summary table from completed Optuna trials with stored NB12 metrics."""
    rows = []

    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue

        summary = trial.user_attrs.get("summary")

        if not isinstance(summary, dict):
            continue

        row = dict(summary)
        row["optuna_value"] = float(trial.value)
        row["optuna_state"] = str(trial.state.name)
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    table = pd.DataFrame(rows)

    sort_cols = [
        "decision_status",
        "validation_risk_score",
        "in_distribution_mean_gain",
        "ood_delta_mean",
        "high_hr_delta_mae",
    ]

    existing_sort_cols = [col for col in sort_cols if col in table.columns]

    if existing_sort_cols:
        table = table.sort_values(existing_sort_cols, ascending=[True] * len(existing_sort_cols))

    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

    return table.reset_index(drop=True)


def save_nb12_search_artifacts(
    study: optuna.Study,
    trial_summary: pd.DataFrame,
    output_dir: Path,
) -> dict:
    """Save NB12 search artifacts for later notebook review."""
    output_dir.mkdir(parents=True, exist_ok=True)

    trial_summary_path = output_dir / "nb12_transfer_exclude_ecg_fitness_trial_summary.csv"
    best_params_path = output_dir / "nb12_transfer_exclude_ecg_fitness_best_params.json"
    study_trials_path = output_dir / "nb12_transfer_exclude_ecg_fitness_optuna_trials.csv"

    trial_summary.to_csv(trial_summary_path, index=False)

    with best_params_path.open("w", encoding="utf-8") as file:
        json.dump(study.best_params, file, indent=2)

    study.trials_dataframe().to_csv(study_trials_path, index=False)

    return {
        "trial_summary_csv": str(trial_summary_path),
        "best_params_json": str(best_params_path),
        "study_trials_csv": str(study_trials_path),
    }


full_sampler = optuna.samplers.TPESampler(seed=SEED)
full_pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=3,
)

transfer_study = optuna.create_study(
    direction="minimize",
    sampler=full_sampler,
    pruner=full_pruner,
    study_name="nb12_transfer_exclude_ecg_fitness_full",
)

transfer_study.enqueue_trial(
    {
        "lr": 3e-5,
        "weight_decay": 1e-4,
        "dropout": FROZEN_MODEL_CONFIG["dropout"],
        "huber_delta": 6.5,
        "batch_size": 256,
        "grad_clip": 5.0,
        "freeze_encoder": False,
        "freeze_batchnorm": False,
        "use_balanced_sampler": False,
    }
)

transfer_study.optimize(
    nb12_transfer_full_objective,
    n_trials=NB12_SEARCH_N_TRIALS,
    show_progress_bar=True,
)

NB12_TRANSFER_TRIAL_SUMMARY = build_completed_trial_summary(transfer_study)

NB12_TRANSFER_SEARCH_ARTIFACTS = save_nb12_search_artifacts(
    study=transfer_study,
    trial_summary=NB12_TRANSFER_TRIAL_SUMMARY,
    output_dir=NB12_TRANSFER_SEARCH_DIR,
)

print("NB12 transfer search best value:")
print(transfer_study.best_value)

print("\nNB12 transfer search best params:")
print(json.dumps(transfer_study.best_params, indent=2))

print("\nNB12 transfer search trial summary:")
display(NB12_TRANSFER_TRIAL_SUMMARY)

print("\nNB12 transfer search artifacts:")
print(json.dumps(NB12_TRANSFER_SEARCH_ARTIFACTS, indent=2))

[I 2026-07-16 16:36:11,682] A new study created in memory with name: nb12_transfer_exclude_ecg_fitness_full


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-07-16 16:38:25,126] Trial 0 finished with value: 5.882099999999999 and parameters: {'lr': 3e-05, 'weight_decay': 0.0001, 'dropout': 0.11331939348791525, 'huber_delta': 6.5, 'batch_size': 256, 'grad_clip': 5.0, 'freeze_encoder': False, 'freeze_batchnorm': False, 'use_balanced_sampler': False}. Best is trial 0 with value: 5.882099999999999.
[I 2026-07-16 16:39:47,227] Trial 1 finished with value: 6.47555 and parameters: {'lr': 1.9906996673933362e-05, 'weight_decay': 0.0032859708169642424, 'dropout': 0.2329984854528513, 'huber_delta': 8.38792635777333, 'batch_size': 128, 'grad_clip': 1.4065852851773961, 'freeze_encoder': False, 'freeze_batchnorm': False, 'use_balanced_sampler': False}. Best is trial 0 with value: 5.882099999999999.
[I 2026-07-16 16:41:30,885] Trial 2 finished with value: 5.8422 and parameters: {'lr': 1.0943342660062629e-05, 'weight_decay': 4.705059281907646e-06, 'dropout': 0.09585112746335846, 'huber_delta': 5.738180186635839, 'batch_size': 128, 'grad_clip': 3.038

,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,decision_status,optuna_value,optuna_state
0,10,False,12,0.0000,0.0000,0.0519,6.1217,256,7.3537,False,...,-0.2732,-0.1466,1.0024,0.8019,-1.9976,0.0466,0.2099,analysis_only_tradeoff,5.7913,COMPLETE
1,11,False,12,0.0000,0.0000,0.0523,5.9690,256,7.8847,False,...,-0.2496,-0.1356,0.8907,0.7249,-1.7832,0.0442,0.1926,analysis_only_tradeoff,5.8169,COMPLETE
2,2,False,9,0.0000,0.0000,0.0959,5.7382,128,3.0386,False,...,-0.1888,-0.0966,0.7825,0.5545,-1.0533,0.0282,0.1427,analysis_only_tradeoff,5.8422,COMPLETE
3,0,False,12,0.0000,0.0001,0.1133,6.5000,256,5.0000,False,...,-0.1509,-0.1976,1.2329,0.5021,-1.3361,0.0235,0.1742,analysis_only_tradeoff,5.8821,COMPLETE
4,8,False,6,0.0001,0.0007,0.1903,9.9387,256,3.9928,True,...,-0.1435,-0.0609,0.7485,0.7613,-0.2930,0.0278,0.1022,analysis_only_tradeoff,5.8860,COMPLETE
5,9,False,8,0.0001,0.0000,0.1526,9.8000,128,3.0283,True,...,-0.1565,-0.2364,2.0587,1.1351,-1.7361,0.0466,0.1964,analysis_only_tradeoff,5.9133,COMPLETE
6,5,False,5,0.0002,0.0020,0.1995,11.2969,256,1.3166,True,...,-0.0887,-0.0040,1.2948,0.9700,-0.6964,0.0348,0.0463,analysis_only_tradeoff,5.9388,COMPLETE
7,3,False,10,0.0000,0.0001,0.1981,3.4181,128,1.4554,True,...,0.0466,0.0936,0.9558,0.8261,0.5197,0.0136,-0.0701,reject_or_debug,6.0741,COMPLETE
8,1,False,7,0.0000,0.0033,0.2330,8.3879,128,1.4066,False,...,0.4209,0.3892,0.4297,0.0012,2.1403,-0.0386,-0.4051,reject_or_debug,6.4756,COMPLETE



NB12 transfer search artifacts:
{
  "trial_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/transfer_exclude_ecg_fitness_search/nb12_transfer_exclude_ecg_fitness_trial_summary.csv",
  "best_params_json": "/kaggle/working/crvse_nb12_optuna_search/transfer_exclude_ecg_fitness_search/nb12_transfer_exclude_ecg_fitness_best_params.json",
  "study_trials_csv": "/kaggle/working/crvse_nb12_optuna_search/transfer_exclude_ecg_fitness_search/nb12_transfer_exclude_ecg_fitness_optuna_trials.csv"
}


## First Transfer Search Interpretation

The first NB12 Optuna transfer search found in-distribution improvements, but did not produce a checkpoint suitable for app adoption.

The best trials improved live-compatible validation and held-out test MAE. However, they also worsened ECG-Fitness and high-HR behavior.

This repeats the main NB11 finding: transfer learning can adapt the model toward the live-compatible training distribution, but the gain comes with a high-HR / exercise-like penalty when ECG-Fitness remains excluded from training.

The best trial is therefore an analysis artifact, not an app checkpoint.

The next analysis step is to inspect the completed trials by validation gain, test gain, ECG-Fitness penalty, high-HR penalty, and severe underprediction change before deciding whether to change the objective or start an ECG-Fitness-included branch.

In [8]:
IMPORTANT_TRIAL_COLUMNS = [
    "trial_number",
    "decision_status",
    "validation_risk_score",
    "val_delta_mean",
    "test_delta_mean",
    "ood_delta_mean",
    "high_hr_delta_mae",
    "high_hr_signed_error_delta",
    "high_hr_severe_underprediction_rate_delta",
    "in_distribution_mean_gain",
    "lr",
    "weight_decay",
    "dropout",
    "huber_delta",
    "batch_size",
    "freeze_encoder",
    "freeze_batchnorm",
    "use_balanced_sampler",
    "epochs_run",
]


def build_tradeoff_review_table(trial_summary: pd.DataFrame) -> pd.DataFrame:
    """Build a compact review table for completed NB12 transfer trials."""
    missing = [col for col in IMPORTANT_TRIAL_COLUMNS if col not in trial_summary.columns]

    if missing:
        raise KeyError(f"Trial summary is missing expected columns: {missing}")

    table = trial_summary[IMPORTANT_TRIAL_COLUMNS].copy()

    table["passes_val_gain"] = (
        table["val_delta_mean"] <= -NB12_DECISION_THRESHOLDS["minimum_val_gain_bpm"]
    )
    table["passes_test_gain"] = (
        table["test_delta_mean"] <= -NB12_DECISION_THRESHOLDS["minimum_test_gain_bpm"]
    )
    table["passes_ood_penalty"] = (
        table["ood_delta_mean"] <= NB12_DECISION_THRESHOLDS["maximum_ood_penalty_bpm"]
    )
    table["passes_high_hr_penalty"] = (
        table["high_hr_delta_mae"] <= NB12_DECISION_THRESHOLDS["maximum_high_hr_mae_penalty_bpm"]
    )
    table["passes_underprediction_penalty"] = (
        table["high_hr_severe_underprediction_rate_delta"]
        <= NB12_DECISION_THRESHOLDS["maximum_severe_underprediction_rate_increase"]
    )

    table["passes_all_review_gates"] = (
        table["passes_val_gain"]
        & table["passes_test_gain"]
        & table["passes_ood_penalty"]
        & table["passes_high_hr_penalty"]
        & table["passes_underprediction_penalty"]
    )

    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

    return table.sort_values(
        [
            "passes_all_review_gates",
            "in_distribution_mean_gain",
            "ood_delta_mean",
            "high_hr_delta_mae",
        ],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)


NB12_TRANSFER_TRADEOFF_REVIEW = build_tradeoff_review_table(
    NB12_TRANSFER_TRIAL_SUMMARY
)

gate_summary = (
    NB12_TRANSFER_TRADEOFF_REVIEW[
        [
            "passes_val_gain",
            "passes_test_gain",
            "passes_ood_penalty",
            "passes_high_hr_penalty",
            "passes_underprediction_penalty",
            "passes_all_review_gates",
        ]
    ]
    .sum()
    .rename("trial_count")
    .reset_index()
    .rename(columns={"index": "gate"})
)

NB12_TRANSFER_TRADEOFF_REVIEW_PATH = (
    NB12_TRANSFER_SEARCH_DIR / "nb12_transfer_exclude_ecg_fitness_tradeoff_review.csv"
)
NB12_TRANSFER_TRADEOFF_REVIEW.to_csv(NB12_TRANSFER_TRADEOFF_REVIEW_PATH, index=False)

print("NB12 transfer tradeoff review:")
display(NB12_TRANSFER_TRADEOFF_REVIEW)

print("\nGate summary:")
display(gate_summary)

print("\nSaved tradeoff review:")
print(NB12_TRANSFER_TRADEOFF_REVIEW_PATH)

NB12 transfer tradeoff review:


,trial_number,decision_status,validation_risk_score,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,...,freeze_encoder,freeze_batchnorm,use_balanced_sampler,epochs_run,passes_val_gain,passes_test_gain,passes_ood_penalty,passes_high_hr_penalty,passes_underprediction_penalty,passes_all_review_gates
0,10,analysis_only_tradeoff,5.7913,-0.2732,-0.1466,1.0024,0.8019,-1.9976,0.0466,0.2099,...,False,True,True,12,True,True,False,False,False,False
1,9,analysis_only_tradeoff,5.9133,-0.1565,-0.2364,2.0587,1.1351,-1.7361,0.0466,0.1964,...,True,False,False,8,False,True,False,False,False,False
2,11,analysis_only_tradeoff,5.8169,-0.2496,-0.1356,0.8907,0.7249,-1.7832,0.0442,0.1926,...,False,True,True,12,True,True,False,False,False,False
3,0,analysis_only_tradeoff,5.8821,-0.1509,-0.1976,1.2329,0.5021,-1.3361,0.0235,0.1742,...,False,False,False,12,False,True,False,False,False,False
4,2,analysis_only_tradeoff,5.8422,-0.1888,-0.0966,0.7825,0.5545,-1.0533,0.0282,0.1427,...,False,True,True,9,False,False,False,False,False,False
5,8,analysis_only_tradeoff,5.8860,-0.1435,-0.0609,0.7485,0.7613,-0.2930,0.0278,0.1022,...,True,True,True,6,False,False,False,False,False,False
6,5,analysis_only_tradeoff,5.9388,-0.0887,-0.0040,1.2948,0.9700,-0.6964,0.0348,0.0463,...,True,True,False,5,False,False,False,False,False,False
7,3,reject_or_debug,6.0741,0.0466,0.0936,0.9558,0.8261,0.5197,0.0136,-0.0701,...,True,False,True,10,False,False,False,False,True,False
8,1,reject_or_debug,6.4756,0.4209,0.3892,0.4297,0.0012,2.1403,-0.0386,-0.4051,...,False,False,False,7,False,False,True,True,True,False



Gate summary:


,gate,trial_count
0,passes_val_gain,2
1,passes_test_gain,4
2,passes_ood_penalty,1
3,passes_high_hr_penalty,1
4,passes_underprediction_penalty,2
5,passes_all_review_gates,0



Saved tradeoff review:
/kaggle/working/crvse_nb12_optuna_search/transfer_exclude_ecg_fitness_search/nb12_transfer_exclude_ecg_fitness_tradeoff_review.csv


## HR Distribution Diagnostic Across Datasets

This section inspects the heart-rate distribution available to the first NB12 transfer branch.

The first transfer branch excludes ECG-Fitness from training and treats it as out-of-domain exercise/high-HR evaluation.

The diagnostic checks whether the non-ECG training data contains enough high-HR and exercise-like windows to support generalization to ECG-Fitness.

This section does not train a model. It explains the data distribution behind the transfer-learning trade-off observed in the first Optuna search.

In [9]:
HR_BIN_EDGES = [0, 60, 80, 100, 120, 140, 220]
HR_BIN_LABELS = [
    "lt_60",
    "60_80",
    "80_100",
    "100_120",
    "120_140",
    "ge_140",
]


def build_metadata_hr_table() -> pd.DataFrame:
    """Combine role metadata and tensor targets into one HR-distribution table."""
    rows = []

    for role in ROLE_ORDER:
        dataset = DATASETS[role]
        metadata = dataset.metadata.copy()
        metadata["role"] = role
        metadata["target_hr"] = dataset.y.astype(float)
        rows.append(metadata)

    table = pd.concat(rows, axis=0).reset_index(drop=True)

    if "dataset" not in table.columns:
        raise KeyError("Combined metadata is missing required column: dataset.")

    table["hr_bin"] = pd.cut(
        table["target_hr"],
        bins=HR_BIN_EDGES,
        labels=HR_BIN_LABELS,
        right=False,
        include_lowest=True,
    )

    table["is_high_hr_ge_100"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    )
    table["is_exercise_tachy_ge_120"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]
    )

    return table


def summarize_hr_distribution_by_dataset_role(hr_table: pd.DataFrame) -> pd.DataFrame:
    """Summarize HR distribution by dataset and role."""
    summary = (
        hr_table
        .groupby(["dataset", "role"], dropna=False)
        .agg(
            rows=("target_hr", "size"),
            target_hr_mean=("target_hr", "mean"),
            target_hr_median=("target_hr", "median"),
            target_hr_p10=("target_hr", lambda values: values.quantile(0.10)),
            target_hr_p90=("target_hr", lambda values: values.quantile(0.90)),
            target_hr_min=("target_hr", "min"),
            target_hr_max=("target_hr", "max"),
            high_hr_rows=("is_high_hr_ge_100", "sum"),
            high_hr_fraction=("is_high_hr_ge_100", "mean"),
            exercise_tachy_rows=("is_exercise_tachy_ge_120", "sum"),
            exercise_tachy_fraction=("is_exercise_tachy_ge_120", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary.sort_values(["role", "dataset"]).reset_index(drop=True)


def summarize_hr_bins(hr_table: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Count target-HR bins by selected grouping columns."""
    counts = (
        hr_table
        .groupby(group_cols + ["hr_bin"], dropna=False, observed=False)
        .size()
        .reset_index(name="rows")
    )

    totals = (
        counts
        .groupby(group_cols, dropna=False)["rows"]
        .transform("sum")
    )

    counts["fraction"] = (counts["rows"] / totals).round(4)

    return counts.sort_values(group_cols + ["hr_bin"]).reset_index(drop=True)


def summarize_training_vs_ood_hr_coverage(hr_table: pd.DataFrame) -> pd.DataFrame:
    """Compare high-HR coverage between training data and ECG-Fitness OOD data."""
    train_mask = hr_table["role"].eq("finetune_train")
    ood_mask = hr_table["role"].eq("ood_eval")

    slices = [
        ("training_non_ecg", train_mask),
        ("validation_non_ecg", hr_table["role"].eq("finetune_val")),
        ("test_non_ecg", hr_table["role"].eq("finetune_test")),
        ("ecg_fitness_ood", ood_mask),
        (
            "training_ubfc_rppg_only",
            train_mask & hr_table["dataset"].eq("ubfc_rppg"),
        ),
        (
            "training_mcd_rppg_only",
            train_mask & hr_table["dataset"].eq("mcd_rppg"),
        ),
        (
            "training_ubfc_phys_only",
            train_mask & hr_table["dataset"].eq("ubfc_phys"),
        ),
    ]

    rows = []

    for slice_name, mask in slices:
        part = hr_table.loc[mask].copy()

        rows.append(
            {
                "slice": slice_name,
                "rows": int(len(part)),
                "target_hr_mean": float(part["target_hr"].mean()),
                "target_hr_p90": float(part["target_hr"].quantile(0.90)),
                "target_hr_max": float(part["target_hr"].max()),
                "high_hr_rows": int(part["is_high_hr_ge_100"].sum()),
                "high_hr_fraction": float(part["is_high_hr_ge_100"].mean()),
                "exercise_tachy_rows": int(part["is_exercise_tachy_ge_120"].sum()),
                "exercise_tachy_fraction": float(part["is_exercise_tachy_ge_120"].mean()),
            }
        )

    summary = pd.DataFrame(rows)
    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def build_hr_distribution_interpretation(coverage_summary: pd.DataFrame) -> dict:
    """Build a compact interpretation of training versus ECG-Fitness HR coverage."""
    indexed = coverage_summary.set_index("slice")

    train = indexed.loc["training_non_ecg"]
    ecg = indexed.loc["ecg_fitness_ood"]

    high_hr_fraction_gap = float(ecg["high_hr_fraction"] - train["high_hr_fraction"])
    tachy_fraction_gap = float(ecg["exercise_tachy_fraction"] - train["exercise_tachy_fraction"])
    p90_gap = float(ecg["target_hr_p90"] - train["target_hr_p90"])

    return {
        "training_rows": int(train["rows"]),
        "ecg_fitness_ood_rows": int(ecg["rows"]),
        "training_high_hr_fraction": float(train["high_hr_fraction"]),
        "ecg_fitness_high_hr_fraction": float(ecg["high_hr_fraction"]),
        "high_hr_fraction_gap": high_hr_fraction_gap,
        "training_exercise_tachy_fraction": float(train["exercise_tachy_fraction"]),
        "ecg_fitness_exercise_tachy_fraction": float(ecg["exercise_tachy_fraction"]),
        "exercise_tachy_fraction_gap": tachy_fraction_gap,
        "training_target_hr_p90": float(train["target_hr_p90"]),
        "ecg_fitness_target_hr_p90": float(ecg["target_hr_p90"]),
        "target_hr_p90_gap": p90_gap,
        "supports_distribution_shift_explanation": bool(
            high_hr_fraction_gap > 0.20
            or tachy_fraction_gap > 0.10
            or p90_gap > 15.0
        ),
    }


NB12_HR_DISTRIBUTION_TABLE = build_metadata_hr_table()

NB12_HR_DISTRIBUTION_BY_DATASET_ROLE = summarize_hr_distribution_by_dataset_role(
    NB12_HR_DISTRIBUTION_TABLE
)

NB12_HR_BINS_BY_DATASET_ROLE = summarize_hr_bins(
    NB12_HR_DISTRIBUTION_TABLE,
    group_cols=["dataset", "role"],
)

NB12_HR_COVERAGE_SUMMARY = summarize_training_vs_ood_hr_coverage(
    NB12_HR_DISTRIBUTION_TABLE
)

NB12_HR_DISTRIBUTION_INTERPRETATION = build_hr_distribution_interpretation(
    NB12_HR_COVERAGE_SUMMARY
)

hr_distribution_dir = NB12_WORKING_DIR / "hr_distribution_diagnostic"
hr_distribution_dir.mkdir(parents=True, exist_ok=True)

hr_distribution_by_dataset_role_path = (
    hr_distribution_dir / "nb12_hr_distribution_by_dataset_role.csv"
)
hr_bins_by_dataset_role_path = (
    hr_distribution_dir / "nb12_hr_bins_by_dataset_role.csv"
)
hr_coverage_summary_path = (
    hr_distribution_dir / "nb12_training_vs_ood_hr_coverage.csv"
)
hr_distribution_interpretation_path = (
    hr_distribution_dir / "nb12_hr_distribution_interpretation.json"
)

NB12_HR_DISTRIBUTION_BY_DATASET_ROLE.to_csv(
    hr_distribution_by_dataset_role_path,
    index=False,
)
NB12_HR_BINS_BY_DATASET_ROLE.to_csv(
    hr_bins_by_dataset_role_path,
    index=False,
)
NB12_HR_COVERAGE_SUMMARY.to_csv(
    hr_coverage_summary_path,
    index=False,
)

with hr_distribution_interpretation_path.open("w", encoding="utf-8") as file:
    json.dump(NB12_HR_DISTRIBUTION_INTERPRETATION, file, indent=2)

print("HR distribution by dataset and role:")
display(NB12_HR_DISTRIBUTION_BY_DATASET_ROLE)

print("\nHR bin counts by dataset and role:")
display(NB12_HR_BINS_BY_DATASET_ROLE)

print("\nTraining versus ECG-Fitness HR coverage:")
display(NB12_HR_COVERAGE_SUMMARY)

print("\nNB12 HR distribution interpretation:")
print(json.dumps(NB12_HR_DISTRIBUTION_INTERPRETATION, indent=2))

print("\nSaved HR distribution diagnostic artifacts:")
print(json.dumps(
    {
        "distribution_by_dataset_role_csv": str(hr_distribution_by_dataset_role_path),
        "bins_by_dataset_role_csv": str(hr_bins_by_dataset_role_path),
        "coverage_summary_csv": str(hr_coverage_summary_path),
        "interpretation_json": str(hr_distribution_interpretation_path),
    },
    indent=2,
))

HR distribution by dataset and role:


,dataset,role,rows,target_hr_mean,target_hr_median,target_hr_p10,target_hr_p90,target_hr_min,target_hr_max,high_hr_rows,high_hr_fraction,exercise_tachy_rows,exercise_tachy_fraction
0,mcd_rppg,finetune_test,2094,79.8533,78.0186,62.9663,99.3013,51.1935,129.0719,192,0.0917,5,0.0024
1,ubfc_phys,finetune_test,658,77.4205,77.4923,65.6681,89.6468,47.6317,109.8425,7,0.0106,0,0.0000
2,ubfc_rppg,finetune_test,76,98.2693,106.5540,62.4339,121.1256,58.1461,124.6941,45,0.5921,12,0.1579
3,mcd_rppg,finetune_train,7446,79.0648,78.1173,62.5504,96.9220,44.9337,125.7102,569,0.0764,13,0.0017
4,ubfc_phys,finetune_train,4595,80.3055,79.3453,64.1272,98.1041,40.0563,121.2303,367,0.0799,1,0.0002
5,ubfc_rppg,finetune_train,462,98.4492,97.3792,80.0433,117.1463,68.0470,127.0000,193,0.4177,32,0.0693
6,mcd_rppg,finetune_val,1743,79.1086,81.5532,58.2631,96.4610,50.2914,114.7758,79,0.0453,0,0.0000
7,ubfc_phys,finetune_val,986,80.0024,80.7107,65.8921,94.2339,40.2964,115.1911,19,0.0193,0,0.0000
8,ubfc_rppg,finetune_val,110,98.2772,107.9003,64.5437,121.0300,57.3827,127.0000,70,0.6364,12,0.1091
9,ecg_fitness,ood_eval,882,109.2048,109.6703,78.7118,140.2072,53.6622,167.9681,584,0.6621,275,0.3118



HR bin counts by dataset and role:


,dataset,role,hr_bin,rows,fraction
0,ecg_fitness,finetune_test,lt_60,0,NaN
1,ecg_fitness,finetune_test,60_80,0,NaN
2,ecg_fitness,finetune_test,80_100,0,NaN
3,ecg_fitness,finetune_test,100_120,0,NaN
4,ecg_fitness,finetune_test,120_140,0,NaN
...,...,...,...,...,...
91,ubfc_rppg,ood_eval,60_80,0,NaN
92,ubfc_rppg,ood_eval,80_100,0,NaN
93,ubfc_rppg,ood_eval,100_120,0,NaN
94,ubfc_rppg,ood_eval,120_140,0,NaN



Training versus ECG-Fitness HR coverage:


,slice,rows,target_hr_mean,target_hr_p90,target_hr_max,high_hr_rows,high_hr_fraction,exercise_tachy_rows,exercise_tachy_fraction
0,training_non_ecg,12503,80.2370,98.9065,127.0000,1129,0.0903,46,0.0037
1,validation_non_ecg,2839,80.1617,96.4772,127.0000,168,0.0592,12,0.0042
2,test_non_ecg,2828,79.7822,98.8912,129.0719,244,0.0863,17,0.0060
3,ecg_fitness_ood,882,109.2048,140.2072,167.9681,584,0.6621,275,0.3118
4,training_ubfc_rppg_only,462,98.4492,117.1463,127.0000,193,0.4177,32,0.0693
5,training_mcd_rppg_only,7446,79.0648,96.9220,125.7102,569,0.0764,13,0.0017
6,training_ubfc_phys_only,4595,80.3055,98.1041,121.2303,367,0.0799,1,0.0002



NB12 HR distribution interpretation:
{
  "training_rows": 12503,
  "ecg_fitness_ood_rows": 882,
  "training_high_hr_fraction": 0.0903,
  "ecg_fitness_high_hr_fraction": 0.6621,
  "high_hr_fraction_gap": 0.5718,
  "training_exercise_tachy_fraction": 0.0037,
  "ecg_fitness_exercise_tachy_fraction": 0.3118,
  "exercise_tachy_fraction_gap": 0.30810000000000004,
  "training_target_hr_p90": 98.9065,
  "ecg_fitness_target_hr_p90": 140.2072,
  "target_hr_p90_gap": 41.300700000000006,
  "supports_distribution_shift_explanation": true
}

Saved HR distribution diagnostic artifacts:
{
  "distribution_by_dataset_role_csv": "/kaggle/working/crvse_nb12_optuna_search/hr_distribution_diagnostic/nb12_hr_distribution_by_dataset_role.csv",
  "bins_by_dataset_role_csv": "/kaggle/working/crvse_nb12_optuna_search/hr_distribution_diagnostic/nb12_hr_bins_by_dataset_role.csv",
  "coverage_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/hr_distribution_diagnostic/nb12_training_vs_ood_hr_coverage.csv",
 

## High-HR-Aware Transfer Search Design

This section defines the second NB12 transfer branch.

ECG-Fitness remains excluded from training and remains an out-of-domain exercise/high-HR stress test.

The first transfer search showed that Optuna can improve validation and test MAE, but the best trials worsened ECG-Fitness and high-HR behavior.

The HR distribution diagnostic showed why this happens: the non-ECG training, validation, and test splits contain relatively few high-HR windows, while ECG-Fitness is dominated by high-HR and exercise-like windows.

This branch tests whether the model can use the limited non-ECG high-HR evidence more deliberately.

The branch changes the training and objective design:

- high-HR non-ECG windows may receive higher sampling weight,
- UBFC-rPPG may receive higher sampling weight because it contains more high-HR examples,
- validation scoring includes penalties for high-HR MAE and severe underprediction,
- ECG-Fitness remains evaluation-only and is not used for trial optimization.

This branch can test whether high-HR-aware training helps, but it cannot fully solve missing exercise-domain coverage if the training data does not contain enough exercise-like examples.

In [10]:
NB12_HIGH_HR_SEARCH_DIR = NB12_WORKING_DIR / "transfer_high_hr_aware_exclude_ecg_search"
NB12_HIGH_HR_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

NB12_HIGH_HR_SEARCH_N_TRIALS = 12

HIGH_HR_OBJECTIVE_WEIGHTS = {
    "val_mae": 1.0,
    "val_high_hr_mae_penalty": 0.30,
    "val_high_hr_underprediction_penalty": 3.0,
    "val_p90_worsening_penalty": 0.10,
}

HIGH_HR_SAMPLER_SPACE = {
    "dataset_balance_power": {
        "low": 0.0,
        "high": 1.0,
        "reason": "Controls how strongly smaller datasets are upweighted.",
    },
    "high_hr_weight": {
        "low": 1.0,
        "high": 8.0,
        "reason": "Controls how strongly windows >=100 BPM are oversampled.",
    },
    "tachy_weight": {
        "low": 1.0,
        "high": 12.0,
        "reason": "Controls how strongly windows >=120 BPM are oversampled.",
    },
    "ubfc_rppg_weight": {
        "low": 1.0,
        "high": 6.0,
        "reason": "Controls extra weight for UBFC-rPPG, the non-ECG dataset with more high-HR examples.",
    },
}


def build_high_hr_sampler_space_table() -> pd.DataFrame:
    """Build a readable table for high-HR sampler parameters."""
    rows = []

    for name, spec in HIGH_HR_SAMPLER_SPACE.items():
        rows.append(
            {
                "parameter": name,
                "low": spec["low"],
                "high": spec["high"],
                "reason": spec["reason"],
            }
        )

    return pd.DataFrame(rows)


def summarize_high_hr_branch_design() -> dict:
    """Summarize the high-HR-aware branch design."""
    return {
        "branch_name": "transfer_high_hr_aware_exclude_ecg_fitness",
        "ecg_fitness_policy": "excluded_from_training_and_evaluated_as_ood_stress_test",
        "training_change": "weighted_sampler_can_upweight_dataset_minority_and_high_hr_windows",
        "objective_change": "validation_score_penalizes_high_hr_error_and_underprediction",
        "n_trials_initial": NB12_HIGH_HR_SEARCH_N_TRIALS,
        "important_limitation": (
            "Only a small fraction of non-ECG training windows are exercise-like, "
            "so this branch may not fully solve ECG-Fitness degradation."
        ),
    }


print("High-HR-aware objective weights:")
print(json.dumps(HIGH_HR_OBJECTIVE_WEIGHTS, indent=2))

print("\nHigh-HR-aware sampler search space:")
display(build_high_hr_sampler_space_table())

print("\nHigh-HR-aware branch design:")
print(json.dumps(summarize_high_hr_branch_design(), indent=2))

High-HR-aware objective weights:
{
  "val_mae": 1.0,
  "val_high_hr_mae_penalty": 0.3,
  "val_high_hr_underprediction_penalty": 3.0,
  "val_p90_worsening_penalty": 0.1
}

High-HR-aware sampler search space:


,parameter,low,high,reason
0,dataset_balance_power,0.0,1.0,Controls how strongly smaller datasets are upweighted.
1,high_hr_weight,1.0,8.0,Controls how strongly windows >=100 BPM are oversampled.
2,tachy_weight,1.0,12.0,Controls how strongly windows >=120 BPM are oversampled.
3,ubfc_rppg_weight,1.0,6.0,"Controls extra weight for UBFC-rPPG, the non-ECG dataset with more high-HR examples."



High-HR-aware branch design:
{
  "branch_name": "transfer_high_hr_aware_exclude_ecg_fitness",
  "ecg_fitness_policy": "excluded_from_training_and_evaluated_as_ood_stress_test",
  "training_change": "weighted_sampler_can_upweight_dataset_minority_and_high_hr_windows",
  "objective_change": "validation_score_penalizes_high_hr_error_and_underprediction",
  "n_trials_initial": 12,
  "important_limitation": "Only a small fraction of non-ECG training windows are exercise-like, so this branch may not fully solve ECG-Fitness degradation."
}


## High-HR-Aware Smoke Trial

This section runs one small smoke trial for the high-HR-aware transfer branch.

The smoke trial verifies that:

- high-HR-aware sampling weights can be built,
- dataset and HR-based sampling can be combined,
- the transfer model can train with the weighted sampler,
- the high-HR-aware validation objective can be computed,
- and ECG-Fitness remains evaluation-only.

The smoke trial is not a candidate checkpoint.

In [11]:
NB12_HH_ROLE_ORDER = ["finetune_train", "finetune_val", "finetune_test", "ood_eval"]
NB12_HH_SMOKE_ROWS_PER_DATASET = 256
NB12_HH_SMOKE_MAX_EPOCHS = 2
NB12_HH_TRIAL_MIN_DELTA = 1e-3

NB12_HH_OBJECTIVE_WEIGHTS = {
    "val_mae": 1.0,
    "val_high_hr_mae_penalty": 0.30,
    "val_high_hr_underprediction_penalty": 3.0,
    "val_p90_worsening_penalty": 0.10,
}

NB12_HH_REQUIRED_NAMES = [
    "DATASETS",
    "EVAL_LOADERS",
    "DEVICE",
    "SEED",
    "CRVSEPhysFormer",
    "FROZEN_MODEL_CONFIG",
    "FROZEN_CHECKPOINT",
    "NB12_DECISION_THRESHOLDS",
    "FROZEN_DECISION_BUNDLE",
    "FROZEN_DECISION_METRICS",
    "FROZEN_VALIDATION_REFERENCE",
]

missing_names = [name for name in NB12_HH_REQUIRED_NAMES if name not in globals()]
if missing_names:
    raise NameError(f"Run earlier NB12 setup cells first. Missing: {missing_names}")


class NB12HighHrIndexedDataset(Dataset):
    """Dataset view over selected row indices from a materialized tensor dataset."""

    def __init__(self, base_dataset: Dataset, indices: np.ndarray, role: str) -> None:
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.role = role
        self.metadata = base_dataset.metadata.iloc[self.indices].reset_index(drop=True)

        if len(self.indices) == 0:
            raise ValueError("NB12HighHrIndexedDataset received no indices.")

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.base_dataset[int(self.indices[index])]


def nb12_hh_select_smoke_indices(
    dataset: Dataset,
    rows_per_dataset: int,
    seed: int,
) -> np.ndarray:
    """Select a small dataset-balanced subset for smoke testing."""
    metadata = dataset.metadata.copy()

    if "dataset" not in metadata.columns:
        raise KeyError("Training metadata is missing required column: dataset.")

    rng = np.random.default_rng(seed)
    metadata["_row_position"] = np.arange(len(metadata))
    selected = []

    for _, group in metadata.groupby("dataset", sort=True):
        positions = group["_row_position"].to_numpy()
        take = min(rows_per_dataset, len(positions))
        selected.extend(rng.choice(positions, size=take, replace=False).tolist())

    return np.asarray(sorted(selected), dtype=np.int64)


def nb12_hh_make_training_dataset(smoke_mode: bool) -> Dataset:
    """Return full or smoke-mode training dataset."""
    train_dataset = DATASETS["finetune_train"]

    if not smoke_mode:
        return train_dataset

    indices = nb12_hh_select_smoke_indices(
        dataset=train_dataset,
        rows_per_dataset=NB12_HH_SMOKE_ROWS_PER_DATASET,
        seed=SEED,
    )

    return NB12HighHrIndexedDataset(
        base_dataset=train_dataset,
        indices=indices,
        role="finetune_train_smoke",
    )


def nb12_hh_collect_targets(dataset: Dataset) -> tuple[pd.DataFrame, np.ndarray]:
    """Return metadata and target HR values for a dataset-like object."""
    metadata = dataset.metadata.copy()

    if isinstance(dataset, NB12HighHrIndexedDataset):
        targets = np.asarray(dataset.base_dataset.y[dataset.indices], dtype=float)
        return metadata, targets

    if hasattr(dataset, "y"):
        targets = np.asarray(dataset.y, dtype=float)
        return metadata, targets

    targets = []
    for index in range(len(dataset)):
        _, y = dataset[index]
        targets.append(float(y.item()))

    return metadata, np.asarray(targets, dtype=float)


def nb12_hh_suggest_params(trial: optuna.Trial) -> dict:
    """Suggest one high-HR-aware transfer trial parameter set."""
    return {
        "lr": trial.suggest_float("lr", 5e-6, 2e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.05, 0.30),
        "huber_delta": trial.suggest_float("huber_delta", 3.0, 12.0),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
        "grad_clip": trial.suggest_float("grad_clip", 1.0, 8.0),
        "freeze_encoder": trial.suggest_categorical("freeze_encoder", [False, True]),
        "freeze_batchnorm": trial.suggest_categorical("freeze_batchnorm", [False, True]),
        "dataset_balance_power": trial.suggest_float("dataset_balance_power", 0.0, 1.0),
        "high_hr_weight": trial.suggest_float("high_hr_weight", 1.0, 8.0),
        "tachy_weight": trial.suggest_float("tachy_weight", 1.0, 12.0),
        "ubfc_rppg_weight": trial.suggest_float("ubfc_rppg_weight", 1.0, 6.0),
    }


def nb12_hh_build_sampling_weights(dataset: Dataset, params: dict) -> np.ndarray:
    """Build sampling weights from dataset size, target HR, and UBFC-rPPG membership."""
    metadata, targets = nb12_hh_collect_targets(dataset)

    if "dataset" not in metadata.columns:
        raise KeyError("Dataset metadata is missing required column: dataset.")

    weights = np.ones(len(metadata), dtype=np.float64)
    dataset_counts = metadata["dataset"].value_counts().to_dict()

    for dataset_name, count in dataset_counts.items():
        mask = metadata["dataset"].eq(dataset_name).to_numpy()
        factor = (len(metadata) / float(count)) ** float(params["dataset_balance_power"])
        weights[mask] *= factor

    high_hr_mask = targets >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    tachy_mask = targets >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]
    ubfc_rppg_mask = metadata["dataset"].eq("ubfc_rppg").to_numpy()

    weights[high_hr_mask] *= float(params["high_hr_weight"])
    weights[tachy_mask] *= float(params["tachy_weight"])
    weights[ubfc_rppg_mask] *= float(params["ubfc_rppg_weight"])

    if not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError("Invalid high-HR sampling weights.")

    return weights


def nb12_hh_summarize_sampling_weights(dataset: Dataset, params: dict) -> pd.DataFrame:
    """Summarize sampling weights by dataset and HR slice."""
    metadata, targets = nb12_hh_collect_targets(dataset)
    weights = nb12_hh_build_sampling_weights(dataset, params)

    table = metadata.copy()
    table["target_hr"] = targets
    table["sampling_weight"] = weights
    table["hr_slice"] = np.select(
        [
            table["target_hr"] >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"],
            table["target_hr"] >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"],
        ],
        ["ge_120", "100_120"],
        default="lt_100",
    )

    summary = (
        table.groupby(["dataset", "hr_slice"], dropna=False)
        .agg(
            rows=("sampling_weight", "size"),
            target_hr_mean=("target_hr", "mean"),
            weight_mean=("sampling_weight", "mean"),
            weight_min=("sampling_weight", "min"),
            weight_max=("sampling_weight", "max"),
            total_weight=("sampling_weight", "sum"),
        )
        .reset_index()
        .sort_values(["dataset", "hr_slice"])
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def nb12_hh_make_training_loader(dataset: Dataset, params: dict, seed: int) -> DataLoader:
    """Create a weighted high-HR-aware training DataLoader."""
    weights = nb12_hh_build_sampling_weights(dataset, params)

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )

    return DataLoader(
        dataset,
        batch_size=int(params["batch_size"]),
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def nb12_hh_freeze_batchnorm(model: nn.Module) -> list[str]:
    """Freeze BatchNorm1d modules and their running statistics."""
    frozen_names = []

    for module_name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()
            frozen_names.append(module_name)

            for parameter in module.parameters():
                parameter.requires_grad = False

    return frozen_names


def nb12_hh_build_model(params: dict) -> tuple[nn.Module, dict]:
    """Build a checkpoint-initialized transfer model for one high-HR trial."""
    model_config = dict(FROZEN_MODEL_CONFIG)
    model_config["dropout"] = float(params["dropout"])

    model = CRVSEPhysFormer(**model_config)
    model.load_state_dict(FROZEN_CHECKPOINT["model_state"], strict=True)
    model.to(DEVICE)

    frozen_batchnorm_names = []

    if bool(params["freeze_encoder"]):
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False

    if bool(params["freeze_batchnorm"]):
        frozen_batchnorm_names = nb12_hh_freeze_batchnorm(model)

    trainable_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if len(trainable_parameters) == 0:
        raise ValueError("Trial produced a model with no trainable parameters.")

    return model, {
        "freeze_encoder": bool(params["freeze_encoder"]),
        "freeze_batchnorm": bool(params["freeze_batchnorm"]),
        "frozen_batchnorm_names": frozen_batchnorm_names,
        "total_parameters": int(sum(parameter.numel() for parameter in model.parameters())),
        "trainable_parameters": int(sum(parameter.numel() for parameter in trainable_parameters)),
    }


def nb12_hh_enforce_training_modes(
    model: nn.Module,
    freeze_encoder: bool,
    freeze_batchnorm: bool,
) -> None:
    """Keep frozen modules in eval mode after model.train()."""
    if freeze_encoder:
        model.encoder.eval()

    if freeze_batchnorm:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm1d):
                module.eval()


def nb12_hh_train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    params: dict,
) -> dict:
    """Train one high-HR-aware transfer epoch."""
    model.train()
    nb12_hh_enforce_training_modes(
        model=model,
        freeze_encoder=bool(params["freeze_encoder"]),
        freeze_batchnorm=bool(params["freeze_batchnorm"]),
    )

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        trainable_parameters = [
            parameter for parameter in model.parameters()
            if parameter.requires_grad
        ]

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=float(params["grad_clip"]),
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def nb12_hh_evaluate_role(
    model: nn.Module,
    dataset: Dataset,
    loader: DataLoader,
) -> pd.DataFrame:
    """Evaluate one model on one role dataset."""
    prediction_batches = []
    target_batches = []

    model.eval()

    with torch.inference_mode():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            pred = model(batch_x).detach().cpu().numpy().astype(np.float32)

            prediction_batches.append(pred)
            target_batches.append(batch_y.numpy().astype(np.float32))

    pred_hr = np.concatenate(prediction_batches)
    target_hr = np.concatenate(target_batches)

    rows = dataset.metadata.copy()
    rows["notebook_model_hr"] = pred_hr
    rows["target_hr_from_tensor"] = target_hr
    rows["notebook_abs_error"] = np.abs(pred_hr - target_hr)
    rows["notebook_signed_error"] = pred_hr - target_hr
    rows["role"] = dataset.role

    return rows


def nb12_hh_prepare_error_table(predictions: pd.DataFrame, experiment_name: str) -> pd.DataFrame:
    """Add standard error columns for high-HR decision analysis."""
    table = predictions.copy()
    table["experiment"] = experiment_name
    table["target_hr"] = table["target_hr_from_tensor"].astype(float)
    table["pred_hr"] = table["notebook_model_hr"].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    table["is_high_hr_ge_100"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    )
    table["is_exercise_tachy_ge_120"] = (
        table["target_hr"] >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]
    )
    table["is_severe_underprediction"] = (
        table["signed_error"] <= NB12_DECISION_THRESHOLDS["severe_underprediction_bpm"]
    )

    return table


def nb12_hh_summarize_groups(error_table: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize HR error by selected groups."""
    summary = (
        error_table.groupby(group_cols, dropna=False)
        .agg(
            rows=("abs_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda values: values.quantile(0.90)),
            mae_p95=("abs_error", lambda values: values.quantile(0.95)),
            mae_max=("abs_error", "max"),
            signed_error_mean=("signed_error", "mean"),
            severe_underprediction_rate=("is_severe_underprediction", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def nb12_hh_build_slice_summary(error_table: pd.DataFrame) -> pd.DataFrame:
    """Build high-HR and ECG-Fitness slice summaries."""
    def one_slice(name: str, mask: pd.Series) -> dict:
        part = error_table.loc[mask].copy()

        if len(part) == 0:
            return {
                "slice": name,
                "rows": 0,
                "target_hr_mean": np.nan,
                "pred_hr_mean": np.nan,
                "mae_mean": np.nan,
                "mae_p90": np.nan,
                "signed_error_mean": np.nan,
                "severe_underprediction_rate": np.nan,
            }

        return {
            "slice": name,
            "rows": int(len(part)),
            "target_hr_mean": float(part["target_hr"].mean()),
            "pred_hr_mean": float(part["pred_hr"].mean()),
            "mae_mean": float(part["abs_error"].mean()),
            "mae_p90": float(part["abs_error"].quantile(0.90)),
            "signed_error_mean": float(part["signed_error"].mean()),
            "severe_underprediction_rate": float(part["is_severe_underprediction"].mean()),
        }

    all_rows = pd.Series(True, index=error_table.index)
    high_hr = error_table["is_high_hr_ge_100"]
    tachy = error_table["is_exercise_tachy_ge_120"]
    ecg = error_table["role"].eq("ood_eval")

    summary = pd.DataFrame(
        [
            one_slice("all_windows", all_rows),
            one_slice("high_hr_ge_100", high_hr),
            one_slice("exercise_tachy_ge_120", tachy),
            one_slice("ecg_fitness_ood", ecg),
            one_slice("ecg_fitness_high_hr_ge_100", ecg & high_hr),
            one_slice("ecg_fitness_exercise_tachy_ge_120", ecg & tachy),
        ]
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def nb12_hh_extract_metric(summary: pd.DataFrame, key_col: str, key_value: str, metric_col: str) -> float:
    """Extract one metric from a summary table."""
    rows = summary.loc[summary[key_col] == key_value]

    if len(rows) != 1:
        raise ValueError(f"Expected one row for {key_col}={key_value}, found {len(rows)}.")

    return float(rows.iloc[0][metric_col])


def nb12_hh_build_bundle(predictions: pd.DataFrame, experiment_name: str) -> dict:
    """Build final full-role summaries for one high-HR trial."""
    error_table = nb12_hh_prepare_error_table(predictions, experiment_name)

    role_summary = nb12_hh_summarize_groups(error_table, ["role"])
    role_order = {role: index for index, role in enumerate(NB12_HH_ROLE_ORDER)}
    role_summary["role_order"] = role_summary["role"].map(role_order).fillna(999).astype(int)
    role_summary = role_summary.sort_values(["role_order", "role"]).drop(columns=["role_order"])

    dataset_role_summary = nb12_hh_summarize_groups(error_table, ["dataset", "role"])
    dataset_role_summary = dataset_role_summary.sort_values(["role", "dataset"]).reset_index(drop=True)

    slice_summary = nb12_hh_build_slice_summary(error_table)

    decision_metrics = {
        "experiment": experiment_name,
        "val_mae_mean": nb12_hh_extract_metric(role_summary, "role", "finetune_val", "mae_mean"),
        "test_mae_mean": nb12_hh_extract_metric(role_summary, "role", "finetune_test", "mae_mean"),
        "ood_mae_mean": nb12_hh_extract_metric(role_summary, "role", "ood_eval", "mae_mean"),
        "val_mae_p90": nb12_hh_extract_metric(role_summary, "role", "finetune_val", "mae_p90"),
        "test_mae_p90": nb12_hh_extract_metric(role_summary, "role", "finetune_test", "mae_p90"),
        "ood_mae_p90": nb12_hh_extract_metric(role_summary, "role", "ood_eval", "mae_p90"),
        "high_hr_mae_mean": nb12_hh_extract_metric(slice_summary, "slice", "high_hr_ge_100", "mae_mean"),
        "high_hr_signed_error_mean": nb12_hh_extract_metric(slice_summary, "slice", "high_hr_ge_100", "signed_error_mean"),
        "high_hr_severe_underprediction_rate": nb12_hh_extract_metric(slice_summary, "slice", "high_hr_ge_100", "severe_underprediction_rate"),
        "ecg_fitness_mae_mean": nb12_hh_extract_metric(slice_summary, "slice", "ecg_fitness_ood", "mae_mean"),
        "ecg_fitness_signed_error_mean": nb12_hh_extract_metric(slice_summary, "slice", "ecg_fitness_ood", "signed_error_mean"),
        "ecg_fitness_severe_underprediction_rate": nb12_hh_extract_metric(slice_summary, "slice", "ecg_fitness_ood", "severe_underprediction_rate"),
    }

    return {
        "error_table": error_table,
        "role_summary": role_summary.reset_index(drop=True),
        "dataset_role_summary": dataset_role_summary,
        "slice_summary": slice_summary,
        "decision_metrics": decision_metrics,
    }


def nb12_hh_compute_validation_score_from_error_table(error_table: pd.DataFrame) -> dict:
    """Compute the high-HR-aware validation score from validation rows."""
    val_rows = error_table.loc[error_table["role"].eq("finetune_val")].copy()

    if len(val_rows) == 0:
        raise ValueError("Validation score requires finetune_val rows.")

    val_mae = float(val_rows["abs_error"].mean())
    val_p90 = float(val_rows["abs_error"].quantile(0.90))

    val_high_hr = val_rows.loc[val_rows["is_high_hr_ge_100"]].copy()

    if len(val_high_hr) == 0:
        val_high_hr_mae = 0.0
        val_high_hr_under_rate = 0.0
    else:
        val_high_hr_mae = float(val_high_hr["abs_error"].mean())
        val_high_hr_under_rate = float(val_high_hr["is_severe_underprediction"].mean())

    frozen_error = FROZEN_DECISION_BUNDLE["error_table"]
    frozen_val_high_hr = frozen_error.loc[
        frozen_error["role"].eq("finetune_val")
        & frozen_error["is_high_hr_ge_100"]
    ].copy()

    frozen_val_high_hr_mae = float(frozen_val_high_hr["abs_error"].mean())
    frozen_val_high_hr_under_rate = float(
        frozen_val_high_hr["is_severe_underprediction"].mean()
    )

    val_high_hr_mae_worsening = max(0.0, val_high_hr_mae - frozen_val_high_hr_mae)
    val_high_hr_under_worsening = max(
        0.0,
        val_high_hr_under_rate - frozen_val_high_hr_under_rate,
    )
    val_p90_worsening = max(0.0, val_p90 - FROZEN_VALIDATION_REFERENCE["val_mae_p90"])

    score = (
        NB12_HH_OBJECTIVE_WEIGHTS["val_mae"] * val_mae
        + NB12_HH_OBJECTIVE_WEIGHTS["val_high_hr_mae_penalty"] * val_high_hr_mae_worsening
        + NB12_HH_OBJECTIVE_WEIGHTS["val_high_hr_underprediction_penalty"] * val_high_hr_under_worsening
        + NB12_HH_OBJECTIVE_WEIGHTS["val_p90_worsening_penalty"] * val_p90_worsening
    )

    return {
        "high_hr_validation_score": float(score),
        "val_mae_mean": float(val_mae),
        "val_mae_p90": float(val_p90),
        "val_high_hr_mae": float(val_high_hr_mae),
        "val_high_hr_underprediction_rate": float(val_high_hr_under_rate),
        "val_high_hr_mae_worsening": float(val_high_hr_mae_worsening),
        "val_high_hr_underprediction_rate_worsening": float(val_high_hr_under_worsening),
        "val_p90_worsening": float(val_p90_worsening),
    }


def nb12_hh_compute_validation_score_from_predictions(
    val_predictions: pd.DataFrame,
    experiment_name: str,
) -> dict:
    """Compute high-HR validation score from validation-only predictions."""
    error_table = nb12_hh_prepare_error_table(val_predictions, experiment_name)
    return nb12_hh_compute_validation_score_from_error_table(error_table)


def nb12_hh_compare_to_frozen(candidate_metrics: dict) -> dict:
    """Compare candidate decision metrics with the frozen baseline."""
    row = dict(candidate_metrics)

    row["val_delta_mean"] = candidate_metrics["val_mae_mean"] - FROZEN_DECISION_METRICS["val_mae_mean"]
    row["test_delta_mean"] = candidate_metrics["test_mae_mean"] - FROZEN_DECISION_METRICS["test_mae_mean"]
    row["ood_delta_mean"] = candidate_metrics["ood_mae_mean"] - FROZEN_DECISION_METRICS["ood_mae_mean"]
    row["high_hr_delta_mae"] = candidate_metrics["high_hr_mae_mean"] - FROZEN_DECISION_METRICS["high_hr_mae_mean"]
    row["high_hr_signed_error_delta"] = (
        candidate_metrics["high_hr_signed_error_mean"]
        - FROZEN_DECISION_METRICS["high_hr_signed_error_mean"]
    )
    row["high_hr_severe_underprediction_rate_delta"] = (
        candidate_metrics["high_hr_severe_underprediction_rate"]
        - FROZEN_DECISION_METRICS["high_hr_severe_underprediction_rate"]
    )
    row["in_distribution_mean_gain"] = -0.5 * (
        row["val_delta_mean"] + row["test_delta_mean"]
    )

    if (
        row["val_delta_mean"] <= -NB12_DECISION_THRESHOLDS["minimum_val_gain_bpm"]
        and row["test_delta_mean"] <= -NB12_DECISION_THRESHOLDS["minimum_test_gain_bpm"]
        and row["ood_delta_mean"] <= NB12_DECISION_THRESHOLDS["maximum_ood_penalty_bpm"]
        and row["high_hr_delta_mae"] <= NB12_DECISION_THRESHOLDS["maximum_high_hr_mae_penalty_bpm"]
        and row["high_hr_severe_underprediction_rate_delta"]
        <= NB12_DECISION_THRESHOLDS["maximum_severe_underprediction_rate_increase"]
    ):
        row["decision_status"] = "research_candidate_for_review"
    elif row["val_delta_mean"] < 0 and row["test_delta_mean"] < 0:
        row["decision_status"] = "analysis_only_tradeoff"
    else:
        row["decision_status"] = "reject_or_debug"

    return row


def nb12_hh_evaluate_all_roles(model: nn.Module, experiment_name: str) -> tuple[pd.DataFrame, dict]:
    """Evaluate a trial model on all NB12 roles."""
    prediction_frames = []

    for role in NB12_HH_ROLE_ORDER:
        prediction_frames.append(
            nb12_hh_evaluate_role(
                model=model,
                dataset=DATASETS[role],
                loader=EVAL_LOADERS[role],
            )
        )

    predictions = pd.concat(prediction_frames, axis=0).reset_index(drop=True)
    bundle = nb12_hh_build_bundle(predictions, experiment_name)

    return predictions, bundle


def nb12_hh_run_trial(trial: optuna.Trial, smoke_mode: bool) -> dict:
    """Run one complete high-HR-aware transfer trial."""
    params = nb12_hh_suggest_params(trial)

    max_epochs = NB12_HH_SMOKE_MAX_EPOCHS if smoke_mode else 12
    patience = max_epochs if smoke_mode else 4

    train_dataset = nb12_hh_make_training_dataset(smoke_mode=smoke_mode)
    sampling_summary = nb12_hh_summarize_sampling_weights(train_dataset, params)
    train_loader = nb12_hh_make_training_loader(
        dataset=train_dataset,
        params=params,
        seed=SEED + trial.number,
    )

    model, freezing_report = nb12_hh_build_model(params)

    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"]),
    )
    loss_fn = nn.HuberLoss(delta=float(params["huber_delta"]))

    best_score = math.inf
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_metrics = nb12_hh_train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            params=params,
        )

        val_predictions = nb12_hh_evaluate_role(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=EVAL_LOADERS["finetune_val"],
        )

        validation_score = nb12_hh_compute_validation_score_from_predictions(
            val_predictions=val_predictions,
            experiment_name=f"high_hr_trial_{trial.number:03d}_epoch_{epoch:02d}",
        )

        improved = validation_score["high_hr_validation_score"] < (
            best_score - NB12_HH_TRIAL_MIN_DELTA
        )

        if improved:
            best_score = float(validation_score["high_hr_validation_score"])
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history.append(
            {
                "epoch": epoch,
                **train_metrics,
                **validation_score,
                "best_epoch_so_far": int(best_epoch),
                "best_score_so_far": float(best_score),
                "improved": bool(improved),
            }
        )

        trial.report(float(validation_score["high_hr_validation_score"]), step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    predictions, bundle = nb12_hh_evaluate_all_roles(
        model=model,
        experiment_name=f"high_hr_trial_{trial.number:03d}",
    )

    final_validation_score = nb12_hh_compute_validation_score_from_error_table(
        bundle["error_table"]
    )
    comparison_row = nb12_hh_compare_to_frozen(bundle["decision_metrics"])

    summary_row = {
        "trial_number": int(trial.number),
        "smoke_mode": bool(smoke_mode),
        "epochs_run": int(len(history)),
        **params,
        **freezing_report,
        **final_validation_score,
        **comparison_row,
    }

    trial.set_user_attr("summary", summary_row)
    trial.set_user_attr("history", history)

    return {
        "summary": summary_row,
        "history": pd.DataFrame(history),
        "sampling_summary": sampling_summary,
        "predictions": predictions,
        "bundle": bundle,
        "objective_value": final_validation_score["high_hr_validation_score"],
    }


NB12_HIGH_HR_SMOKE_RESULTS = []


def nb12_hh_smoke_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the high-HR-aware smoke test."""
    result = nb12_hh_run_trial(trial=trial, smoke_mode=True)
    NB12_HIGH_HR_SMOKE_RESULTS.append(result)
    return float(result["objective_value"])


high_hr_smoke_sampler = optuna.samplers.TPESampler(seed=SEED)
high_hr_smoke_study = optuna.create_study(
    direction="minimize",
    sampler=high_hr_smoke_sampler,
    study_name="nb12_high_hr_aware_transfer_smoke_clean",
)

high_hr_smoke_study.enqueue_trial(
    {
        "lr": 8e-6,
        "weight_decay": 2e-5,
        "dropout": 0.08,
        "huber_delta": 6.0,
        "batch_size": 256,
        "grad_clip": 6.0,
        "freeze_encoder": False,
        "freeze_batchnorm": True,
        "dataset_balance_power": 0.7,
        "high_hr_weight": 4.0,
        "tachy_weight": 8.0,
        "ubfc_rppg_weight": 3.0,
    }
)

high_hr_smoke_study.optimize(
    nb12_hh_smoke_objective,
    n_trials=1,
    show_progress_bar=False,
)

NB12_HIGH_HR_SMOKE_RESULT = NB12_HIGH_HR_SMOKE_RESULTS[0]
NB12_HIGH_HR_SMOKE_HISTORY = NB12_HIGH_HR_SMOKE_RESULT["history"]
NB12_HIGH_HR_SMOKE_BUNDLE = NB12_HIGH_HR_SMOKE_RESULT["bundle"]
NB12_HIGH_HR_SMOKE_SAMPLING_SUMMARY = NB12_HIGH_HR_SMOKE_RESULT["sampling_summary"]
NB12_HIGH_HR_SMOKE_SUMMARY = pd.DataFrame([NB12_HIGH_HR_SMOKE_RESULT["summary"]])

numeric_cols = NB12_HIGH_HR_SMOKE_SUMMARY.select_dtypes(include=[np.number]).columns
NB12_HIGH_HR_SMOKE_SUMMARY[numeric_cols] = NB12_HIGH_HR_SMOKE_SUMMARY[numeric_cols].round(4)

print("High-HR-aware smoke sampling summary:")
display(NB12_HIGH_HR_SMOKE_SAMPLING_SUMMARY)

print("\nHigh-HR-aware smoke trial history:")
display(NB12_HIGH_HR_SMOKE_HISTORY.round(4))

print("\nHigh-HR-aware smoke role summary:")
display(NB12_HIGH_HR_SMOKE_BUNDLE["role_summary"])

print("\nHigh-HR-aware smoke slice summary:")
display(NB12_HIGH_HR_SMOKE_BUNDLE["slice_summary"])

print("\nHigh-HR-aware smoke trial summary:")
display(NB12_HIGH_HR_SMOKE_SUMMARY)

print("\nBest high-HR smoke study value:")
print(high_hr_smoke_study.best_value)

[I 2026-07-16 16:54:41,190] A new study created in memory with name: nb12_high_hr_aware_transfer_smoke_clean
[I 2026-07-16 16:54:49,169] Trial 0 finished with value: 6.081617200931174 and parameters: {'lr': 8e-06, 'weight_decay': 2e-05, 'dropout': 0.08, 'huber_delta': 6.0, 'batch_size': 256, 'grad_clip': 6.0, 'freeze_encoder': False, 'freeze_batchnorm': True, 'dataset_balance_power': 0.7, 'high_hr_weight': 4.0, 'tachy_weight': 8.0, 'ubfc_rppg_weight': 3.0}. Best is trial 0 with value: 6.081617200931174.


High-HR-aware smoke sampling summary:


,dataset,hr_slice,rows,target_hr_mean,weight_mean,weight_min,weight_max,total_weight
0,mcd_rppg,100_120,14,109.2166,8.6307,8.6307,8.6307,120.8295
1,mcd_rppg,lt_100,242,76.9521,2.1577,2.1577,2.1577,522.1560
2,ubfc_phys,100_120,22,104.1228,8.6307,8.6307,8.6307,189.8749
3,ubfc_phys,lt_100,234,78.2026,2.1577,2.1577,2.1577,504.8946
4,ubfc_rppg,100_120,92,110.6043,25.8920,25.8920,25.8920,2382.0669
5,ubfc_rppg,ge_120,13,125.6811,207.1363,207.1363,207.1363,2692.7713
6,ubfc_rppg,lt_100,151,87.8259,6.4730,6.4730,6.4730,977.4242



High-HR-aware smoke trial history:


,epoch,train_loss,train_mae,grad_norm_mean,grad_norm_max,high_hr_validation_score,val_mae_mean,val_mae_p90,val_high_hr_mae,val_high_hr_underprediction_rate,val_high_hr_mae_worsening,val_high_hr_underprediction_rate_worsening,val_p90_worsening,best_epoch_so_far,best_score_so_far,improved
0,1,38.9487,8.9301,291.8706,303.5443,6.0816,6.0731,14.1550,8.7601,0.3214,0.0,0.0,0.0849,1,6.0816,True
1,2,39.3273,9.0382,162.9579,228.0955,6.1411,6.1227,14.2538,8.6334,0.3036,0.0,0.0,0.1837,1,6.0816,False


...


High-HR-aware smoke slice summary:


,slice,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_p90,signed_error_mean,severe_underprediction_rate
0,all_windows,19052,81.4994,82.8389,6.2676,14.6501,1.3395,0.0679
1,high_hr_ge_100,2125,111.5762,104.7885,10.5069,24.9613,-6.7877,0.2965
2,exercise_tachy_ge_120,350,133.0930,118.6972,17.6822,53.8493,-14.3957,0.3829
3,ecg_fitness_ood,882,109.2048,101.6220,13.9599,37.5042,-7.5827,0.3050
4,ecg_fitness_high_hr_ge_100,584,122.0550,111.2257,17.2126,43.9988,-10.8293,0.3904
5,ecg_fitness_exercise_tachy_ge_120,275,135.6331,118.5475,20.2959,54.5580,-17.0856,0.4327



High-HR-aware smoke trial summary:


,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,ecg_fitness_signed_error_mean,ecg_fitness_severe_underprediction_rate,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,decision_status
0,0,True,2,0.0,0.0,0.08,6.0,256,6.0,False,...,-7.5827,0.305,0.0456,0.0453,-0.0756,-0.1442,0.3805,-0.0061,-0.0455,reject_or_debug



Best high-HR smoke study value:
6.081617200931174


## Full High-HR-Aware Transfer Search

This section runs the full high-HR-aware transfer search.

ECG-Fitness remains excluded from training and remains an out-of-domain exercise/high-HR stress test.

This search tests whether weighted sampling and high-HR-aware validation scoring can reduce the trade-off observed in the first transfer search.

A useful trial should not be judged by validation score alone. It should be compared against the frozen baseline and against the first NB12 transfer search using validation gain, test gain, ECG-Fitness penalty, high-HR MAE change, and severe underprediction change.

In [12]:
NB12_HIGH_HR_FULL_RESULTS = []
NB12_HIGH_HR_FULL_N_TRIALS = 12
NB12_HIGH_HR_FULL_DIR = NB12_WORKING_DIR / "transfer_high_hr_aware_exclude_ecg_full"
NB12_HIGH_HR_FULL_DIR.mkdir(parents=True, exist_ok=True)


def nb12_hh_full_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the full high-HR-aware transfer search."""
    result = nb12_hh_run_trial(trial=trial, smoke_mode=False)
    NB12_HIGH_HR_FULL_RESULTS.append(result)
    return float(result["objective_value"])


def nb12_hh_build_completed_trial_summary(study: optuna.Study) -> pd.DataFrame:
    """Build a completed-trial summary table from the high-HR-aware study."""
    rows = []

    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue

        summary = trial.user_attrs.get("summary")

        if not isinstance(summary, dict):
            continue

        row = dict(summary)
        row["optuna_value"] = float(trial.value)
        row["optuna_state"] = str(trial.state.name)
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    table = pd.DataFrame(rows)

    sort_cols = [
        "decision_status",
        "high_hr_validation_score",
        "in_distribution_mean_gain",
        "ood_delta_mean",
        "high_hr_delta_mae",
    ]
    existing_sort_cols = [col for col in sort_cols if col in table.columns]

    if existing_sort_cols:
        table = table.sort_values(existing_sort_cols, ascending=[True] * len(existing_sort_cols))

    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

    return table.reset_index(drop=True)


def nb12_hh_save_full_search_artifacts(
    study: optuna.Study,
    trial_summary: pd.DataFrame,
    output_dir: Path,
) -> dict:
    """Save high-HR-aware full search artifacts."""
    output_dir.mkdir(parents=True, exist_ok=True)

    trial_summary_path = output_dir / "nb12_high_hr_aware_trial_summary.csv"
    best_params_path = output_dir / "nb12_high_hr_aware_best_params.json"
    study_trials_path = output_dir / "nb12_high_hr_aware_optuna_trials.csv"

    trial_summary.to_csv(trial_summary_path, index=False)

    with best_params_path.open("w", encoding="utf-8") as file:
        json.dump(study.best_params, file, indent=2)

    study.trials_dataframe().to_csv(study_trials_path, index=False)

    return {
        "trial_summary_csv": str(trial_summary_path),
        "best_params_json": str(best_params_path),
        "study_trials_csv": str(study_trials_path),
    }


high_hr_full_sampler = optuna.samplers.TPESampler(seed=SEED)
high_hr_full_pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=3,
)

high_hr_full_study = optuna.create_study(
    direction="minimize",
    sampler=high_hr_full_sampler,
    pruner=high_hr_full_pruner,
    study_name="nb12_high_hr_aware_transfer_full",
)

high_hr_full_study.enqueue_trial(
    {
        "lr": 8e-6,
        "weight_decay": 2e-5,
        "dropout": 0.08,
        "huber_delta": 6.0,
        "batch_size": 256,
        "grad_clip": 6.0,
        "freeze_encoder": False,
        "freeze_batchnorm": True,
        "dataset_balance_power": 0.7,
        "high_hr_weight": 4.0,
        "tachy_weight": 8.0,
        "ubfc_rppg_weight": 3.0,
    }
)

high_hr_full_study.optimize(
    nb12_hh_full_objective,
    n_trials=NB12_HIGH_HR_FULL_N_TRIALS,
    show_progress_bar=True,
)

NB12_HIGH_HR_TRIAL_SUMMARY = nb12_hh_build_completed_trial_summary(
    high_hr_full_study
)

NB12_HIGH_HR_FULL_ARTIFACTS = nb12_hh_save_full_search_artifacts(
    study=high_hr_full_study,
    trial_summary=NB12_HIGH_HR_TRIAL_SUMMARY,
    output_dir=NB12_HIGH_HR_FULL_DIR,
)

print("NB12 high-HR-aware search best value:")
print(high_hr_full_study.best_value)

print("\nNB12 high-HR-aware search best params:")
print(json.dumps(high_hr_full_study.best_params, indent=2))

print("\nNB12 high-HR-aware trial summary:")
display(NB12_HIGH_HR_TRIAL_SUMMARY)

print("\nNB12 high-HR-aware search artifacts:")
print(json.dumps(NB12_HIGH_HR_FULL_ARTIFACTS, indent=2))

[I 2026-07-16 16:54:49,305] A new study created in memory with name: nb12_high_hr_aware_transfer_full


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-07-16 16:57:05,047] Trial 0 finished with value: 6.233108373697516 and parameters: {'lr': 8e-06, 'weight_decay': 2e-05, 'dropout': 0.08, 'huber_delta': 6.0, 'batch_size': 256, 'grad_clip': 6.0, 'freeze_encoder': False, 'freeze_batchnorm': True, 'dataset_balance_power': 0.7, 'high_hr_weight': 4.0, 'tachy_weight': 8.0, 'ubfc_rppg_weight': 3.0}. Best is trial 0 with value: 6.233108373697516.
[I 2026-07-16 16:58:16,048] Trial 1 finished with value: 8.391270454382338 and parameters: {'lr': 1.9906996673933362e-05, 'weight_decay': 0.0032859708169642424, 'dropout': 0.2329984854528513, 'huber_delta': 8.38792635777333, 'batch_size': 128, 'grad_clip': 1.4065852851773961, 'freeze_encoder': False, 'freeze_batchnorm': False, 'dataset_balance_power': 0.9699098521619943, 'high_hr_weight': 6.827098485602952, 'tachy_weight': 3.3357302174610375, 'ubfc_rppg_weight': 1.909124836035503}. Best is trial 0 with value: 6.233108373697516.
[I 2026-07-16 17:00:30,912] Trial 2 finished with value: 6.8127849

,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,decision_status,optuna_value,optuna_state
0,11,False,12,0.0001,0.0001,0.1156,3.2605,128,7.9825,True,...,-0.2283,-0.0476,1.0852,0.3735,-1.3725,0.0287,0.1379,analysis_only_tradeoff,5.8103,COMPLETE
1,3,False,12,0.0000,0.0000,0.0663,11.5400,128,3.1323,True,...,-0.1828,-0.0482,1.4839,0.4524,-1.5384,0.0310,0.1155,analysis_only_tradeoff,5.8447,COMPLETE
2,9,False,6,0.0001,0.0021,0.1295,3.9905,256,6.7261,False,...,-0.0002,0.1285,0.2212,-0.2594,0.5610,-0.0141,-0.0642,reject_or_debug,6.0273,COMPLETE
3,0,False,12,0.0000,0.0000,0.0800,6.0000,256,6.0000,False,...,0.1928,0.3416,-0.6741,-1.1751,1.7964,-0.0522,-0.2672,reject_or_debug,6.2331,COMPLETE
4,6,False,5,0.0001,0.0007,0.0685,6.2262,256,5.3631,False,...,0.3650,0.5358,-1.1060,-1.6778,2.4192,-0.0824,-0.4504,reject_or_debug,6.4486,COMPLETE
5,2,False,12,0.0000,0.0000,0.1812,6.8875,256,1.9765,True,...,0.6939,0.7134,-0.4063,-0.9781,3.9562,-0.0908,-0.7037,reject_or_debug,6.8128,COMPLETE
6,1,False,6,0.0000,0.0033,0.2330,8.3879,128,1.4066,False,...,2.0533,2.0558,-0.2231,-0.6880,7.0304,-0.1412,-2.0546,reject_or_debug,8.3913,COMPLETE



NB12 high-HR-aware search artifacts:
{
  "trial_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/transfer_high_hr_aware_exclude_ecg_full/nb12_high_hr_aware_trial_summary.csv",
  "best_params_json": "/kaggle/working/crvse_nb12_optuna_search/transfer_high_hr_aware_exclude_ecg_full/nb12_high_hr_aware_best_params.json",
  "study_trials_csv": "/kaggle/working/crvse_nb12_optuna_search/transfer_high_hr_aware_exclude_ecg_full/nb12_high_hr_aware_optuna_trials.csv"
}


## Transfer Search Comparison

This section compares the frozen source-FPS baseline, the first transfer-search best trial, and the high-HR-aware transfer-search best trial.

The goal is not only to identify the lowest validation MAE. The goal is to inspect the trade-off between in-distribution improvement and high-HR / ECG-Fitness degradation.

A candidate that improves validation and test while worsening ECG-Fitness and high-HR behavior remains a research artifact, not an app checkpoint.

In [13]:
def get_best_trial_row(trial_summary: pd.DataFrame, label: str, score_col: str) -> dict:
    """Return the best completed trial row from one search summary."""
    if trial_summary.empty:
        raise ValueError(f"{label} trial summary is empty.")

    if score_col not in trial_summary.columns:
        raise KeyError(f"{label} summary is missing score column: {score_col}")

    best = trial_summary.sort_values(score_col, ascending=True).iloc[0].to_dict()
    best["comparison_label"] = label
    best["selection_score_col"] = score_col
    best["selection_score_value"] = float(best[score_col])

    return best


def build_frozen_comparison_row() -> dict:
    """Build the frozen baseline row for transfer-search comparison."""
    return {
        "comparison_label": "frozen_sourcefps_baseline",
        "trial_number": -1,
        "selection_score_col": "baseline",
        "selection_score_value": 0.0,
        "val_delta_mean": 0.0,
        "test_delta_mean": 0.0,
        "ood_delta_mean": 0.0,
        "high_hr_delta_mae": 0.0,
        "high_hr_signed_error_delta": 0.0,
        "high_hr_severe_underprediction_rate_delta": 0.0,
        "in_distribution_mean_gain": 0.0,
        "decision_status": "frozen_reference",
        "val_mae_mean": FROZEN_DECISION_METRICS["val_mae_mean"],
        "test_mae_mean": FROZEN_DECISION_METRICS["test_mae_mean"],
        "ood_mae_mean": FROZEN_DECISION_METRICS["ood_mae_mean"],
        "high_hr_mae_mean": FROZEN_DECISION_METRICS["high_hr_mae_mean"],
        "high_hr_signed_error_mean": FROZEN_DECISION_METRICS["high_hr_signed_error_mean"],
        "high_hr_severe_underprediction_rate": FROZEN_DECISION_METRICS[
            "high_hr_severe_underprediction_rate"
        ],
    }


comparison_rows = [
    build_frozen_comparison_row(),
    get_best_trial_row(
        NB12_TRANSFER_TRIAL_SUMMARY,
        label="first_transfer_best",
        score_col="validation_risk_score",
    ),
    get_best_trial_row(
        NB12_HIGH_HR_TRIAL_SUMMARY,
        label="high_hr_aware_best",
        score_col="high_hr_validation_score",
    ),
]

NB12_TRANSFER_SEARCH_COMPARISON = pd.DataFrame(comparison_rows)

comparison_columns = [
    "comparison_label",
    "trial_number",
    "selection_score_col",
    "selection_score_value",
    "decision_status",
    "val_delta_mean",
    "test_delta_mean",
    "ood_delta_mean",
    "high_hr_delta_mae",
    "high_hr_signed_error_delta",
    "high_hr_severe_underprediction_rate_delta",
    "in_distribution_mean_gain",
    "val_mae_mean",
    "test_mae_mean",
    "ood_mae_mean",
    "high_hr_mae_mean",
    "high_hr_signed_error_mean",
    "high_hr_severe_underprediction_rate",
]

NB12_TRANSFER_SEARCH_COMPARISON = NB12_TRANSFER_SEARCH_COMPARISON[
    comparison_columns
].copy()

numeric_cols = NB12_TRANSFER_SEARCH_COMPARISON.select_dtypes(include=[np.number]).columns
NB12_TRANSFER_SEARCH_COMPARISON[numeric_cols] = (
    NB12_TRANSFER_SEARCH_COMPARISON[numeric_cols].round(4)
)

comparison_path = NB12_WORKING_DIR / "nb12_transfer_search_comparison.csv"
NB12_TRANSFER_SEARCH_COMPARISON.to_csv(comparison_path, index=False)

NB12_TRANSFER_COMPARISON_INTERPRETATION = {
    "frozen_checkpoint_remains_app_checkpoint": True,
    "first_transfer_result": (
        "Largest in-distribution gain but larger ECG-Fitness and high-HR penalty."
    ),
    "high_hr_aware_result": (
        "Smaller in-distribution gain but reduced high-HR harm compared with first transfer search."
    ),
    "adopt_any_transfer_checkpoint": False,
    "recommended_next_branch": (
        "ECG-Fitness-included subject-level split, where ECG-Fitness is no longer called OOD."
    ),
}

interpretation_path = NB12_WORKING_DIR / "nb12_transfer_comparison_interpretation.json"

with interpretation_path.open("w", encoding="utf-8") as file:
    json.dump(NB12_TRANSFER_COMPARISON_INTERPRETATION, file, indent=2)

print("NB12 transfer search comparison:")
display(NB12_TRANSFER_SEARCH_COMPARISON)

print("\nNB12 transfer comparison interpretation:")
print(json.dumps(NB12_TRANSFER_COMPARISON_INTERPRETATION, indent=2))

print("\nSaved comparison artifacts:")
print(json.dumps(
    {
        "comparison_csv": str(comparison_path),
        "interpretation_json": str(interpretation_path),
    },
    indent=2,
))

NB12 transfer search comparison:


,comparison_label,trial_number,selection_score_col,selection_score_value,decision_status,val_delta_mean,test_delta_mean,ood_delta_mean,high_hr_delta_mae,high_hr_signed_error_delta,high_hr_severe_underprediction_rate_delta,in_distribution_mean_gain,val_mae_mean,test_mae_mean,ood_mae_mean,high_hr_mae_mean,high_hr_signed_error_mean,high_hr_severe_underprediction_rate
0,frozen_sourcefps_baseline,-1,baseline,0.0000,frozen_reference,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,6.0275,5.7696,14.0355,10.6511,-7.1682,0.3026
1,first_transfer_best,10,validation_risk_score,5.7913,analysis_only_tradeoff,-0.2732,-0.1466,1.0024,0.8019,-1.9976,0.0466,0.2099,5.7543,5.6230,15.0379,11.4530,-9.1658,0.3492
2,high_hr_aware_best,11,high_hr_validation_score,5.8103,analysis_only_tradeoff,-0.2283,-0.0476,1.0852,0.3735,-1.3725,0.0287,0.1379,5.7992,5.7220,15.1207,11.0246,-8.5407,0.3313



NB12 transfer comparison interpretation:
{
  "frozen_checkpoint_remains_app_checkpoint": true,
  "first_transfer_result": "Largest in-distribution gain but larger ECG-Fitness and high-HR penalty.",
  "high_hr_aware_result": "Smaller in-distribution gain but reduced high-HR harm compared with first transfer search.",
  "adopt_any_transfer_checkpoint": false,
  "recommended_next_branch": "ECG-Fitness-included subject-level split, where ECG-Fitness is no longer called OOD."
}

Saved comparison artifacts:
{
  "comparison_csv": "/kaggle/working/crvse_nb12_optuna_search/nb12_transfer_search_comparison.csv",
  "interpretation_json": "/kaggle/working/crvse_nb12_optuna_search/nb12_transfer_comparison_interpretation.json"
}


## ECG-Fitness-Included Subject-Level Branch

This section starts the next NB12 experiment branch.

The previous branches excluded ECG-Fitness from training and treated it as out-of-domain exercise/high-HR evaluation. Those branches showed that transfer learning can improve non-ECG validation and test windows, but still worsens ECG-Fitness and high-HR behavior.

This branch changes the data policy.

ECG-Fitness will be included in training only if subject-level separation can be preserved. Once ECG-Fitness is included, it must no longer be described as out-of-domain. It becomes held-out exercise/high-HR evaluation.

The first step is to audit the available ECG-Fitness metadata and determine whether a subject-level split can be constructed safely.

In [14]:
ECG_BRANCH_ROLE_NAME = "ecg_fitness_subject_split"

ECG_SPLIT_FRACTIONS = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15,
}

ECG_SPLIT_SEED = SEED + 1200


def assert_ecg_branch_prerequisites() -> None:
    """Check that the HR distribution table exists before ECG split auditing."""
    required_names = [
        "NB12_HR_DISTRIBUTION_TABLE",
        "DATASETS",
        "SEED",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the HR distribution diagnostic before this ECG branch. "
            f"Missing names: {missing}"
        )


def get_candidate_subject_columns(df: pd.DataFrame) -> list[str]:
    """Return likely subject identifier columns available in a metadata table."""
    candidates = [
        "subject_id",
        "subject",
        "participant_id",
        "participant",
        "person_id",
        "user_id",
    ]

    return [column for column in candidates if column in df.columns]


def choose_subject_column(df: pd.DataFrame) -> str:
    """Choose the subject identifier column for subject-level splitting."""
    candidates = get_candidate_subject_columns(df)

    if len(candidates) == 0:
        raise KeyError(
            "Could not find a subject identifier column. "
            f"Available columns: {list(df.columns)}"
        )

    if "subject_id" in candidates:
        return "subject_id"

    return candidates[0]


def build_ecg_metadata_audit() -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """Build an ECG-Fitness metadata audit table for subject-level splitting."""
    assert_ecg_branch_prerequisites()

    ecg_rows = NB12_HR_DISTRIBUTION_TABLE.loc[
        NB12_HR_DISTRIBUTION_TABLE["dataset"].eq("ecg_fitness")
    ].copy()

    if len(ecg_rows) == 0:
        raise ValueError("No ECG-Fitness rows were found in NB12_HR_DISTRIBUTION_TABLE.")

    subject_col = choose_subject_column(ecg_rows)
    ecg_rows[subject_col] = ecg_rows[subject_col].astype(str)

    subject_summary = (
        ecg_rows
        .groupby(subject_col, dropna=False)
        .agg(
            rows=("target_hr", "size"),
            target_hr_mean=("target_hr", "mean"),
            target_hr_min=("target_hr", "min"),
            target_hr_max=("target_hr", "max"),
            target_hr_p90=("target_hr", lambda values: values.quantile(0.90)),
            high_hr_rows=("is_high_hr_ge_100", "sum"),
            high_hr_fraction=("is_high_hr_ge_100", "mean"),
            exercise_tachy_rows=("is_exercise_tachy_ge_120", "sum"),
            exercise_tachy_fraction=("is_exercise_tachy_ge_120", "mean"),
        )
        .reset_index()
        .rename(columns={subject_col: "subject_id_for_split"})
    )

    numeric_cols = subject_summary.select_dtypes(include=[np.number]).columns
    subject_summary[numeric_cols] = subject_summary[numeric_cols].round(4)

    dataset_audit = pd.DataFrame(
        [
            {
                "dataset": "ecg_fitness",
                "rows": int(len(ecg_rows)),
                "subject_column": subject_col,
                "subjects": int(subject_summary["subject_id_for_split"].nunique()),
                "target_hr_mean": float(ecg_rows["target_hr"].mean()),
                "target_hr_p90": float(ecg_rows["target_hr"].quantile(0.90)),
                "target_hr_min": float(ecg_rows["target_hr"].min()),
                "target_hr_max": float(ecg_rows["target_hr"].max()),
                "high_hr_fraction": float(ecg_rows["is_high_hr_ge_100"].mean()),
                "exercise_tachy_fraction": float(ecg_rows["is_exercise_tachy_ge_120"].mean()),
                "can_subject_split": bool(subject_summary["subject_id_for_split"].nunique() >= 5),
            }
        ]
    )

    numeric_cols = dataset_audit.select_dtypes(include=[np.number]).columns
    dataset_audit[numeric_cols] = dataset_audit[numeric_cols].round(4)

    return dataset_audit, subject_summary, subject_col


ECG_DATASET_AUDIT, ECG_SUBJECT_SUMMARY, ECG_SUBJECT_COLUMN = build_ecg_metadata_audit()

ecg_branch_dir = NB12_WORKING_DIR / "ecg_fitness_subject_split_branch"
ecg_branch_dir.mkdir(parents=True, exist_ok=True)

ecg_dataset_audit_path = ecg_branch_dir / "ecg_fitness_dataset_audit.csv"
ecg_subject_summary_path = ecg_branch_dir / "ecg_fitness_subject_summary.csv"

ECG_DATASET_AUDIT.to_csv(ecg_dataset_audit_path, index=False)
ECG_SUBJECT_SUMMARY.to_csv(ecg_subject_summary_path, index=False)

print("ECG-Fitness dataset audit:")
display(ECG_DATASET_AUDIT)

print("\nECG-Fitness subject summary:")
display(ECG_SUBJECT_SUMMARY)

print("\nECG branch artifact paths:")
print(json.dumps(
    {
        "dataset_audit_csv": str(ecg_dataset_audit_path),
        "subject_summary_csv": str(ecg_subject_summary_path),
    },
    indent=2,
))

ECG-Fitness dataset audit:


,dataset,rows,subject_column,subjects,target_hr_mean,target_hr_p90,target_hr_min,target_hr_max,high_hr_fraction,exercise_tachy_fraction,can_subject_split
0,ecg_fitness,882,subject_id,17,109.2048,140.2072,53.6622,167.9681,0.6621,0.3118,True



ECG-Fitness subject summary:


,subject_id_for_split,rows,target_hr_mean,target_hr_min,target_hr_max,target_hr_p90,high_hr_rows,high_hr_fraction,exercise_tachy_rows,exercise_tachy_fraction
0,0,41,106.7226,86.5141,143.8089,128.2949,22,0.5366,8,0.1951
1,1,56,99.4836,73.4702,114.8775,111.7872,34,0.6071,0,0.0000
2,10,56,117.0770,79.5454,154.7931,144.2288,42,0.7500,29,0.5179
3,11,41,106.3466,91.3110,130.8454,124.9723,23,0.5610,10,0.2439
4,12,42,111.3591,93.8810,130.4199,123.0164,35,0.8333,8,0.1905
5,13,42,86.7743,53.6622,134.8392,129.7546,14,0.3333,14,0.3333
6,14,55,113.6116,73.6142,149.1069,139.8814,42,0.7636,23,0.4182
7,15,56,103.9838,74.6032,129.7591,121.6088,37,0.6607,8,0.1429
8,16,56,98.7626,79.8704,118.0914,112.9393,27,0.4821,0,0.0000
9,2,54,95.2953,55.9702,130.2315,120.3095,27,0.5000,6,0.1111



ECG branch artifact paths:
{
  "dataset_audit_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_dataset_audit.csv",
  "subject_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_subject_summary.csv"
}


## ECG-Fitness Subject-Level Split Selection

This section creates an ECG-Fitness subject-level split.

The split is selected at the subject level, not the window level, to avoid subject leakage.

Because ECG-Fitness has only 17 subjects, a single random split may accidentally place most exercise/tachy subjects into one role. This section samples many candidate subject splits and selects the split that best balances:

- subject count,
- window count,
- high-HR fraction,
- exercise/tachy fraction,
- target HR p90.

After this split, ECG-Fitness is no longer treated as out-of-domain for the ECG-included branch. It becomes held-out exercise/high-HR evaluation.

In [15]:
ECG_SUBJECT_SPLIT_COUNTS = {
    "train": 11,
    "val": 3,
    "test": 3,
}

ECG_SPLIT_SEARCH_CANDIDATES = 5000


def assign_ecg_subject_split_from_subjects(
    train_subjects: set[str],
    val_subjects: set[str],
    test_subjects: set[str],
) -> pd.DataFrame:
    """Assign ECG-Fitness rows to train, validation, or test by subject."""
    ecg_rows = NB12_HR_DISTRIBUTION_TABLE.loc[
        NB12_HR_DISTRIBUTION_TABLE["dataset"].eq("ecg_fitness")
    ].copy()

    ecg_rows[ECG_SUBJECT_COLUMN] = ecg_rows[ECG_SUBJECT_COLUMN].astype(str)

    def assign_role(subject_id: str) -> str:
        if subject_id in train_subjects:
            return "ecg_subject_train"
        if subject_id in val_subjects:
            return "ecg_subject_val"
        if subject_id in test_subjects:
            return "ecg_subject_test"
        raise ValueError(f"Subject {subject_id} was not assigned to a split.")

    ecg_rows["ecg_subject_role"] = ecg_rows[ECG_SUBJECT_COLUMN].apply(assign_role)

    return ecg_rows


def summarize_ecg_subject_split(split_rows: pd.DataFrame) -> pd.DataFrame:
    """Summarize an ECG-Fitness subject split by role."""
    summary = (
        split_rows
        .groupby("ecg_subject_role", dropna=False)
        .agg(
            subjects=(ECG_SUBJECT_COLUMN, lambda values: values.astype(str).nunique()),
            rows=("target_hr", "size"),
            target_hr_mean=("target_hr", "mean"),
            target_hr_p90=("target_hr", lambda values: values.quantile(0.90)),
            target_hr_min=("target_hr", "min"),
            target_hr_max=("target_hr", "max"),
            high_hr_fraction=("is_high_hr_ge_100", "mean"),
            exercise_tachy_fraction=("is_exercise_tachy_ge_120", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    role_order = {
        "ecg_subject_train": 0,
        "ecg_subject_val": 1,
        "ecg_subject_test": 2,
    }

    summary["role_order"] = summary["ecg_subject_role"].map(role_order)
    summary = summary.sort_values("role_order").drop(columns=["role_order"])

    return summary.reset_index(drop=True)


def score_ecg_subject_split(split_summary: pd.DataFrame) -> float:
    """Score how balanced an ECG-Fitness subject split is."""
    indexed = split_summary.set_index("ecg_subject_role")

    train = indexed.loc["ecg_subject_train"]
    val = indexed.loc["ecg_subject_val"]
    test = indexed.loc["ecg_subject_test"]

    eval_roles = [val, test]

    score = 0.0

    for eval_role in eval_roles:
        score += abs(float(eval_role["high_hr_fraction"]) - float(train["high_hr_fraction"])) * 4.0
        score += abs(float(eval_role["exercise_tachy_fraction"]) - float(train["exercise_tachy_fraction"])) * 6.0
        score += abs(float(eval_role["target_hr_p90"]) - float(train["target_hr_p90"])) / 20.0
        score += abs(float(eval_role["target_hr_mean"]) - float(train["target_hr_mean"])) / 20.0

    score += abs(float(val["rows"]) - float(test["rows"])) / 100.0
    score += abs(float(val["subjects"]) - ECG_SUBJECT_SPLIT_COUNTS["val"]) * 2.0
    score += abs(float(test["subjects"]) - ECG_SUBJECT_SPLIT_COUNTS["test"]) * 2.0

    return float(score)


def sample_ecg_subject_split(
    subjects: list[str],
    rng: np.random.Generator,
) -> tuple[set[str], set[str], set[str]]:
    """Sample one ECG-Fitness subject split."""
    shuffled = list(rng.permutation(subjects))

    n_train = ECG_SUBJECT_SPLIT_COUNTS["train"]
    n_val = ECG_SUBJECT_SPLIT_COUNTS["val"]
    n_test = ECG_SUBJECT_SPLIT_COUNTS["test"]

    train_subjects = set(shuffled[:n_train])
    val_subjects = set(shuffled[n_train:n_train + n_val])
    test_subjects = set(shuffled[n_train + n_val:n_train + n_val + n_test])

    if len(train_subjects | val_subjects | test_subjects) != len(subjects):
        raise ValueError("Subject split lost or duplicated subjects.")

    return train_subjects, val_subjects, test_subjects


def find_balanced_ecg_subject_split() -> dict:
    """Search random subject-level splits and keep the most balanced ECG split."""
    subjects = sorted(ECG_SUBJECT_SUMMARY["subject_id_for_split"].astype(str).tolist())

    expected_total = sum(ECG_SUBJECT_SPLIT_COUNTS.values())

    if len(subjects) != expected_total:
        raise ValueError(
            f"Expected {expected_total} ECG subjects, found {len(subjects)}."
        )

    rng = np.random.default_rng(ECG_SPLIT_SEED)

    best = None
    candidates = []

    for candidate_index in range(ECG_SPLIT_SEARCH_CANDIDATES):
        train_subjects, val_subjects, test_subjects = sample_ecg_subject_split(
            subjects=subjects,
            rng=rng,
        )

        split_rows = assign_ecg_subject_split_from_subjects(
            train_subjects=train_subjects,
            val_subjects=val_subjects,
            test_subjects=test_subjects,
        )

        split_summary = summarize_ecg_subject_split(split_rows)
        split_score = score_ecg_subject_split(split_summary)

        candidate = {
            "candidate_index": candidate_index,
            "score": split_score,
            "train_subjects": sorted(train_subjects),
            "val_subjects": sorted(val_subjects),
            "test_subjects": sorted(test_subjects),
            "split_rows": split_rows,
            "split_summary": split_summary,
        }

        candidates.append(
            {
                "candidate_index": candidate_index,
                "score": split_score,
                "train_subjects": ",".join(sorted(train_subjects)),
                "val_subjects": ",".join(sorted(val_subjects)),
                "test_subjects": ",".join(sorted(test_subjects)),
            }
        )

        if best is None or split_score < best["score"]:
            best = candidate

    candidate_table = pd.DataFrame(candidates).sort_values("score").reset_index(drop=True)

    return {
        "best": best,
        "candidate_table": candidate_table,
    }


ECG_SPLIT_SEARCH_RESULT = find_balanced_ecg_subject_split()
ECG_SUBJECT_SPLIT = ECG_SPLIT_SEARCH_RESULT["best"]
ECG_SUBJECT_SPLIT_CANDIDATES = ECG_SPLIT_SEARCH_RESULT["candidate_table"]

ECG_SUBJECT_SPLIT_ROWS = ECG_SUBJECT_SPLIT["split_rows"]
ECG_SUBJECT_SPLIT_SUMMARY = ECG_SUBJECT_SPLIT["split_summary"]

ecg_split_rows_path = ecg_branch_dir / "ecg_fitness_subject_split_rows.csv"
ecg_split_summary_path = ecg_branch_dir / "ecg_fitness_subject_split_summary.csv"
ecg_split_candidates_path = ecg_branch_dir / "ecg_fitness_subject_split_candidates.csv"
ecg_split_config_path = ecg_branch_dir / "ecg_fitness_subject_split_config.json"

ECG_SUBJECT_SPLIT_ROWS.to_csv(ecg_split_rows_path, index=False)
ECG_SUBJECT_SPLIT_SUMMARY.to_csv(ecg_split_summary_path, index=False)
ECG_SUBJECT_SPLIT_CANDIDATES.head(100).to_csv(ecg_split_candidates_path, index=False)

split_config = {
    "split_counts": ECG_SUBJECT_SPLIT_COUNTS,
    "split_seed": ECG_SPLIT_SEED,
    "search_candidates": ECG_SPLIT_SEARCH_CANDIDATES,
    "best_candidate_index": int(ECG_SUBJECT_SPLIT["candidate_index"]),
    "best_score": float(ECG_SUBJECT_SPLIT["score"]),
    "train_subjects": ECG_SUBJECT_SPLIT["train_subjects"],
    "val_subjects": ECG_SUBJECT_SPLIT["val_subjects"],
    "test_subjects": ECG_SUBJECT_SPLIT["test_subjects"],
    "interpretation": (
        "ECG-Fitness is included by subject-level split. It should be described "
        "as held-out exercise/high-HR evaluation, not OOD, in this branch."
    ),
}

with ecg_split_config_path.open("w", encoding="utf-8") as file:
    json.dump(split_config, file, indent=2)

print("Selected ECG-Fitness subject split config:")
print(json.dumps(split_config, indent=2))

print("\nSelected ECG-Fitness subject split summary:")
display(ECG_SUBJECT_SPLIT_SUMMARY)

print("\nTop 10 ECG subject split candidates:")
display(ECG_SUBJECT_SPLIT_CANDIDATES.head(10))

print("\nSaved ECG subject split artifacts:")
print(json.dumps(
    {
        "split_rows_csv": str(ecg_split_rows_path),
        "split_summary_csv": str(ecg_split_summary_path),
        "top_candidates_csv": str(ecg_split_candidates_path),
        "split_config_json": str(ecg_split_config_path),
    },
    indent=2,
))

Selected ECG-Fitness subject split config:
{
  "split_counts": {
    "train": 11,
    "val": 3,
    "test": 3
  },
  "split_seed": 1242,
  "search_candidates": 5000,
  "best_candidate_index": 2206,
  "best_score": 0.8656149999999998,
  "train_subjects": [
    "1",
    "11",
    "12",
    "14",
    "16",
    "2",
    "3",
    "4",
    "6",
    "7",
    "9"
  ],
  "val_subjects": [
    "0",
    "15",
    "8"
  ],
  "test_subjects": [
    "10",
    "13",
    "5"
  ],
  "interpretation": "ECG-Fitness is included by subject-level split. It should be described as held-out exercise/high-HR evaluation, not OOD, in this branch."
}

Selected ECG-Fitness subject split summary:


,ecg_subject_role,subjects,rows,target_hr_mean,target_hr_p90,target_hr_min,target_hr_max,high_hr_fraction,exercise_tachy_fraction
0,ecg_subject_train,11,576,109.8421,141.7694,55.9702,160.7531,0.6667,0.3056
1,ecg_subject_val,3,153,109.2220,142.0442,62.2921,167.9681,0.6536,0.3072
2,ecg_subject_test,3,153,106.7882,134.8099,53.6622,154.7931,0.6536,0.3399



Top 10 ECG subject split candidates:


,candidate_index,score,train_subjects,val_subjects,test_subjects
0,2206,0.865615,"1,11,12,14,16,2,3,4,6,7,9","0,15,8","10,13,5"
1,1524,0.876330,"0,10,11,12,13,15,2,5,6,7,9","1,3,4","14,16,8"
2,2507,0.926740,"0,1,10,11,12,13,14,2,5,6,8","16,4,9","15,3,7"
3,3149,0.939655,"0,11,12,13,16,2,3,5,6,7,9","10,15,4","1,14,8"
4,2420,0.946795,"0,10,12,13,14,16,3,4,5,6,7","1,2,9","11,15,8"
5,2515,0.971650,"1,10,11,12,13,14,15,4,6,7,9","16,2,3","0,5,8"
6,3611,1.011295,"1,11,13,14,15,16,2,3,6,7,9","0,5,8","10,12,4"
7,1708,1.034405,"1,11,12,13,16,3,4,5,6,7,9","0,10,15","14,2,8"
8,1092,1.039375,"0,1,10,11,12,16,2,3,4,6,8","15,7,9","13,14,5"
9,3856,1.050965,"1,10,11,12,13,14,16,5,6,7,9","2,3,4","0,15,8"



Saved ECG subject split artifacts:
{
  "split_rows_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_subject_split_rows.csv",
  "split_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_subject_split_summary.csv",
  "top_candidates_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_subject_split_candidates.csv",
  "split_config_json": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_fitness_subject_split_config.json"
}


## ECG-Included Combined Dataset Audit

This section builds the dataset roles for the ECG-Fitness-included branch.

The branch combines the original non-ECG live-compatible roles with the ECG-Fitness subject-level split:

- training uses original non-ECG training windows plus ECG-Fitness training subjects,
- validation uses original non-ECG validation windows plus ECG-Fitness validation subjects,
- test uses original non-ECG test windows plus ECG-Fitness test subjects.

ECG-Fitness is no longer out-of-domain in this branch. It is included by subject-level split and evaluated on held-out ECG-Fitness subjects.

In [16]:
class CombinedTensorDataset(Dataset):
    """Dataset that stores materialized tensor rows and row-level metadata."""

    def __init__(
        self,
        role: str,
        x: np.ndarray,
        y: np.ndarray,
        metadata: pd.DataFrame,
    ) -> None:
        self.role = role
        self.x = np.asarray(x, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.float32)
        self.metadata = metadata.reset_index(drop=True).copy()

        if len(self.x) != len(self.y):
            raise ValueError(f"{role}: x/y row mismatch: {len(self.x)} vs {len(self.y)}.")

        if len(self.x) != len(self.metadata):
            raise ValueError(
                f"{role}: tensor/metadata row mismatch: {len(self.x)} vs {len(self.metadata)}."
            )

        self.metadata["combined_role"] = role

        if "source_role" not in self.metadata.columns:
            self.metadata["source_role"] = role

    def __len__(self) -> int:
        return int(len(self.x))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.x[index]).float()
        y = torch.tensor(self.y[index], dtype=torch.float32)
        return x, y


def get_ecg_indices_for_subject_role(ecg_subject_role: str) -> np.ndarray:
    """Return row positions in the original ood_eval tensor for one ECG subject role."""
    if "ECG_SUBJECT_SPLIT_ROWS" not in globals():
        raise NameError("Run ECG subject split selection before building combined datasets.")

    role_rows = ECG_SUBJECT_SPLIT_ROWS.loc[
        ECG_SUBJECT_SPLIT_ROWS["ecg_subject_role"].eq(ecg_subject_role)
    ].copy()

    if len(role_rows) == 0:
        raise ValueError(f"No ECG rows found for {ecg_subject_role}.")

    if "row_index" in role_rows.columns:
        indices = role_rows["row_index"].to_numpy(dtype=np.int64)
    else:
        indices = role_rows.index.to_numpy(dtype=np.int64)

    if np.any(indices < 0) or np.any(indices >= len(DATASETS["ood_eval"])):
        raise ValueError(f"Invalid ECG tensor indices for {ecg_subject_role}.")

    return indices


def build_ecg_subject_dataset(ecg_subject_role: str) -> CombinedTensorDataset:
    """Build one ECG-Fitness subject-split dataset from original ECG tensors."""
    indices = get_ecg_indices_for_subject_role(ecg_subject_role)
    base = DATASETS["ood_eval"]

    x = base.x[indices]
    y = base.y[indices]

    metadata = base.metadata.iloc[indices].reset_index(drop=True).copy()
    metadata["ecg_subject_role"] = ecg_subject_role
    metadata["dataset_role_policy"] = "ecg_fitness_subject_split"
    metadata["source_role"] = ecg_subject_role

    return CombinedTensorDataset(
        role=ecg_subject_role,
        x=x,
        y=y,
        metadata=metadata,
    )


def concatenate_datasets(
    role: str,
    datasets: list[Dataset],
) -> CombinedTensorDataset:
    """Concatenate multiple tensor datasets into one combined dataset."""
    xs = []
    ys = []
    metadata_parts = []

    for dataset in datasets:
        xs.append(np.asarray(dataset.x, dtype=np.float32))
        ys.append(np.asarray(dataset.y, dtype=np.float32))

        metadata = dataset.metadata.copy()
        metadata["source_role"] = dataset.role
        metadata_parts.append(metadata)

    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    metadata = pd.concat(metadata_parts, axis=0).reset_index(drop=True)

    return CombinedTensorDataset(
        role=role,
        x=x,
        y=y,
        metadata=metadata,
    )


def summarize_combined_dataset(dataset: CombinedTensorDataset) -> dict:
    """Summarize one tensor dataset used in the ECG-included branch."""
    metadata = dataset.metadata.copy()

    if "dataset" not in metadata.columns:
        raise KeyError(f"{dataset.role}: metadata missing dataset column.")

    if "source_role" not in metadata.columns:
        metadata["source_role"] = dataset.role

    target_hr = dataset.y.astype(float)
    high_hr = target_hr >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    tachy = target_hr >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]

    dataset_counts = metadata["dataset"].value_counts().to_dict()
    source_role_counts = metadata["source_role"].value_counts().to_dict()

    return {
        "role": dataset.role,
        "rows": int(len(dataset)),
        "x_shape": tuple(dataset.x.shape),
        "y_shape": tuple(dataset.y.shape),
        "x_finite": bool(np.isfinite(dataset.x).all()),
        "y_finite": bool(np.isfinite(dataset.y).all()),
        "target_hr_mean": float(np.mean(target_hr)),
        "target_hr_p90": float(np.quantile(target_hr, 0.90)),
        "target_hr_min": float(np.min(target_hr)),
        "target_hr_max": float(np.max(target_hr)),
        "high_hr_rows": int(np.sum(high_hr)),
        "high_hr_fraction": float(np.mean(high_hr)),
        "exercise_tachy_rows": int(np.sum(tachy)),
        "exercise_tachy_fraction": float(np.mean(tachy)),
        "dataset_counts": dataset_counts,
        "source_role_counts": source_role_counts,
    }


ECG_SUBJECT_DATASETS = {
    "ecg_subject_train": build_ecg_subject_dataset("ecg_subject_train"),
    "ecg_subject_val": build_ecg_subject_dataset("ecg_subject_val"),
    "ecg_subject_test": build_ecg_subject_dataset("ecg_subject_test"),
}

ECG_INCLUDED_DATASETS = {
    "ecg_included_train": concatenate_datasets(
        role="ecg_included_train",
        datasets=[
            DATASETS["finetune_train"],
            ECG_SUBJECT_DATASETS["ecg_subject_train"],
        ],
    ),
    "ecg_included_val": concatenate_datasets(
        role="ecg_included_val",
        datasets=[
            DATASETS["finetune_val"],
            ECG_SUBJECT_DATASETS["ecg_subject_val"],
        ],
    ),
    "ecg_included_test": concatenate_datasets(
        role="ecg_included_test",
        datasets=[
            DATASETS["finetune_test"],
            ECG_SUBJECT_DATASETS["ecg_subject_test"],
        ],
    ),
}

ECG_INCLUDED_LOADERS = {
    role: make_eval_loader(dataset)
    for role, dataset in ECG_INCLUDED_DATASETS.items()
}

ECG_INCLUDED_DATASET_AUDIT = pd.DataFrame(
    [summarize_combined_dataset(dataset) for dataset in ECG_INCLUDED_DATASETS.values()]
)

ECG_SUBJECT_DATASET_AUDIT = pd.DataFrame(
    [summarize_combined_dataset(dataset) for dataset in ECG_SUBJECT_DATASETS.values()]
)

for audit_table in [ECG_INCLUDED_DATASET_AUDIT, ECG_SUBJECT_DATASET_AUDIT]:
    numeric_cols = audit_table.select_dtypes(include=[np.number]).columns
    audit_table[numeric_cols] = audit_table[numeric_cols].round(4)

ecg_included_audit_path = ecg_branch_dir / "ecg_included_combined_dataset_audit.csv"
ecg_subject_audit_path = ecg_branch_dir / "ecg_subject_split_dataset_audit.csv"

ECG_INCLUDED_DATASET_AUDIT.to_csv(ecg_included_audit_path, index=False)
ECG_SUBJECT_DATASET_AUDIT.to_csv(ecg_subject_audit_path, index=False)

print("ECG-included combined dataset audit:")
display(ECG_INCLUDED_DATASET_AUDIT)

print("\nECG subject split dataset audit:")
display(ECG_SUBJECT_DATASET_AUDIT)

print("\nSaved ECG-included dataset audits:")
print(json.dumps(
    {
        "combined_dataset_audit_csv": str(ecg_included_audit_path),
        "ecg_subject_dataset_audit_csv": str(ecg_subject_audit_path),
    },
    indent=2,
))

ECG-included combined dataset audit:


,role,rows,x_shape,y_shape,x_finite,y_finite,target_hr_mean,target_hr_p90,target_hr_min,target_hr_max,high_hr_rows,high_hr_fraction,exercise_tachy_rows,exercise_tachy_fraction,dataset_counts,source_role_counts
0,ecg_included_train,13079,"(13079, 3, 240)","(13079,)",True,True,81.5409,101.7016,40.0563,160.7531,1513,0.1157,222,0.0170,"{'mcd_rppg': 7446, 'ubfc_phys': 4595, 'ecg_fitness': 576, 'ubfc_rppg': 462}","{'finetune_train': 12503, 'ecg_subject_train': 576}"
1,ecg_included_val,2992,"(2992, 3, 240)","(2992,)",True,True,81.6478,98.4706,40.2964,167.9681,268,0.0896,59,0.0197,"{'mcd_rppg': 1743, 'ubfc_phys': 986, 'ecg_fitness': 153, 'ubfc_rppg': 110}","{'finetune_val': 2839, 'ecg_subject_val': 153}"
2,ecg_included_test,2981,"(2981, 3, 240)","(2981,)",True,True,81.1683,101.3893,47.6317,154.7931,344,0.1154,69,0.0231,"{'mcd_rppg': 2094, 'ubfc_phys': 658, 'ecg_fitness': 153, 'ubfc_rppg': 76}","{'finetune_test': 2828, 'ecg_subject_test': 153}"



ECG subject split dataset audit:


,role,rows,x_shape,y_shape,x_finite,y_finite,target_hr_mean,target_hr_p90,target_hr_min,target_hr_max,high_hr_rows,high_hr_fraction,exercise_tachy_rows,exercise_tachy_fraction,dataset_counts,source_role_counts
0,ecg_subject_train,576,"(576, 3, 240)","(576,)",True,True,109.8421,141.7694,55.9702,160.7531,384,0.6667,176,0.3056,{'ecg_fitness': 576},{'ecg_subject_train': 576}
1,ecg_subject_val,153,"(153, 3, 240)","(153,)",True,True,109.2220,142.0442,62.2921,167.9681,100,0.6536,47,0.3072,{'ecg_fitness': 153},{'ecg_subject_val': 153}
2,ecg_subject_test,153,"(153, 3, 240)","(153,)",True,True,106.7882,134.8099,53.6622,154.7931,100,0.6536,52,0.3399,{'ecg_fitness': 153},{'ecg_subject_test': 153}



Saved ECG-included dataset audits:
{
  "combined_dataset_audit_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_included_combined_dataset_audit.csv",
  "ecg_subject_dataset_audit_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_subject_split_dataset_audit.csv"
}


## Frozen Baseline Under ECG-Included Policy

This section evaluates the frozen source-FPS checkpoint under the ECG-Fitness-included subject-split policy.

This is a new baseline because ECG-Fitness is no longer treated as a single OOD block. Instead, ECG-Fitness is represented by training subjects, validation subjects, and test subjects.

The purpose is to establish what the current frozen checkpoint does before any ECG-included transfer learning.

In [17]:
def evaluate_model_on_named_datasets(
    model: nn.Module,
    datasets: dict[str, Dataset],
    loaders: dict[str, DataLoader],
    experiment_name: str,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate a model on named datasets and return predictions plus summaries."""
    prediction_frames = []

    for role, dataset in datasets.items():
        predictions = nb12_hh_evaluate_role(
            model=model,
            dataset=dataset,
            loader=loaders[role],
        )
        predictions["evaluation_role"] = role
        prediction_frames.append(predictions)

    all_predictions = pd.concat(prediction_frames, axis=0).reset_index(drop=True)
    error_table = nb12_hh_prepare_error_table(all_predictions, experiment_name)

    role_summary = nb12_hh_summarize_groups(error_table, ["evaluation_role"])
    dataset_role_summary = nb12_hh_summarize_groups(error_table, ["dataset", "evaluation_role"])
    source_role_summary = nb12_hh_summarize_groups(error_table, ["source_role", "evaluation_role"])
    slice_summary = nb12_hh_build_slice_summary(error_table)

    return all_predictions, {
        "error_table": error_table,
        "role_summary": role_summary,
        "dataset_role_summary": dataset_role_summary,
        "source_role_summary": source_role_summary,
        "slice_summary": slice_summary,
    }


ECG_SUBJECT_LOADERS = {
    role: make_eval_loader(dataset)
    for role, dataset in ECG_SUBJECT_DATASETS.items()
}

FROZEN_ECG_INCLUDED_PREDICTIONS, FROZEN_ECG_INCLUDED_BUNDLE = evaluate_model_on_named_datasets(
    model=FROZEN_MODEL,
    datasets=ECG_INCLUDED_DATASETS,
    loaders=ECG_INCLUDED_LOADERS,
    experiment_name="frozen_ecg_included_policy",
)

FROZEN_ECG_SUBJECT_PREDICTIONS, FROZEN_ECG_SUBJECT_BUNDLE = evaluate_model_on_named_datasets(
    model=FROZEN_MODEL,
    datasets=ECG_SUBJECT_DATASETS,
    loaders=ECG_SUBJECT_LOADERS,
    experiment_name="frozen_ecg_subject_split_only",
)

frozen_ecg_dir = ecg_branch_dir / "frozen_ecg_included_baseline"
frozen_ecg_dir.mkdir(parents=True, exist_ok=True)

frozen_ecg_included_role_path = frozen_ecg_dir / "frozen_ecg_included_role_summary.csv"
frozen_ecg_included_dataset_role_path = frozen_ecg_dir / "frozen_ecg_included_dataset_role_summary.csv"
frozen_ecg_included_source_role_path = frozen_ecg_dir / "frozen_ecg_included_source_role_summary.csv"
frozen_ecg_subject_role_path = frozen_ecg_dir / "frozen_ecg_subject_role_summary.csv"

FROZEN_ECG_INCLUDED_BUNDLE["role_summary"].to_csv(frozen_ecg_included_role_path, index=False)
FROZEN_ECG_INCLUDED_BUNDLE["dataset_role_summary"].to_csv(
    frozen_ecg_included_dataset_role_path,
    index=False,
)
FROZEN_ECG_INCLUDED_BUNDLE["source_role_summary"].to_csv(
    frozen_ecg_included_source_role_path,
    index=False,
)
FROZEN_ECG_SUBJECT_BUNDLE["role_summary"].to_csv(frozen_ecg_subject_role_path, index=False)

print("Frozen baseline on ECG-included combined roles:")
display(FROZEN_ECG_INCLUDED_BUNDLE["role_summary"])

print("\nFrozen baseline by dataset within ECG-included roles:")
display(FROZEN_ECG_INCLUDED_BUNDLE["dataset_role_summary"].sort_values(["evaluation_role", "dataset"]))

print("\nFrozen baseline by source role within ECG-included roles:")
display(FROZEN_ECG_INCLUDED_BUNDLE["source_role_summary"].sort_values(["evaluation_role", "source_role"]))

print("\nFrozen baseline on ECG-Fitness subject split only:")
display(FROZEN_ECG_SUBJECT_BUNDLE["role_summary"])

print("\nSaved frozen ECG-included baseline artifacts:")
print(json.dumps(
    {
        "included_role_summary_csv": str(frozen_ecg_included_role_path),
        "included_dataset_role_summary_csv": str(frozen_ecg_included_dataset_role_path),
        "included_source_role_summary_csv": str(frozen_ecg_included_source_role_path),
        "ecg_subject_role_summary_csv": str(frozen_ecg_subject_role_path),
    },
    indent=2,
))

Frozen baseline on ECG-included combined roles:


,evaluation_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,ecg_included_test,2981,81.1683,82.4818,5.8906,3.6987,13.9309,19.6023,72.2401,1.3136,0.0651
1,ecg_included_train,13079,81.5409,82.6109,6.2520,3.8635,14.6407,20.4439,85.1423,1.0700,0.0702
2,ecg_included_val,2992,81.6478,82.5983,6.4474,3.9530,14.7248,20.2721,89.9549,0.9505,0.0809



Frozen baseline by dataset within ECG-included roles:


,dataset,evaluation_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,ecg_fitness,ecg_included_test,153,106.7882,106.1829,8.1276,5.5600,16.8855,32.3661,41.6111,-0.6052,0.1176
3,mcd_rppg,ecg_included_test,2094,79.8533,80.7775,4.8847,3.1502,10.9700,15.5490,49.4722,0.9242,0.0540
6,ubfc_phys,ecg_included_test,658,77.4205,80.6185,8.8584,5.6735,20.2862,25.6733,72.2401,3.1980,0.0942
9,ubfc_rppg,ecg_included_test,76,98.2693,97.8591,3.4085,3.1453,6.0835,6.8394,13.5339,-0.4102,0.0132
1,ecg_fitness,ecg_included_train,576,109.8421,100.1514,15.5510,8.9434,41.9863,53.6865,85.1423,-9.6907,0.3542
4,mcd_rppg,ecg_included_train,7446,79.0648,80.3294,4.6876,3.1639,10.3722,13.9969,54.9838,1.2645,0.0377
7,ubfc_phys,ecg_included_train,4595,80.3055,82.6246,7.7819,5.4248,18.5928,23.4842,50.9827,2.3192,0.0892
10,ubfc_rppg,ecg_included_train,462,98.4492,97.3769,4.6548,3.1447,9.9189,13.1687,30.5685,-1.0723,0.0498
2,ecg_fitness,ecg_included_val,153,109.2220,100.4879,14.2380,7.9894,34.5519,54.8437,89.9549,-8.7341,0.3137
5,mcd_rppg,ecg_included_val,1743,79.1086,80.3743,5.1325,3.3746,11.5368,15.4784,46.6088,1.2656,0.0528



Frozen baseline by source role within ECG-included roles:


,source_role,evaluation_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,ecg_subject_test,ecg_included_test,153,106.7882,106.1829,8.1276,5.5600,16.8855,32.3661,41.6111,-0.6052,0.1176
3,finetune_test,ecg_included_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,72.2401,1.4174,0.0622
1,ecg_subject_train,ecg_included_train,576,109.8421,100.1514,15.5510,8.9434,41.9863,53.6865,85.1423,-9.6907,0.3542
4,finetune_train,ecg_included_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,54.9838,1.5658,0.0571
2,ecg_subject_val,ecg_included_val,153,109.2220,100.4879,14.2380,7.9894,34.5519,54.8437,89.9549,-8.7341,0.3137
5,finetune_val,ecg_included_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,50.2965,1.4724,0.0683



Frozen baseline on ECG-Fitness subject split only:


,evaluation_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,ecg_subject_test,153,106.7882,106.1829,8.1276,5.5600,16.8855,32.3661,41.6111,-0.6052,0.1176
1,ecg_subject_train,576,109.8421,100.1514,15.5510,8.9434,41.9863,53.6865,85.1423,-9.6907,0.3542
2,ecg_subject_val,153,109.2220,100.4879,14.2380,7.9894,34.5519,54.8437,89.9549,-8.7341,0.3137



Saved frozen ECG-included baseline artifacts:
{
  "included_role_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/frozen_ecg_included_baseline/frozen_ecg_included_role_summary.csv",
  "included_dataset_role_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/frozen_ecg_included_baseline/frozen_ecg_included_dataset_role_summary.csv",
  "included_source_role_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/frozen_ecg_included_baseline/frozen_ecg_included_source_role_summary.csv",
  "ecg_subject_role_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/frozen_ecg_included_baseline/frozen_ecg_subject_role_summary.csv"
}


## ECG-Included Transfer Objective Design

This section defines the objective for ECG-Fitness-included transfer learning.

ECG-Fitness is no longer out-of-domain in this branch. It is included by subject-level split.

The objective must avoid hiding ECG behavior inside the larger combined validation set. ECG-Fitness validation subjects are only a small fraction of the combined validation windows, so combined validation MAE alone would underweight the exercise/high-HR question.

The objective therefore separates:

- non-ECG validation behavior,
- ECG-Fitness validation-subject behavior,
- ECG-Fitness validation underprediction,
- combined validation tail error.

A useful model should improve ECG-Fitness validation-subject behavior while preserving non-ECG validation and held-out test behavior.

In [18]:
ECG_INCLUDED_BRANCH_DIR = ecg_branch_dir / "ecg_included_transfer_search"
ECG_INCLUDED_BRANCH_DIR.mkdir(parents=True, exist_ok=True)

ECG_INCLUDED_OBJECTIVE_WEIGHTS = {
    "combined_val_mae": 0.45,
    "non_ecg_val_mae": 0.25,
    "ecg_val_mae": 0.70,
    "ecg_val_underprediction_penalty": 4.0,
    "combined_val_p90_worsening_penalty": 0.10,
}

ECG_INCLUDED_SEARCH_N_TRIALS = 12

ECG_INCLUDED_SAMPLER_SPACE = {
    "dataset_balance_power": {
        "low": 0.0,
        "high": 1.0,
        "reason": "Controls how strongly minority datasets, including ECG-Fitness, are upweighted.",
    },
    "ecg_weight": {
        "low": 1.0,
        "high": 10.0,
        "reason": "Controls how strongly ECG-Fitness training-subject windows are sampled.",
    },
    "high_hr_weight": {
        "low": 1.0,
        "high": 8.0,
        "reason": "Controls how strongly windows >=100 BPM are sampled.",
    },
    "tachy_weight": {
        "low": 1.0,
        "high": 12.0,
        "reason": "Controls how strongly windows >=120 BPM are sampled.",
    },
}


def build_ecg_included_objective_design_table() -> pd.DataFrame:
    """Build a readable ECG-included objective design table."""
    rows = []

    for component, weight in ECG_INCLUDED_OBJECTIVE_WEIGHTS.items():
        rows.append(
            {
                "component": component,
                "weight": weight,
            }
        )

    return pd.DataFrame(rows)


def build_ecg_included_sampler_space_table() -> pd.DataFrame:
    """Build a readable ECG-included sampler search-space table."""
    rows = []

    for name, spec in ECG_INCLUDED_SAMPLER_SPACE.items():
        rows.append(
            {
                "parameter": name,
                "low": spec["low"],
                "high": spec["high"],
                "reason": spec["reason"],
            }
        )

    return pd.DataFrame(rows)


ECG_INCLUDED_OBJECTIVE_DESIGN = {
    "branch_name": "ecg_fitness_included_subject_level_transfer",
    "ecg_fitness_policy": "included_by_subject_split_not_ood",
    "training_dataset": "ecg_included_train",
    "validation_dataset": "ecg_included_val",
    "test_dataset": "ecg_included_test",
    "main_question": (
        "Can transfer learning improve held-out ECG-Fitness validation-subject "
        "behavior while preserving non-ECG validation/test behavior?"
    ),
    "important_warning": (
        "Combined validation MAE can hide ECG-Fitness behavior because ECG-Fitness "
        "is a minority of validation rows."
    ),
    "n_trials_initial": ECG_INCLUDED_SEARCH_N_TRIALS,
}

print("ECG-included objective design:")
print(json.dumps(ECG_INCLUDED_OBJECTIVE_DESIGN, indent=2))

print("\nECG-included objective weights:")
display(build_ecg_included_objective_design_table())

print("\nECG-included sampler search space:")
display(build_ecg_included_sampler_space_table())

ECG-included objective design:
{
  "branch_name": "ecg_fitness_included_subject_level_transfer",
  "ecg_fitness_policy": "included_by_subject_split_not_ood",
  "training_dataset": "ecg_included_train",
  "validation_dataset": "ecg_included_val",
  "test_dataset": "ecg_included_test",
  "main_question": "Can transfer learning improve held-out ECG-Fitness validation-subject behavior while preserving non-ECG validation/test behavior?",
  "important_warning": "Combined validation MAE can hide ECG-Fitness behavior because ECG-Fitness is a minority of validation rows.",
  "n_trials_initial": 12
}

ECG-included objective weights:


,component,weight
0,combined_val_mae,0.45
1,non_ecg_val_mae,0.25
2,ecg_val_mae,0.70
3,ecg_val_underprediction_penalty,4.00
4,combined_val_p90_worsening_penalty,0.10



ECG-included sampler search space:


,parameter,low,high,reason
0,dataset_balance_power,0.0,1.0,"Controls how strongly minority datasets, including ECG-Fitness, are upweighted."
1,ecg_weight,1.0,10.0,Controls how strongly ECG-Fitness training-subject windows are sampled.
2,high_hr_weight,1.0,8.0,Controls how strongly windows >=100 BPM are sampled.
3,tachy_weight,1.0,12.0,Controls how strongly windows >=120 BPM are sampled.


## ECG-Included Transfer Smoke Trial

This section runs one small smoke trial for the ECG-Fitness-included transfer branch.

The smoke trial is not intended to produce a useful checkpoint. It verifies that the ECG-included training dataset, weighted sampler, validation objective, and full evaluation path work together.

ECG-Fitness is included by subject-level split in this branch. It should not be described as out-of-domain here.

In [19]:
ECG_INCLUDED_SMOKE_RESULTS = []
ECG_INCLUDED_SMOKE_ROWS_PER_SOURCE = 384
ECG_INCLUDED_SMOKE_MAX_EPOCHS = 2
ECG_INCLUDED_TRIAL_MIN_DELTA = 1e-3


class SourceBalancedIndexedDataset(Dataset):
    """Dataset view over selected row indices from a combined dataset."""

    def __init__(
        self,
        base_dataset: CombinedTensorDataset,
        indices: np.ndarray,
        role: str,
    ) -> None:
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.role = role
        self.x = base_dataset.x[self.indices]
        self.y = base_dataset.y[self.indices]
        self.metadata = base_dataset.metadata.iloc[self.indices].reset_index(drop=True).copy()

        if len(self.indices) == 0:
            raise ValueError("SourceBalancedIndexedDataset received no indices.")

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        original_index = int(self.indices[index])
        return self.base_dataset[original_index]


def select_ecg_included_smoke_indices(
    dataset: CombinedTensorDataset,
    rows_per_source: int,
    seed: int,
) -> np.ndarray:
    """Select a source-role-balanced smoke subset from the ECG-included training set."""
    metadata = dataset.metadata.copy()

    if "source_role" not in metadata.columns:
        raise KeyError("Combined training metadata is missing source_role.")

    rng = np.random.default_rng(seed)
    metadata["_row_position"] = np.arange(len(metadata))
    selected = []

    for _, group in metadata.groupby("source_role", sort=True):
        positions = group["_row_position"].to_numpy()
        take = min(rows_per_source, len(positions))
        selected.extend(rng.choice(positions, size=take, replace=False).tolist())

    return np.asarray(sorted(selected), dtype=np.int64)


def make_ecg_included_training_dataset(smoke_mode: bool) -> Dataset:
    """Return the ECG-included training dataset for smoke or full search."""
    train_dataset = ECG_INCLUDED_DATASETS["ecg_included_train"]

    if not smoke_mode:
        return train_dataset

    smoke_indices = select_ecg_included_smoke_indices(
        dataset=train_dataset,
        rows_per_source=ECG_INCLUDED_SMOKE_ROWS_PER_SOURCE,
        seed=SEED,
    )

    return SourceBalancedIndexedDataset(
        base_dataset=train_dataset,
        indices=smoke_indices,
        role="ecg_included_train_smoke",
    )


def suggest_ecg_included_trial_params(trial: optuna.Trial) -> dict:
    """Suggest transfer and ECG-included sampling parameters for one trial."""
    return {
        "lr": trial.suggest_float("lr", 5e-6, 2e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.05, 0.30),
        "huber_delta": trial.suggest_float("huber_delta", 3.0, 12.0),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
        "grad_clip": trial.suggest_float("grad_clip", 1.0, 8.0),
        "freeze_encoder": trial.suggest_categorical("freeze_encoder", [False, True]),
        "freeze_batchnorm": trial.suggest_categorical("freeze_batchnorm", [False, True]),
        "dataset_balance_power": trial.suggest_float("dataset_balance_power", 0.0, 1.0),
        "ecg_weight": trial.suggest_float("ecg_weight", 1.0, 10.0),
        "high_hr_weight": trial.suggest_float("high_hr_weight", 1.0, 8.0),
        "tachy_weight": trial.suggest_float("tachy_weight", 1.0, 12.0),
    }


def build_ecg_included_sampling_weights(dataset: Dataset, params: dict) -> np.ndarray:
    """Build ECG-included sampling weights from dataset, ECG, and HR factors."""
    metadata = dataset.metadata.copy()

    if "dataset" not in metadata.columns:
        raise KeyError("Training metadata is missing dataset column.")

    if "source_role" not in metadata.columns:
        raise KeyError("Training metadata is missing source_role column.")

    targets = np.asarray(dataset.y, dtype=float)
    weights = np.ones(len(metadata), dtype=np.float64)

    dataset_counts = metadata["dataset"].value_counts().to_dict()

    for dataset_name, count in dataset_counts.items():
        mask = metadata["dataset"].eq(dataset_name).to_numpy()
        factor = (len(metadata) / float(count)) ** float(params["dataset_balance_power"])
        weights[mask] *= factor

    ecg_mask = metadata["dataset"].eq("ecg_fitness").to_numpy()
    high_hr_mask = targets >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"]
    tachy_mask = targets >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"]

    weights[ecg_mask] *= float(params["ecg_weight"])
    weights[high_hr_mask] *= float(params["high_hr_weight"])
    weights[tachy_mask] *= float(params["tachy_weight"])

    if not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError("Invalid ECG-included sampling weights.")

    return weights


def summarize_ecg_included_sampling_weights(dataset: Dataset, params: dict) -> pd.DataFrame:
    """Summarize ECG-included sampling weights by dataset and HR slice."""
    metadata = dataset.metadata.copy()
    targets = np.asarray(dataset.y, dtype=float)
    weights = build_ecg_included_sampling_weights(dataset, params)

    table = metadata.copy()
    table["target_hr"] = targets
    table["sampling_weight"] = weights
    table["hr_slice"] = np.select(
        [
            table["target_hr"] >= NB12_DECISION_THRESHOLDS["exercise_tachy_threshold_bpm"],
            table["target_hr"] >= NB12_DECISION_THRESHOLDS["high_hr_threshold_bpm"],
        ],
        ["ge_120", "100_120"],
        default="lt_100",
    )

    summary = (
        table
        .groupby(["dataset", "source_role", "hr_slice"], dropna=False)
        .agg(
            rows=("sampling_weight", "size"),
            target_hr_mean=("target_hr", "mean"),
            weight_mean=("sampling_weight", "mean"),
            weight_min=("sampling_weight", "min"),
            weight_max=("sampling_weight", "max"),
            total_weight=("sampling_weight", "sum"),
        )
        .reset_index()
        .sort_values(["dataset", "source_role", "hr_slice"])
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)

    return summary


def make_ecg_included_training_loader(
    dataset: Dataset,
    params: dict,
    seed: int,
) -> DataLoader:
    """Create a weighted training DataLoader for ECG-included transfer."""
    weights = build_ecg_included_sampling_weights(dataset, params)

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )

    return DataLoader(
        dataset,
        batch_size=int(params["batch_size"]),
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def build_ecg_included_trial_model(params: dict) -> tuple[nn.Module, dict]:
    """Build checkpoint-initialized model for one ECG-included transfer trial."""
    model_config = dict(FROZEN_MODEL_CONFIG)
    model_config["dropout"] = float(params["dropout"])

    model = CRVSEPhysFormer(**model_config)
    model.load_state_dict(FROZEN_CHECKPOINT["model_state"], strict=True)
    model.to(DEVICE)

    frozen_batchnorm_names = []

    if bool(params["freeze_encoder"]):
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False

    if bool(params["freeze_batchnorm"]):
        frozen_batchnorm_names = nb12_hh_freeze_batchnorm(model)

    trainable_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if len(trainable_parameters) == 0:
        raise ValueError("Trial produced a model with no trainable parameters.")

    return model, {
        "freeze_encoder": bool(params["freeze_encoder"]),
        "freeze_batchnorm": bool(params["freeze_batchnorm"]),
        "frozen_batchnorm_names": frozen_batchnorm_names,
        "total_parameters": int(sum(parameter.numel() for parameter in model.parameters())),
        "trainable_parameters": int(sum(parameter.numel() for parameter in trainable_parameters)),
    }


def enforce_ecg_included_training_modes(
    model: nn.Module,
    params: dict,
) -> None:
    """Keep frozen modules in eval mode after model.train()."""
    if bool(params["freeze_encoder"]):
        model.encoder.eval()

    if bool(params["freeze_batchnorm"]):
        for module in model.modules():
            if isinstance(module, nn.BatchNorm1d):
                module.eval()


def train_one_ecg_included_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    params: dict,
) -> dict:
    """Train one ECG-included transfer epoch."""
    model.train()
    enforce_ecg_included_training_modes(model, params)

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        trainable_parameters = [
            parameter for parameter in model.parameters()
            if parameter.requires_grad
        ]

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=float(params["grad_clip"]),
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def evaluate_ecg_included_model(
    model: nn.Module,
    experiment_name: str,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate one model on ECG-included combined roles."""
    predictions, bundle = evaluate_model_on_named_datasets(
        model=model,
        datasets=ECG_INCLUDED_DATASETS,
        loaders=ECG_INCLUDED_LOADERS,
        experiment_name=experiment_name,
    )

    return predictions, bundle


def extract_ecg_included_metric(
    summary: pd.DataFrame,
    key_col: str,
    key_value: str,
    metric_col: str,
) -> float:
    """Extract one metric from an ECG-included summary table."""
    rows = summary.loc[summary[key_col].eq(key_value)]

    if len(rows) != 1:
        raise ValueError(
            f"Expected one row for {key_col}={key_value}, found {len(rows)}."
        )

    return float(rows.iloc[0][metric_col])


def compute_ecg_included_validation_score(bundle: dict) -> dict:
    """Compute the ECG-included validation score from a full evaluation bundle."""
    role_summary = bundle["role_summary"]
    source_role_summary = bundle["source_role_summary"]

    combined_val_mae = extract_ecg_included_metric(
        role_summary,
        "evaluation_role",
        "ecg_included_val",
        "mae_mean",
    )
    combined_val_p90 = extract_ecg_included_metric(
        role_summary,
        "evaluation_role",
        "ecg_included_val",
        "mae_p90",
    )

    non_ecg_val_mae = extract_ecg_included_metric(
        source_role_summary,
        "source_role",
        "finetune_val",
        "mae_mean",
    )
    ecg_val_mae = extract_ecg_included_metric(
        source_role_summary,
        "source_role",
        "ecg_subject_val",
        "mae_mean",
    )
    ecg_val_under_rate = extract_ecg_included_metric(
        source_role_summary,
        "source_role",
        "ecg_subject_val",
        "severe_underprediction_rate",
    )

    frozen_source_summary = FROZEN_ECG_INCLUDED_BUNDLE["source_role_summary"]
    frozen_role_summary = FROZEN_ECG_INCLUDED_BUNDLE["role_summary"]

    frozen_ecg_val_under_rate = extract_ecg_included_metric(
        frozen_source_summary,
        "source_role",
        "ecg_subject_val",
        "severe_underprediction_rate",
    )
    frozen_combined_val_p90 = extract_ecg_included_metric(
        frozen_role_summary,
        "evaluation_role",
        "ecg_included_val",
        "mae_p90",
    )

    ecg_val_under_worsening = max(0.0, ecg_val_under_rate - frozen_ecg_val_under_rate)
    combined_val_p90_worsening = max(0.0, combined_val_p90 - frozen_combined_val_p90)

    score = (
        ECG_INCLUDED_OBJECTIVE_WEIGHTS["combined_val_mae"] * combined_val_mae
        + ECG_INCLUDED_OBJECTIVE_WEIGHTS["non_ecg_val_mae"] * non_ecg_val_mae
        + ECG_INCLUDED_OBJECTIVE_WEIGHTS["ecg_val_mae"] * ecg_val_mae
        + ECG_INCLUDED_OBJECTIVE_WEIGHTS["ecg_val_underprediction_penalty"] * ecg_val_under_worsening
        + ECG_INCLUDED_OBJECTIVE_WEIGHTS["combined_val_p90_worsening_penalty"] * combined_val_p90_worsening
    )

    return {
        "ecg_included_validation_score": float(score),
        "combined_val_mae": float(combined_val_mae),
        "combined_val_p90": float(combined_val_p90),
        "non_ecg_val_mae": float(non_ecg_val_mae),
        "ecg_val_mae": float(ecg_val_mae),
        "ecg_val_severe_underprediction_rate": float(ecg_val_under_rate),
        "ecg_val_severe_underprediction_rate_worsening": float(ecg_val_under_worsening),
        "combined_val_p90_worsening": float(combined_val_p90_worsening),
    }


def build_ecg_included_candidate_summary(
    params: dict,
    freezing_report: dict,
    history: list[dict],
    bundle: dict,
    validation_score: dict,
    trial_number: int,
    smoke_mode: bool,
) -> dict:
    """Build one ECG-included candidate summary row."""
    role_summary = bundle["role_summary"]
    source_role_summary = bundle["source_role_summary"]

    frozen_role_summary = FROZEN_ECG_INCLUDED_BUNDLE["role_summary"]
    frozen_source_summary = FROZEN_ECG_INCLUDED_BUNDLE["source_role_summary"]

    def delta_role(role: str, metric: str) -> float:
        return (
            extract_ecg_included_metric(role_summary, "evaluation_role", role, metric)
            - extract_ecg_included_metric(frozen_role_summary, "evaluation_role", role, metric)
        )

    def delta_source(source_role: str, metric: str) -> float:
        return (
            extract_ecg_included_metric(source_role_summary, "source_role", source_role, metric)
            - extract_ecg_included_metric(frozen_source_summary, "source_role", source_role, metric)
        )

    summary = {
        "trial_number": int(trial_number),
        "smoke_mode": bool(smoke_mode),
        "epochs_run": int(len(history)),
        **params,
        **freezing_report,
        **validation_score,
        "combined_val_delta_mae": delta_role("ecg_included_val", "mae_mean"),
        "combined_test_delta_mae": delta_role("ecg_included_test", "mae_mean"),
        "non_ecg_val_delta_mae": delta_source("finetune_val", "mae_mean"),
        "non_ecg_test_delta_mae": delta_source("finetune_test", "mae_mean"),
        "ecg_val_delta_mae": delta_source("ecg_subject_val", "mae_mean"),
        "ecg_test_delta_mae": delta_source("ecg_subject_test", "mae_mean"),
        "ecg_val_signed_error_delta": delta_source("ecg_subject_val", "signed_error_mean"),
        "ecg_test_signed_error_delta": delta_source("ecg_subject_test", "signed_error_mean"),
        "ecg_val_underprediction_delta": delta_source(
            "ecg_subject_val",
            "severe_underprediction_rate",
        ),
        "ecg_test_underprediction_delta": delta_source(
            "ecg_subject_test",
            "severe_underprediction_rate",
        ),
    }

    if (
        summary["ecg_val_delta_mae"] < 0
        and summary["non_ecg_val_delta_mae"] <= 0.25
        and summary["combined_test_delta_mae"] <= 0.25
    ):
        summary["decision_status"] = "ecg_subject_candidate_for_review"
    elif summary["ecg_val_delta_mae"] < 0:
        summary["decision_status"] = "ecg_improves_but_tradeoff"
    else:
        summary["decision_status"] = "reject_or_debug"

    return summary


def run_ecg_included_trial(
    trial: optuna.Trial,
    smoke_mode: bool,
) -> dict:
    """Run one ECG-included transfer trial."""
    params = suggest_ecg_included_trial_params(trial)

    max_epochs = ECG_INCLUDED_SMOKE_MAX_EPOCHS if smoke_mode else 12
    patience = max_epochs if smoke_mode else 4

    train_dataset = make_ecg_included_training_dataset(smoke_mode=smoke_mode)

    sampling_summary = summarize_ecg_included_sampling_weights(
        dataset=train_dataset,
        params=params,
    )

    train_loader = make_ecg_included_training_loader(
        dataset=train_dataset,
        params=params,
        seed=SEED + trial.number,
    )

    model, freezing_report = build_ecg_included_trial_model(params)

    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"]),
    )

    loss_fn = nn.HuberLoss(delta=float(params["huber_delta"]))

    best_score = math.inf
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_metrics = train_one_ecg_included_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            params=params,
        )

        _, val_bundle = evaluate_model_on_named_datasets(
            model=model,
            datasets={"ecg_included_val": ECG_INCLUDED_DATASETS["ecg_included_val"]},
            loaders={"ecg_included_val": ECG_INCLUDED_LOADERS["ecg_included_val"]},
            experiment_name=f"ecg_included_trial_{trial.number:03d}_epoch_{epoch:02d}",
        )

        validation_score = compute_ecg_included_validation_score(val_bundle)

        improved = validation_score["ecg_included_validation_score"] < (
            best_score - ECG_INCLUDED_TRIAL_MIN_DELTA
        )

        if improved:
            best_score = float(validation_score["ecg_included_validation_score"])
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history.append(
            {
                "epoch": epoch,
                **train_metrics,
                **validation_score,
                "best_epoch_so_far": int(best_epoch),
                "best_score_so_far": float(best_score),
                "improved": bool(improved),
            }
        )

        trial.report(float(validation_score["ecg_included_validation_score"]), step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    predictions, bundle = evaluate_ecg_included_model(
        model=model,
        experiment_name=f"ecg_included_trial_{trial.number:03d}",
    )

    final_validation_score = compute_ecg_included_validation_score(bundle)

    summary = build_ecg_included_candidate_summary(
        params=params,
        freezing_report=freezing_report,
        history=history,
        bundle=bundle,
        validation_score=final_validation_score,
        trial_number=trial.number,
        smoke_mode=smoke_mode,
    )

    trial.set_user_attr("summary", summary)
    trial.set_user_attr("history", history)

    return {
        "summary": summary,
        "history": pd.DataFrame(history),
        "sampling_summary": sampling_summary,
        "predictions": predictions,
        "bundle": bundle,
        "objective_value": final_validation_score["ecg_included_validation_score"],
    }


def ecg_included_smoke_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the ECG-included smoke trial."""
    result = run_ecg_included_trial(trial=trial, smoke_mode=True)
    ECG_INCLUDED_SMOKE_RESULTS.append(result)
    return float(result["objective_value"])


ecg_smoke_sampler = optuna.samplers.TPESampler(seed=SEED)
ecg_smoke_study = optuna.create_study(
    direction="minimize",
    sampler=ecg_smoke_sampler,
    study_name="nb12_ecg_included_transfer_smoke",
)

ecg_smoke_study.enqueue_trial(
    {
        "lr": 8e-6,
        "weight_decay": 2e-5,
        "dropout": 0.08,
        "huber_delta": 6.0,
        "batch_size": 256,
        "grad_clip": 6.0,
        "freeze_encoder": False,
        "freeze_batchnorm": True,
        "dataset_balance_power": 0.7,
        "ecg_weight": 5.0,
        "high_hr_weight": 3.0,
        "tachy_weight": 6.0,
    }
)

ecg_smoke_study.optimize(
    ecg_included_smoke_objective,
    n_trials=1,
    show_progress_bar=False,
)

ECG_INCLUDED_SMOKE_RESULT = ECG_INCLUDED_SMOKE_RESULTS[0]
ECG_INCLUDED_SMOKE_HISTORY = ECG_INCLUDED_SMOKE_RESULT["history"]
ECG_INCLUDED_SMOKE_BUNDLE = ECG_INCLUDED_SMOKE_RESULT["bundle"]
ECG_INCLUDED_SMOKE_SAMPLING_SUMMARY = ECG_INCLUDED_SMOKE_RESULT["sampling_summary"]
ECG_INCLUDED_SMOKE_SUMMARY = pd.DataFrame([ECG_INCLUDED_SMOKE_RESULT["summary"]])

numeric_cols = ECG_INCLUDED_SMOKE_SUMMARY.select_dtypes(include=[np.number]).columns
ECG_INCLUDED_SMOKE_SUMMARY[numeric_cols] = ECG_INCLUDED_SMOKE_SUMMARY[numeric_cols].round(4)

print("ECG-included smoke sampling summary:")
display(ECG_INCLUDED_SMOKE_SAMPLING_SUMMARY)

print("\nECG-included smoke trial history:")
display(ECG_INCLUDED_SMOKE_HISTORY.round(4))

print("\nECG-included smoke role summary:")
display(ECG_INCLUDED_SMOKE_BUNDLE["role_summary"])

print("\nECG-included smoke source-role summary:")
display(ECG_INCLUDED_SMOKE_BUNDLE["source_role_summary"])

print("\nECG-included smoke trial summary:")
display(ECG_INCLUDED_SMOKE_SUMMARY)

print("\nBest ECG-included smoke study value:")
print(ecg_smoke_study.best_value)

[I 2026-07-16 17:11:56,777] A new study created in memory with name: nb12_ecg_included_transfer_smoke
[I 2026-07-16 17:12:04,958] Trial 0 finished with value: 14.383985 and parameters: {'lr': 8e-06, 'weight_decay': 2e-05, 'dropout': 0.08, 'huber_delta': 6.0, 'batch_size': 256, 'grad_clip': 6.0, 'freeze_encoder': False, 'freeze_batchnorm': True, 'dataset_balance_power': 0.7, 'ecg_weight': 5.0, 'high_hr_weight': 3.0, 'tachy_weight': 6.0}. Best is trial 0 with value: 14.383985.


ECG-included smoke sampling summary:


,dataset,source_role,hr_slice,rows,target_hr_mean,weight_mean,weight_min,weight_max,total_weight
0,ecg_fitness,ecg_subject_train,100_120,137,109.5871,24.3676,24.3676,24.3676,3338.3573
1,ecg_fitness,ecg_subject_train,ge_120,121,137.3166,146.2054,146.2054,146.2054,17690.8572
2,ecg_fitness,ecg_subject_train,lt_100,126,85.7290,8.1225,8.1225,8.1225,1023.4380
3,mcd_rppg,finetune_train,100_120,14,105.8095,7.0631,7.0631,7.0631,98.8840
4,mcd_rppg,finetune_train,lt_100,212,77.2322,2.3544,2.3544,2.3544,499.1290
5,ubfc_phys,finetune_train,100_120,9,104.7690,9.8270,9.8270,9.8270,88.4430
6,ubfc_phys,finetune_train,lt_100,132,77.2388,3.2757,3.2757,3.2757,432.3878
7,ubfc_rppg,finetune_train,100_120,8,108.4387,43.2076,43.2076,43.2076,345.6606
8,ubfc_rppg,finetune_train,ge_120,2,125.3064,259.2454,259.2454,259.2454,518.4909
9,ubfc_rppg,finetune_train,lt_100,7,84.9780,14.4025,14.4025,14.4025,100.8177



ECG-included smoke trial history:


,epoch,train_loss,train_mae,grad_norm_mean,grad_norm_max,ecg_included_validation_score,combined_val_mae,combined_val_p90,non_ecg_val_mae,ecg_val_mae,ecg_val_severe_underprediction_rate,ecg_val_severe_underprediction_rate_worsening,combined_val_p90_worsening,best_epoch_so_far,best_score_so_far,improved
0,1,124.7885,23.5851,937.9685,1023.6002,14.3840,6.4997,14.8112,6.0856,14.1844,0.3072,0.0,0.0864,1,14.384,True
1,2,131.6848,24.7486,941.7744,1034.6964,14.4121,6.5640,14.9698,6.1559,14.1355,0.2941,0.0,0.2450,1,14.384,False


...


ECG-included smoke source-role summary:


,source_role,evaluation_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,mae_max,signed_error_mean,severe_underprediction_rate
0,ecg_subject_test,ecg_included_test,153,106.7882,106.5349,8.1263,5.5881,17.2623,32.4792,40.9555,-0.2533,0.1176
1,ecg_subject_train,ecg_included_train,576,109.8421,100.6015,15.4129,8.7071,41.6673,53.3431,85.0397,-9.2406,0.3507
2,ecg_subject_val,ecg_included_val,153,109.2220,100.9235,14.1844,8.2189,34.2480,54.5251,89.7908,-8.2985,0.3072
3,finetune_test,ecg_included_test,2828,79.7822,81.4748,5.8242,3.6910,13.6874,19.1731,72.4598,1.6926,0.0573
4,finetune_train,ecg_included_train,12503,80.2370,82.0918,5.8840,3.7789,13.8118,19.2061,54.6674,1.8548,0.0536
5,finetune_val,ecg_included_val,2839,80.1617,81.9245,6.0856,3.8644,14.1942,19.4362,51.0565,1.7628,0.0652



ECG-included smoke trial summary:


,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,combined_test_delta_mae,non_ecg_val_delta_mae,non_ecg_test_delta_mae,ecg_val_delta_mae,ecg_test_delta_mae,ecg_val_signed_error_delta,ecg_test_signed_error_delta,ecg_val_underprediction_delta,ecg_test_underprediction_delta,decision_status
0,0,True,2,0.0,0.0,0.08,6.0,256,6.0,False,...,0.0518,0.0581,0.0546,-0.0536,-0.0013,0.4356,0.3519,-0.0065,0.0,ecg_subject_candidate_for_review



Best ECG-included smoke study value:
14.383985


## Full ECG-Included Transfer Search

This section runs the full ECG-Fitness-included transfer search.

ECG-Fitness is included by subject-level split. It is not treated as out-of-domain in this branch.

The search asks whether transfer learning can improve held-out ECG-Fitness validation-subject behavior while preserving non-ECG validation and held-out test behavior.

In [20]:
ECG_INCLUDED_FULL_RESULTS = []
ECG_INCLUDED_FULL_N_TRIALS = 12


def ecg_included_full_objective(trial: optuna.Trial) -> float:
    """Optuna objective wrapper for the full ECG-included transfer search."""
    result = run_ecg_included_trial(trial=trial, smoke_mode=False)
    ECG_INCLUDED_FULL_RESULTS.append(result)
    return float(result["objective_value"])


def build_ecg_included_completed_trial_summary(study: optuna.Study) -> pd.DataFrame:
    """Build a completed-trial summary table for the ECG-included search."""
    rows = []

    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue

        summary = trial.user_attrs.get("summary")

        if not isinstance(summary, dict):
            continue

        row = dict(summary)
        row["optuna_value"] = float(trial.value)
        row["optuna_state"] = str(trial.state.name)
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    table = pd.DataFrame(rows)

    sort_cols = [
        "decision_status",
        "ecg_included_validation_score",
        "ecg_val_delta_mae",
        "non_ecg_val_delta_mae",
        "combined_test_delta_mae",
    ]
    existing_sort_cols = [col for col in sort_cols if col in table.columns]

    if existing_sort_cols:
        table = table.sort_values(existing_sort_cols, ascending=[True] * len(existing_sort_cols))

    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

    return table.reset_index(drop=True)


def save_ecg_included_search_artifacts(
    study: optuna.Study,
    trial_summary: pd.DataFrame,
    output_dir: Path,
) -> dict:
    """Save ECG-included search artifacts."""
    output_dir.mkdir(parents=True, exist_ok=True)

    trial_summary_path = output_dir / "nb12_ecg_included_trial_summary.csv"
    best_params_path = output_dir / "nb12_ecg_included_best_params.json"
    study_trials_path = output_dir / "nb12_ecg_included_optuna_trials.csv"

    trial_summary.to_csv(trial_summary_path, index=False)

    with best_params_path.open("w", encoding="utf-8") as file:
        json.dump(study.best_params, file, indent=2)

    study.trials_dataframe().to_csv(study_trials_path, index=False)

    return {
        "trial_summary_csv": str(trial_summary_path),
        "best_params_json": str(best_params_path),
        "study_trials_csv": str(study_trials_path),
    }


ecg_full_sampler = optuna.samplers.TPESampler(seed=SEED)
ecg_full_pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=3,
)

ecg_full_study = optuna.create_study(
    direction="minimize",
    sampler=ecg_full_sampler,
    pruner=ecg_full_pruner,
    study_name="nb12_ecg_included_transfer_full",
)

ecg_full_study.enqueue_trial(
    {
        "lr": 8e-6,
        "weight_decay": 2e-5,
        "dropout": 0.08,
        "huber_delta": 6.0,
        "batch_size": 256,
        "grad_clip": 6.0,
        "freeze_encoder": False,
        "freeze_batchnorm": True,
        "dataset_balance_power": 0.7,
        "ecg_weight": 5.0,
        "high_hr_weight": 3.0,
        "tachy_weight": 6.0,
    }
)

ecg_full_study.optimize(
    ecg_included_full_objective,
    n_trials=ECG_INCLUDED_FULL_N_TRIALS,
    show_progress_bar=True,
)

ECG_INCLUDED_TRIAL_SUMMARY = build_ecg_included_completed_trial_summary(
    ecg_full_study
)

ECG_INCLUDED_SEARCH_ARTIFACTS = save_ecg_included_search_artifacts(
    study=ecg_full_study,
    trial_summary=ECG_INCLUDED_TRIAL_SUMMARY,
    output_dir=ECG_INCLUDED_BRANCH_DIR,
)

print("NB12 ECG-included search best value:")
print(ecg_full_study.best_value)

print("\nNB12 ECG-included search best params:")
print(json.dumps(ecg_full_study.best_params, indent=2))

print("\nNB12 ECG-included trial summary:")
display(ECG_INCLUDED_TRIAL_SUMMARY)

print("\nNB12 ECG-included search artifacts:")
print(json.dumps(ECG_INCLUDED_SEARCH_ARTIFACTS, indent=2))

[I 2026-07-16 17:12:05,112] A new study created in memory with name: nb12_ecg_included_transfer_full


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-07-16 17:13:08,543] Trial 0 finished with value: 16.701725 and parameters: {'lr': 8e-06, 'weight_decay': 2e-05, 'dropout': 0.08, 'huber_delta': 6.0, 'batch_size': 256, 'grad_clip': 6.0, 'freeze_encoder': False, 'freeze_batchnorm': True, 'dataset_balance_power': 0.7, 'ecg_weight': 5.0, 'high_hr_weight': 3.0, 'tachy_weight': 6.0}. Best is trial 0 with value: 16.701725.
[I 2026-07-16 17:14:11,767] Trial 1 finished with value: 28.180555 and parameters: {'lr': 1.9906996673933362e-05, 'weight_decay': 0.0032859708169642424, 'dropout': 0.2329984854528513, 'huber_delta': 8.38792635777333, 'batch_size': 128, 'grad_clip': 1.4065852851773961, 'freeze_encoder': False, 'freeze_batchnorm': False, 'dataset_balance_power': 0.9699098521619943, 'ecg_weight': 8.491983767203795, 'high_hr_weight': 2.486373774747933, 'tachy_weight': 3.0000746392781066}. Best is trial 0 with value: 16.701725.
[I 2026-07-16 17:15:13,437] Trial 2 finished with value: 16.93384 and parameters: {'lr': 9.835468046820027e-06

,trial_number,smoke_mode,epochs_run,lr,weight_decay,dropout,huber_delta,batch_size,grad_clip,freeze_encoder,...,non_ecg_test_delta_mae,ecg_val_delta_mae,ecg_test_delta_mae,ecg_val_signed_error_delta,ecg_test_signed_error_delta,ecg_val_underprediction_delta,ecg_test_underprediction_delta,decision_status,optuna_value,optuna_state
0,11,False,5,0.0000,0.0002,0.1191,9.3471,128,7.9225,False,...,0.5728,2.0954,2.1017,-1.4379,-0.4145,0.0392,0.0719,reject_or_debug,16.5379,COMPLETE
1,0,False,5,0.0000,0.0000,0.0800,6.0000,256,6.0000,False,...,2.3406,0.3738,1.3110,7.0844,5.5439,-0.0915,-0.0457,reject_or_debug,16.7017,COMPLETE
2,7,False,5,0.0000,0.0004,0.2402,8.0515,128,4.6591,False,...,1.8771,1.3438,2.4074,3.9488,4.5811,-0.0523,-0.0196,reject_or_debug,16.7866,COMPLETE
3,2,False,5,0.0000,0.0000,0.1812,6.8875,256,1.9765,True,...,2.4760,0.5668,1.7076,7.3631,6.0740,-0.1045,-0.0457,reject_or_debug,16.9338,COMPLETE
4,4,False,5,0.0001,0.0000,0.1800,7.9204,256,6.4259,False,...,2.3289,3.0093,2.4509,11.2499,7.4616,-0.1045,-0.0522,reject_or_debug,19.0161,COMPLETE
5,8,False,5,0.0000,0.0000,0.2389,5.0592,256,2.1285,False,...,4.7783,2.3371,4.3994,12.2201,10.1269,-0.1372,-0.0457,reject_or_debug,20.3028,COMPLETE
6,5,False,5,0.0000,0.0000,0.2572,6.2108,256,1.9865,False,...,4.8791,2.8302,5.3098,11.8349,10.7709,-0.1372,-0.0522,reject_or_debug,20.5546,COMPLETE
7,3,False,5,0.0000,0.0000,0.0663,11.5400,128,3.1323,True,...,4.5319,2.6226,2.1326,11.9862,7.2406,-0.1176,-0.0522,reject_or_debug,21.2841,COMPLETE
8,9,False,5,0.0001,0.0021,0.1295,3.9905,256,6.7261,False,...,5.8714,6.7043,5.2845,17.9572,11.9776,-0.1764,-0.0849,reject_or_debug,25.7660,COMPLETE
9,1,False,5,0.0000,0.0033,0.2330,8.3879,128,1.4066,False,...,11.3196,5.3545,8.8714,18.1392,15.8649,-0.1764,-0.0784,reject_or_debug,28.1806,COMPLETE



NB12 ECG-included search artifacts:
{
  "trial_summary_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_included_transfer_search/nb12_ecg_included_trial_summary.csv",
  "best_params_json": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_included_transfer_search/nb12_ecg_included_best_params.json",
  "study_trials_csv": "/kaggle/working/crvse_nb12_optuna_search/ecg_fitness_subject_split_branch/ecg_included_transfer_search/nb12_ecg_included_optuna_trials.csv"
}


## NB12 Final Branch Comparison

This section compares the main NB12 branches against the frozen source-FPS baseline.

The comparison separates three research questions:

1. Does transfer improve non-ECG live-compatible validation and test performance?
2. Does high-HR-aware sampling reduce the ECG-Fitness / high-HR penalty?
3. Does including ECG-Fitness by subject-level split improve held-out ECG-Fitness subject behavior?

No checkpoint should be adopted unless it improves the intended target while preserving important failure modes.

In [21]:
def build_nb12_final_branch_comparison() -> pd.DataFrame:
    """Build the final NB12 branch comparison table."""
    rows = []

    rows.append(
        {
            "branch": "frozen_sourcefps_baseline",
            "selected_trial": None,
            "selection_basis": "current_app_checkpoint",
            "adopt_checkpoint": False,
            "main_result": "reference_checkpoint",
            "val_delta_mean": 0.0,
            "test_delta_mean": 0.0,
            "ecg_or_ood_delta_mean": 0.0,
            "high_hr_delta_mae": 0.0,
            "underprediction_delta": 0.0,
            "interpretation": "Current app checkpoint remains the reference.",
        }
    )

    first_transfer = get_best_trial_row(
        NB12_TRANSFER_TRIAL_SUMMARY,
        label="first_transfer_best",
        score_col="validation_risk_score",
    )

    rows.append(
        {
            "branch": "transfer_exclude_ecg",
            "selected_trial": int(first_transfer["trial_number"]),
            "selection_basis": "validation_risk_score",
            "adopt_checkpoint": False,
            "main_result": "in_distribution_gain_with_ecg_ood_penalty",
            "val_delta_mean": float(first_transfer["val_delta_mean"]),
            "test_delta_mean": float(first_transfer["test_delta_mean"]),
            "ecg_or_ood_delta_mean": float(first_transfer["ood_delta_mean"]),
            "high_hr_delta_mae": float(first_transfer["high_hr_delta_mae"]),
            "underprediction_delta": float(
                first_transfer["high_hr_severe_underprediction_rate_delta"]
            ),
            "interpretation": (
                "Improved non-ECG validation/test but worsened ECG-Fitness OOD and high-HR behavior."
            ),
        }
    )

    high_hr_transfer = get_best_trial_row(
        NB12_HIGH_HR_TRIAL_SUMMARY,
        label="high_hr_aware_best",
        score_col="high_hr_validation_score",
    )

    rows.append(
        {
            "branch": "high_hr_aware_transfer_exclude_ecg",
            "selected_trial": int(high_hr_transfer["trial_number"]),
            "selection_basis": "high_hr_validation_score",
            "adopt_checkpoint": False,
            "main_result": "reduced_high_hr_harm_but_still_tradeoff",
            "val_delta_mean": float(high_hr_transfer["val_delta_mean"]),
            "test_delta_mean": float(high_hr_transfer["test_delta_mean"]),
            "ecg_or_ood_delta_mean": float(high_hr_transfer["ood_delta_mean"]),
            "high_hr_delta_mae": float(high_hr_transfer["high_hr_delta_mae"]),
            "underprediction_delta": float(
                high_hr_transfer["high_hr_severe_underprediction_rate_delta"]
            ),
            "interpretation": (
                "Reduced high-HR harm compared with first transfer, but still worsened ECG-Fitness OOD."
            ),
        }
    )

    ecg_included = get_best_trial_row(
        ECG_INCLUDED_TRIAL_SUMMARY,
        label="ecg_included_best",
        score_col="ecg_included_validation_score",
    )

    rows.append(
        {
            "branch": "ecg_included_subject_split_transfer",
            "selected_trial": int(ecg_included["trial_number"]),
            "selection_basis": "ecg_included_validation_score",
            "adopt_checkpoint": False,
            "main_result": "held_out_ecg_subjects_not_improved",
            "val_delta_mean": float(ecg_included["non_ecg_val_delta_mae"]),
            "test_delta_mean": float(ecg_included["non_ecg_test_delta_mae"]),
            "ecg_or_ood_delta_mean": float(ecg_included["ecg_val_delta_mae"]),
            "high_hr_delta_mae": np.nan,
            "underprediction_delta": float(ecg_included["ecg_val_underprediction_delta"]),
            "interpretation": (
                "Including ECG-Fitness train subjects did not improve held-out ECG validation subjects."
            ),
        }
    )

    table = pd.DataFrame(rows)

    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

    return table


NB12_FINAL_BRANCH_COMPARISON = build_nb12_final_branch_comparison()

NB12_FINAL_CONCLUSION = {
    "notebook": "NB_P2_12_Live-Compatible_CRVSE_Optuna_Search",
    "adopt_checkpoint_now": False,
    "current_app_checkpoint_remains": "frozen_sourcefps_baseline",
    "main_findings": [
        "Standard transfer search improved non-ECG live-compatible validation/test performance but worsened ECG-Fitness OOD and high-HR behavior.",
        "High-HR-aware transfer reduced the high-HR harm compared with standard transfer but still did not preserve ECG-Fitness OOD behavior enough for adoption.",
        "ECG-Fitness inclusion by subject-level split did not improve held-out ECG-Fitness validation subjects under this transfer setup.",
        "The model repeatedly shows a trade-off between normal live-compatible performance and exercise/high-HR behavior.",
    ],
    "scientific_interpretation": (
        "NB12 supports keeping the frozen source-FPS checkpoint as the app checkpoint. "
        "The searched transfer-learning variants are useful research artifacts, but none "
        "solves the ECG-Fitness/high-HR failure mode robustly enough for app adoption."
    ),
    "recommended_next_step": (
        "Document NB12 as a negative/constraint-finding result. Future work should focus "
        "on signal-quality-aware ECG-Fitness analysis, calibration/bias correction, or "
        "architecture/loss changes rather than promoting a transfer checkpoint."
    ),
}

nb12_final_dir = NB12_WORKING_DIR / "nb12_final_summary"
nb12_final_dir.mkdir(parents=True, exist_ok=True)

final_comparison_path = nb12_final_dir / "nb12_final_branch_comparison.csv"
final_conclusion_path = nb12_final_dir / "nb12_final_conclusion.json"

NB12_FINAL_BRANCH_COMPARISON.to_csv(final_comparison_path, index=False)

with final_conclusion_path.open("w", encoding="utf-8") as file:
    json.dump(NB12_FINAL_CONCLUSION, file, indent=2)

print("NB12 final branch comparison:")
display(NB12_FINAL_BRANCH_COMPARISON)

print("\nNB12 final conclusion:")
print(json.dumps(NB12_FINAL_CONCLUSION, indent=2))

print("\nSaved NB12 final summary artifacts:")
print(json.dumps(
    {
        "final_branch_comparison_csv": str(final_comparison_path),
        "final_conclusion_json": str(final_conclusion_path),
    },
    indent=2,
))

NB12 final branch comparison:


,branch,selected_trial,selection_basis,adopt_checkpoint,main_result,val_delta_mean,test_delta_mean,ecg_or_ood_delta_mean,high_hr_delta_mae,underprediction_delta,interpretation
0,frozen_sourcefps_baseline,NaN,current_app_checkpoint,False,reference_checkpoint,0.0000,0.0000,0.0000,0.0000,0.0000,Current app checkpoint remains the reference.
1,transfer_exclude_ecg,10.0,validation_risk_score,False,in_distribution_gain_with_ecg_ood_penalty,-0.2732,-0.1466,1.0024,0.8019,0.0466,Improved non-ECG validation/test but worsened ECG-Fitness OOD and high-HR behavior.
2,high_hr_aware_transfer_exclude_ecg,11.0,high_hr_validation_score,False,reduced_high_hr_harm_but_still_tradeoff,-0.2283,-0.0476,1.0852,0.3735,0.0287,"Reduced high-HR harm compared with first transfer, but still worsened ECG-Fitness OOD."
3,ecg_included_subject_split_transfer,11.0,ecg_included_validation_score,False,held_out_ecg_subjects_not_improved,0.6186,0.5728,2.0954,NaN,0.0392,Including ECG-Fitness train subjects did not improve held-out ECG validation subjects.



NB12 final conclusion:
{
  "notebook": "NB_P2_12_Live-Compatible_CRVSE_Optuna_Search",
  "adopt_checkpoint_now": false,
  "current_app_checkpoint_remains": "frozen_sourcefps_baseline",
  "main_findings": [
    "Standard transfer search improved non-ECG live-compatible validation/test performance but worsened ECG-Fitness OOD and high-HR behavior.",
    "High-HR-aware transfer reduced the high-HR harm compared with standard transfer but still did not preserve ECG-Fitness OOD behavior enough for adoption.",
    "ECG-Fitness inclusion by subject-level split did not improve held-out ECG-Fitness validation subjects under this transfer setup.",
    "The model repeatedly shows a trade-off between normal live-compatible performance and exercise/high-HR behavior."
  ],
  "scientific_interpretation": "NB12 supports keeping the frozen source-FPS checkpoint as the app checkpoint. The searched transfer-learning variants are useful research artifacts, but none solves the ECG-Fitness/high-HR failure 

## Final NB12 Conclusion

NB12 tested whether live-compatible source-FPS tensors could produce a better CRVSE PhysFormer checkpoint through Optuna-guided transfer learning.

The result does not support replacing the current app checkpoint.

The frozen source-FPS checkpoint remains the app checkpoint.

### Main Findings

The standard transfer search improved non-ECG live-compatible validation and test performance, but worsened ECG-Fitness out-of-domain and high-HR behavior.

The high-HR-aware transfer search reduced the high-HR harm compared with standard transfer, but still did not preserve ECG-Fitness behavior well enough for adoption.

The ECG-Fitness-included subject-level branch did not improve held-out ECG-Fitness validation subjects under this transfer setup. Once ECG-Fitness was included by subject split, it was no longer treated as out-of-domain; it became held-out exercise/high-HR subject evaluation.

Across branches, the model repeatedly showed a trade-off between normal live-compatible performance and exercise/high-HR behavior.

### App Checkpoint Decision

No NB12 checkpoint should replace the frozen source-FPS baseline checkpoint.

The current app should continue to treat the CRVSE model output as experimental and secondary to spectral consensus HR.

### Scientific Interpretation

NB12 is a useful negative result.

It suggests that simple transfer learning, high-HR-aware sampling, and ECG-Fitness subject-split inclusion are not sufficient to solve the ECG-Fitness / high-HR failure mode for the current PhysFormer checkpoint.

The issue is likely broader than hyperparameter selection. It may involve signal quality, exercise-domain acquisition differences, label/window behavior, calibration bias, or model/loss design.
